# Polynomial identities and primitivity in double extension regular algebras of type (14641)

**James Gómez**<sup>1</sup> · **Helbert Venegas**<sup>2</sup>

<sup>1</sup> Grupo de Álgebra y Análisis, Universidad Pedagógica y Tecnológica de Colombia, Tunja, Colombia<br>
<sup>2</sup> Departamento de Matemáticas, Universidad Militar Nueva Granada, Bogotá, Colombia

**Abstract.** Zhang and Zhang listed twenty-six families of double extension regular algebras of type $(14641)$ and asked which members are PI and which are primitive. We answer the PI question for every member, degenerate ones included, and the answer only sees the twist parameter up to roots of unity. The PI degree is computed in twenty-one of the twenty-two families with PI members and bounded in the last one, $\mathbb C$. PI members are never primitive. For the non-PI members we settle primitivity in twenty-two families, leaving out a few degenerate members; in $\mathbb D$ and $\mathbb G$ a member is primitive exactly when its quotient division ring has trivial center.

**Keywords.** Double Ore extension · Artin–Schelter regular algebra · Polynomial identity · PI degree · Primitive ring · Zhang twist

**MSC.** Primary 16R20 · 16S36 · Secondary 16S35 · 16D60 · 16E65

## About this notebook

This notebook mirrors the paper section by section; numbering of results, equations and tables is the same as in the PDF. After the relevant sections there are code cells that check the computations: normal forms in the PBW basis with exact symbolic coefficients (SymPy), for symbolic $h$, $f$, $s$ whenever possible. A few checks are numerical (sample roots of unity, random values), and they say so. The proofs in the paper are by hand and do not depend on these checks.

Each check prints `ok` or `FAIL`. Run the next two cells first.

In [1]:
%%writefile pilib.py
"""Small exact toolkit for the double extension regular algebras of type (14641).

Elements of an algebra are dicts {word: coefficient}; a word is a tuple of
letter indices and the normal words are the non-decreasing ones (PBW basis).
Coefficients are sympy expressions, so identities are checked for symbolic
parameters (h, f, s, ...), not only for sample values.
"""
import sympy as sp
import itertools, math

I = sp.I
h, f, g, s, q, p = sp.symbols('h f g s q p', nonzero=True)


class Alg:
    """Algebra given by rewriting rules (a, b) -> [(coef, word)] for a > b."""

    def __init__(self, names, rules):
        self.names = names
        self.rules = rules
        self.cache = {}

    def nfw(self, w):
        if w in self.cache:
            return self.cache[w]
        for k in range(len(w) - 1):
            if w[k] > w[k + 1]:
                out = {}
                for c, rep in self.rules[(w[k], w[k + 1])]:
                    for m, d in self.nfw(w[:k] + tuple(rep) + w[k + 2:]).items():
                        out[m] = out.get(m, 0) + c * d
                self.cache[w] = out
                return out
        self.cache[w] = {w: 1}
        return self.cache[w]

    def gen(self, i):
        return {(i,): sp.Integer(1)}

    def mul(self, *els):
        out = els[0]
        for B in els[1:]:
            new = {}
            for u, a in out.items():
                for v, b in B.items():
                    for m, c in self.nfw(u + v).items():
                        new[m] = new.get(m, 0) + a * b * c
            out = new
        return out


def add(*terms):
    """add((c1, A1), (c2, A2), ...) = c1*A1 + c2*A2 + ..."""
    out = {}
    for c, A in terms:
        for m, d in A.items():
            out[m] = out.get(m, 0) + c * d
    return out


def zero(c, tol=1e-9):
    """Exact zero test for symbolic coefficients, tolerance for numeric ones."""
    if isinstance(c, (int, float, complex)):
        return abs(c) < tol
    e = sp.simplify(sp.expand(c))
    if e.is_zero:
        return True
    if e.free_symbols:
        return False
    return abs(complex(sp.N(e))) < tol


def is_zero(A, tol=1e-9):
    return all(zero(c, tol) for c in A.values())


def ratio(L, R):
    """The scalar c with L = c R, read off at a monomial where R does not vanish (None if none)."""
    for m, v in R.items():
        if not zero(v):
            return sp.simplify(L.get(m, 0) / v)
    return None


def skew(alg, a, b, c):
    """True if a*b == c*b*a."""
    return is_zero(add((1, alg.mul(a, b)), (-c, alg.mul(b, a))))


def check(label, ok):
    print(('  ok   ' if ok else '  FAIL ') + label)
    return ok


# ---------------------------------------------------------------------------
# Double extensions: letters x1=0, x2=1, y1=2, y2=3.
# Sigma rows: y1x1, y1x2, y2x1, y2x2 ; columns: x1y1, x2y1, x1y2, x2y2.
# ---------------------------------------------------------------------------
def double_ext(Q, P, Sigma):
    q12, q11 = Q
    p12, p11 = P
    cols = [(0, 2), (1, 2), (0, 3), (1, 3)]
    rules = {(1, 0): [(q12, (0, 1))] + ([(q11, (0, 0))] if q11 != 0 else []),
             (3, 2): [(p12, (2, 3))] + ([(p11, (2, 2))] if p11 != 0 else [])}
    for j in range(2):
        for i in range(2):
            row = Sigma[2 * j + i]
            rules[(2 + j, i)] = [(c, cols[k]) for k, c in enumerate(row) if c != 0]
    return Alg(['x1', 'x2', 'y1', 'y2'], rules)


def family(name, h=h, **par):
    """Families of [ZZ09, Section 4]; G is given in the basis of Section 5."""
    f_ = par.get('f', f)
    p_ = par.get('p', p)
    q_ = par.get('q', q)
    g_ = par.get('g', g)
    H = lambda rows: [[h * c for c in r] for r in rows]
    if name == 'B':
        return double_ext((p_, 0), (p_, 0), H([[0, 0, 0, 1], [0, 0, 1, 0], [0, -1, 0, 0], [1, 0, 0, 0]]))
    if name == 'C':
        return double_ext((p_, 0), (p_, 0), H([[-1, p_**2, 1, -p_], [-p_, 1, 1, -p_], [-p_, -2*p_**2, p_, -p_], [-p_, p_**2, 1, -1]]))
    if name == 'D':
        return double_ext((-1, 0), (p_, 0), H([[-p_, 0, 0, 0], [0, -p_**2, 1, 0], [0, 0, p_, 0], [1, 0, 0, 1]]))
    if name == 'E':
        return double_ext((-1, 0), (p_, 0), H([[0, 0, 1, 1], [0, 0, 1, -1], [-1, 1, 0, 0], [1, 1, 0, 0]]))
    if name == 'F':
        return double_ext((-1, 0), (p_, 0), H([[-1, -p_, 1, -1], [-p_, 1, 1, 1], [-p_, p_, p_, 1], [-p_, -p_, 1, -p_]]))
    if name == 'G':   # basis x2 + x1/(p-1)
        return double_ext((1, 0), (p_, 0), H([[p_, 0, 0, 0], [0, p_**2, 1, 0], [0, 0, p_, 0], [f_, 0, 0, 1]]))
    if name == 'I':
        return double_ext((q_, 0), (-1, 0), H([[-q_, -q_, 1, -q_], [1, 1, 1, -q_], [1, q_, q_, -q_], [-1, -q_, 1, -1]]))
    if name == 'J':
        return double_ext((q_, 0), (-1, 0), H([[0, 1, 0, 1], [-1, 0, 1, 0], [0, 1, 0, -1], [1, 0, 1, 0]]))
    if name == 'K':
        return double_ext((q_, 0), (-1, 0), H([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, f_, 0, 0]]))
    if name == 'L':
        return double_ext((q_, 0), (-1, 0), H([[0, 0, f_, 0], [0, 0, 0, 1], [f_, 0, 0, 0], [0, 1, 0, 0]]))
    if name == 'M':
        return double_ext((-1, 0), (-1, 0), H([[0, 1, 1, 0], [f_, 0, 0, -1], [1, 0, 0, -1], [0, -1, -f_, 0]]))
    if name == 'N':
        return double_ext((-1, 0), (-1, 0), [[0, -g_, 0, f_], [g_, 0, f_, 0], [0, f_, 0, -g_], [f_, 0, g_, 0]])
    if name == 'O':
        return double_ext((-1, 0), (-1, 0), H([[1, 0, 0, f_], [0, -1, 1, 0], [0, f_, -1, 0], [1, 0, 0, 1]]))
    if name == 'P':
        return double_ext((-1, 0), (-1, 0), H([[0, 0, 1, f_], [0, 0, 1, 1], [1, -f_, 0, 0], [-1, 1, 0, 0]]))
    if name == 'Q':
        return double_ext((-1, 0), (-1, 0), H([[0, 0, 1, 0], [1, 1, 1, 0], [-1, 0, 0, 0], [1, 0, -1, 1]]))
    if name == 'R':
        return double_ext((-1, 0), (-1, 0), H([[1, 1, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, -1, -1, 1]]))
    if name == 'S':
        return double_ext((-1, 0), (-1, 0), H([[-1, 1, 1, 1], [1, -1, 1, 1], [1, 1, -1, 1], [1, 1, 1, -1]]))
    if name == 'T':
        return double_ext((-1, 0), (-1, 0), H([[-1, 1, 1, 1], [1, -1, 1, 1], [1, 1, 1, -1], [1, 1, -1, 1]]))
    if name == 'U':
        return double_ext((-1, 0), (-1, 0), H([[-1, 1, 1, 1], [1, 1, 1, -1], [1, 1, -1, 1], [1, -1, 1, 1]]))
    if name == 'V':
        return double_ext((1, 0), (-1, 0), H([[0, 1, 1, 0], [0, 1, 0, 0], [-1, 1, 0, 0], [0, 0, 0, 1]]))
    if name == 'W':
        return double_ext((1, 0), (-1, 0), H([[0, f_, 1, 0], [1, 0, 0, -1], [1, 0, 0, f_], [0, -1, 1, 0]]))
    if name == 'X':
        return double_ext((1, 0), (-1, 0), H([[0, 0, 1, 0], [0, 0, 1, 1], [1, 0, 0, 0], [1, 1, 0, 0]]))
    if name == 'Y':
        return double_ext((1, 0), (-1, 0), H([[1, 0, 0, 0], [f_, -1, 1, 0], [0, 0, 1, 0], [1, 0, f_, -1]]))
    if name == 'Z':
        return double_ext((-1, 0), (1, 0), H([[1, 0, 0, 1], [0, 1, 1, 0], [0, f_, -1, 0], [f_, 0, 0, -1]]))
    raise ValueError(name)


def gens(A):
    return [A.gen(i) for i in range(4)]


def lin(A, a, b, i, j):
    """a*gen_i + b*gen_j."""
    return add((a, A.gen(i)), (b, A.gen(j)))


# ---------------------------------------------------------------------------
# Monomial calculus in Q(B) for a monomial datum (Appendix A).
# Plane normal form (c, j, k, eps) = c * w^j V^k v1^eps.
# ---------------------------------------------------------------------------
class Plane:
    def __init__(self, kind, q=None):
        if kind == 'balanced':
            self.al, self.s, self.be = 1, -1, 1
        elif kind == 'commutative':
            self.al, self.s, self.be = 1, 1, 1
        else:
            self.al, self.s, self.be = 1 / q, 1, q**2

    def mul(self, A, B):
        c, j, k, e = A
        c2, j2, k2, e2 = B
        c = c * c2
        if e:
            c *= self.al**j2
            j2 = self.s * j2
        c *= self.be**(-k * j2)
        j, k, e = j + j2, k + k2, e + e2
        if e == 2:
            e, k = 0, k + 1
        return (c, j, k, e)

    def letter(self, b, t):
        v1, w, winv, Vinv = (1, 0, 0, 1), (1, 1, 0, 0), (1, -1, 0, 0), (1, 0, -1, 0)
        v1inv = self.mul(Vinv, v1)
        if b == 0:
            return v1 if t == 1 else v1inv
        return self.mul(v1, w) if t == 1 else self.mul(winv, v1inv)


class Mono:
    """z_a e_b = lam[a][b] e_{sigma(b)} z_{tau(a)}; elements (coef, xword, yword)."""

    def __init__(self, lam, sigma, tau, px, py):
        self.l, self.sg, self.ta, self.px, self.py = lam, sigma, tau, px, py

    def L(self, a, b):
        return self.l[2 * a + b]

    def cross(self, a, sa, b, tb):
        sg, ta = self.sg, self.ta
        if sa == 1 and tb == 1:
            return self.L(a, b), sg[b], ta[a]
        if sa == 1:
            return 1 / self.L(ta[a], b), sg[b], ta[a]
        if tb == 1:
            return 1 / self.L(a, sg[b]), sg[b], ta[a]
        return self.L(ta[a], sg[b]), sg[b], ta[a]

    def mul(self, A, B):
        c1, x1, y1 = A
        c2, x2, y2 = B
        c, x2, newy = c1 * c2, list(x2), []
        for (a, sa) in reversed(y1):
            nx = []
            for (b, tb) in x2:
                cc, b2, a2 = self.cross(a, sa, b, tb)
                c *= cc
                nx.append((b2, tb))
                a = a2
            x2 = nx
            newy.insert(0, (a, sa))
        return (c, list(x1) + x2, newy + list(y2))

    def inv(self, A):
        c, x, y = A
        return self.mul((sp.S(1) / c, [], [(a, -t) for (a, t) in reversed(y)]),
                        (1, [(b, -t) for (b, t) in reversed(x)], []))

    def nf(self, A):
        c, x, y = A
        X = (1, 0, 0, 0)
        for (b, t) in x:
            X = self.px.mul(X, self.px.letter(b, t))
        Y = (1, 0, 0, 0)
        for (a, t) in y:
            Y = self.py.mul(Y, self.py.letter(a, t))
        return (c * X[0] * Y[0], X[1:], Y[1:])

    def conj(self, a, b):
        return self.nf(self.mul(self.mul(a, b), self.inv(a)))

    def same(self, A, B):
        """A, B normal forms; equal as elements of Q(B)."""
        return A[1:] == B[1:] and zero(A[0] - B[0])


mono_e = (1, [(0, 1)], [])
mono_z = (1, [], [(0, 1)])
mono_r = (1, [(0, -1), (1, 1)], [])
mono_X = (1, [(0, 1), (0, 1)], [])
mono_rho = (1, [], [(0, -1), (1, 1)])
mono_Y = (1, [], [(0, 1), (0, 1)])


def sc(c, el):
    return (c * el[0], el[1], el[2])


# Table 2 of the paper: z-basis, e-basis, type, lambda/h  (s^2 = f)
TYPES = {'0': ((0, 1), (0, 1)), 'I': ((1, 0), (1, 0)), 'II': ((0, 1), (1, 0)), 'III': ((1, 0), (0, 1))}


def table2(name, p=p, q=q):
    return {
        'B': ([(1, 0), (0, 1)], [(1, 0), (0, 1)], 'I', (1, 1, -1, 1)),
        'E': ([(1, 0), (0, 1)], [(1, I), (1, -I)], 'I', (1 + I, 1 - I, -1 + I, -1 - I)),
        'J': ([(1, I), (1, -I)], [(1, 0), (0, 1)], 'I', (1 + I, -1 + I, 1 - I, -1 - I)),
        'K': ([(1, 1 / s), (1, -1 / s)], [(1, 0), (0, 1)], '0', (1, s, 1, -s)),
        'L': ([(1, 0), (0, 1)], [(1, 0), (0, 1)], 'II', (s**2, 1, s**2, 1)),
        'M': ([(1, I), (1, -I)], [(1, I / s), (1, -I / s)], 'I', (I*(s+1), -I*(s-1), I*(s-1), -I*(s+1))),
        'N': ([(1, 1), (1, -1)], [(1, 0), (0, 1)], 'III', (f-g, f+g, -f-g, g-f)),
        'O': ([(1, I), (1, -I)], [(1, I*s), (1, -I*s)], 'I', (1-s, 1+s, 1+s, 1-s)),
        'P': ([(1, 0), (0, 1)], [(1, s), (1, -s)], 'II', (1+s, 1-s, 1-s, 1+s)),
        'S': ([(1, 1), (1, -1)], [(1, 1), (1, -1)], '0', (2, -2, -2, -2)),
        'T': ([(1, I), (1, -I)], [(1, 1), (1, -1)], 'II', (2*I, -2, -2*I, -2)),
        'U': ([(1, 1), (1, -1)], [(1, I), (1, -I)], 'III', (2*I, -2*I, -2, -2)),
        'W': ([(1, 1), (1, -1)], [(1, I*s), (1, -I*s)], 'III', (I*(s-I), -I*(s+I), I*(s+I), -I*(s-I))),
        'Z': ([(1, I / s), (1, -I / s)], [(1, 1), (1, -1)], 'II', (I*(s-I), -I*(s+I), -I*(s+I), I*(s-I))),
    }[name]


PLANES = {'B': ('coord', 'coord'), 'K': ('coord', 'balanced'), 'L': ('coord', 'coord'), 'E': ('balanced', 'coord'), 'M': ('balanced', 'balanced'),
          'N': ('coord', 'balanced'), 'O': ('balanced', 'balanced'), 'P': ('balanced', 'coord'),
          'S': ('balanced', 'balanced'), 'T': ('balanced', 'balanced'), 'U': ('balanced', 'balanced'),
          'W': ('commutative', 'balanced'), 'Z': ('balanced', 'commutative')}


def mono_model(name, h=h, p=p, q=q):
    zc, ec, ty, lam = table2(name, p=p)
    hh = 1 if name == 'N' else h
    lam = [hh * l for l in lam]
    sg, ta = TYPES[ty]
    kx, ky = PLANES[name]
    qx = {'B': p, 'N': -1, 'K': q, 'L': q}.get(name)
    qy = {'B': p, 'E': p, 'P': -1, 'L': -1}.get(name)
    return Mono(lam, sg, ta, Plane(kx, qx), Plane(ky, qy)), lam


# ---------------------------------------------------------------------------
# PI degree of a crossed product T' * G (Proposition 7.1) for numeric data.
# ---------------------------------------------------------------------------
def _ang(c, N):
    c = complex(c)
    k = math.atan2(c.imag, c.real) / (2 * math.pi) * N
    kr = round(k)
    return kr % N if abs(abs(c) - 1) < 1e-8 and abs(k - kr) < 1e-6 else None


def _order_group(cs):
    for N in range(1, 241):
        if all(_ang(c, N) is not None for c in cs):
            return N
    return None


def mono_pideg(M):
    """PI degree of Q(B) from the monomial model with numeric (root of unity) scalars."""
    G4 = [mono_r, mono_rho, mono_X, mono_Y]
    C = [[complex(sp.N(M.conj(a, b)[0] / M.nf(b)[0])) for b in G4] for a in G4]
    N = _order_group([c for row in C for c in row])
    K = [[_ang(c, N) for c in row] for row in C]
    sols = [v for v in itertools.product(range(N), repeat=4)
            if all(sum(K[i][j] * v[j] for j in range(4)) % N == 0 for i in range(4))]
    n2 = N**4 // len(sols)
    n = math.isqrt(n2)
    tests = sols + [tuple(N if i == j else 0 for j in range(4)) for i in range(4)]
    triv = 0
    for w in (mono_e, mono_z, M.mul(mono_e, mono_z)):
        chis, ident = [], True
        for b in G4:
            c, xp, yp = M.conj(w, b)
            t = M.nf(b)
            if (xp, yp) != t[1:]:
                ident = False
                break
            chis.append(complex(sp.N(c / t[0])))
        if not ident:
            continue
        NN = _order_group(chis)
        ks = [_ang(c, NN * N) for c in chis]
        if all(sum(k * vi for k, vi in zip(ks, v)) % (NN * N) == 0 for v in tests):
            triv += 1
    return (4 // (1 + triv)) * n


def root(k, j=1):
    return sp.exp(2 * sp.pi * sp.I * sp.Rational(j, k))


def order(c, bound=400):
    c = complex(sp.N(c))
    for k in range(1, bound):
        if abs(c**k - 1) < 1e-9:
            return k
    return None


def torus_pideg(E, N):
    """PI degree of the quantum torus with commutation exponents E (mod N)."""
    n = len(E)
    cnt = sum(1 for v in itertools.product(range(N), repeat=n)
              if all(sum(E[i][j] * v[j] for j in range(n)) % N == 0 for i in range(n)))
    return math.isqrt(N**n // cnt)


Overwriting pilib.py


In [2]:
from pilib import *
import sympy as sp, itertools, math
sp.init_printing()

# 1 Introduction

Zhang and Zhang introduced double Ore extensions in \[ZZ08\] and used them in \[ZZ09\] to build Artin--Schelter regular algebras of global dimension four. A trimmed double extension of type $(14641)$ that is not an Ore extension of a three-dimensional regular algebra lies in one of twenty-six families, $\mathbb A$ to $\mathbb Z$ \[ZZ09, Section 4\], although some members of the list are still Ore extensions \[RR\]. The introduction of \[ZZ09\] ends with questions about each algebra $B$ in the list, the first being

> \(a\) Is $B$ primitive? Does $B$ satisfy a polynomial identity? What is $\operatorname{Spec}B$?

For the algebra $B(h)$ of \[ZZ08\], which is family $\mathbb R$, they had already shown that it is PI exactly when $h$ is a root of unity \[ZZ08, Corollary 4.8\].

We settle the PI part of (a) for the whole list, compute the PI degree in all families but one, and answer the primitivity part for the PI members and for most of the others. The families have also been studied through their centers \[Rub-center\], Gröbner--Shirshov bases \[HHR\], differential smoothness \[RR\], and homological invariants and normal ordering \[Rub-homol, Rub-normal\]. For $\mathbb D$, $\mathbb G$ and $\mathbb K$ we computed the PI degree, the center and the ozone group in \[GV-D, GV-G, GV-K\], following \[CGWZ1, CGWZ2, LWZ, GY, GV-Bq, GV-Uq, GV-H\]; for centers of skew PBW extensions see \[LV-GK, LV-center, LV-fractions\].

In \[HHR, Rub-center\] the twist parameter is set to $h=1$. That is harmless for PBW bases, not here: by Table 1, no member of $\mathbb C$, $\mathbb E$, $\mathbb F$, $\mathbb I$, $\mathbb J$, $\mathbb S$, $\mathbb T$, $\mathbb U$ with $h=1$ is PI, though all eight families have PI members. So $h$ stays.

Write $\mu_\infty$ for the group of roots of unity in $\Bbbk$, and $s$ for a square root of $f$ whenever a family has a parameter $f$; the conditions below do not depend on the choice of $s$, and $i$ is a fixed square root of $-1$.

**Theorem 1.1.** Let $\Bbbk$ be algebraically closed of characteristic zero, and let $B$ be an algebra in one of the twenty-six families of \[ZZ09, Section 4\], with admissible parameters. Then $B$ is PI if and only if the condition in Table 1 holds.

|                Family                 |        Parameters        |       $B$ is PI if and only if       |            Proof             |
|:-------------------------------------:|:------------------------:|:------------------------------------:|:----------------------------:|
|              $\mathbb A$              |        $P=(1,1)$         |                never                 |          Prop. 3.1           |
|              $\mathbb B$              |         $p^2=-1$         |           $h\in\mu_\infty$           |           Thm. 4.4           |
|              $\mathbb C$              |       $p^2+p+1=0$        |         $3h^2\in\mu_\infty$          |           Thm. 6.5           |
|              $\mathbb D$              |         $p\ne0$          |  $p,h\in\mu_\infty$ and $p\ne\pm1$   |     Prop. 3.2, Thm. 5.3      |
|       $\mathbb E$, $\mathbb J$        | $p^2=-1$, resp. $q^2=-1$ |         $2h^2\in\mu_\infty$          |           Thm. 4.4           |
|       $\mathbb F$, $\mathbb I$        | $p^2=-1$, resp. $q^2=-1$ |         $2h^2\in\mu_\infty$          |        Thms. 6.3, 6.4        |
|              $\mathbb G$              |  $p\ne0,\pm1$, $f\ne0$   |          $p,h\in\mu_\infty$          |           Thm. 5.3           |
|              $\mathbb H$              |        $Q=(1,1)$         |                never                 |          Prop. 3.1           |
|       $\mathbb K$, $\mathbb L$        |     $q\ne0$, $f\ne0$     |         $q,f,h\in\mu_\infty$         |           Thm. 4.4           |
|              $\mathbb M$              |         $f\ne1$          | $f\ne0$ and $h(s\pm1)\in\mu_\infty$  |     Thm. 4.4, Prop. 3.2      |
|              $\mathbb N$              |       $f^2\ne g^2$       |        $f\pm g\in\mu_\infty$         |           Thm. 4.4           |
|       $\mathbb O$, $\mathbb P$        |         $f\ne1$          | $f\ne0$ and $h(1\pm s)\in\mu_\infty$ |     Thm. 4.4, Prop. 3.2      |
|       $\mathbb Q$, $\mathbb V$        |           ---            |           $h\in\mu_\infty$           |          Prop. 5.4           |
|              $\mathbb R$              |           ---            |           $h\in\mu_\infty$           | \[ZZ08, Cor. 4.8\], Thm. 8.6 |
| $\mathbb S$, $\mathbb T$, $\mathbb U$ |           ---            |          $2h\in\mu_\infty$           |           Thm. 4.4           |
|              $\mathbb W$              |         $f\ne-1$         | $f\ne0$ and $h(s\pm i)\in\mu_\infty$ |     Thm. 4.4, Prop. 3.2      |
|              $\mathbb X$              |           ---            |                never                 |          Prop. 3.2           |
|              $\mathbb Y$              |       $f\in\Bbbk$        |                never                 |          Prop. 3.2           |
|              $\mathbb Z$              |       $f(1+f)\ne0$       |       $h(s\pm i)\in\mu_\infty$       |           Thm. 4.4           |

*Table 1. The PI criterion. The second column lists the parameters as in \[ZZ09, Section 4\]; $h\in\Bbbk^\times$ is the twist parameter.*

The families $\mathbb M$, $\mathbb O$, $\mathbb P$, $\mathbb W$ allow $f=0$, where the monomial description breaks down. Those members are never PI, like $\mathbb D(\pm1,h)$; in fact $\mathbb O(0,h)=\mathbb D(-1,h)$. The twist only enters through its class in $\Bbbk^\times/\mu_\infty$:

**Theorem 1.2.** Let $B(h)$ be the double extension with mixed matrix $h\Sigma_0$, for a fixed $\Sigma_0$ of type $(14641)$. If $\zeta\in\mu_\infty$, then $B(h)$ is PI if and only if $B(h\zeta)$ is PI.

This is Theorem 2.5. A Zhang twist does not preserve the PI property in general ($\Bbbk_q[x,y]$ is a twist of $\Bbbk[x,y]$); a twist by a root of unity does.

For the PI degree we write $Q(B)$ as a crossed product over a quantum torus. If $B$ has a monomial datum whose bases are balanced (equal squares), lie in a commutative plane or are coordinate bases, the torus has rank four and the group order at most four (Proposition 7.1). This gives closed formulas for twelve families (Theorem 7.2), for instance $$\begin{gathered}
\operatorname{PIdeg}\mathbb S(h)=\operatorname{PIdeg}\mathbb T(h)=\operatorname{PIdeg}\mathbb U(h)=8\operatorname{ord}(16h^4),\\
\operatorname{PIdeg}\mathbb E(h)=\operatorname{PIdeg}\mathbb J(h)=8\operatorname{ord}(4h^4),\qquad \operatorname{PIdeg}\mathbb B(p,h)=8\operatorname{ord}(h^4).
\end{gathered}$$ The families $\mathbb F$, $\mathbb I$, $\mathbb Q$, $\mathbb R$, $\mathbb V$ have no monomial datum, but elements of bidegree $(1,1)$ or $(1,2)$ normalizing one of the planes do the same job.

**Theorem 1.3.** For the PI members of the families below, $$\begin{aligned}
\operatorname{PIdeg}\mathbb Q(h)&=\operatorname{PIdeg}\mathbb R(h)=\operatorname{PIdeg}\mathbb V(h)=4\operatorname{ord}(h^2),\\
\operatorname{PIdeg}\mathbb F(p,h)&=\operatorname{PIdeg}\mathbb I(q,h)=8\operatorname{ord}(-4h^4).
\end{aligned}$$

This is Theorem 8.10. With $\mathbb D$, $\mathbb G$, $\mathbb K$ \[GV-D, GV-G, GV-K\] and $\mathbb L$ (Theorem 5.5) this makes twenty-one families. For $\mathbb C$ we only have bounds (Proposition 9.1). Table 3 collects the formulas.

No connected graded PI algebra with nonzero degree-one part is primitive (Proposition 11.1). For the non-PI members the answer is complete in twenty-two families, except for $\mathbb D(\pm1,h)$, $\mathbb M(0,h)$, $\mathbb P(0,h)$ and $\mathbb W(0,h)$. In $\mathbb B$, $\mathbb C$, $\mathbb E$, $\mathbb F$, $\mathbb I$, $\mathbb J$, $\mathbb K$, $\mathbb L$, $\mathbb Q$, $\mathbb R$, $\mathbb S$, $\mathbb T$, $\mathbb U$, $\mathbb V$ the quotient division ring has a central element that does not depend on the twist, such as $(x_1^4+x_2^4)^2/(x_1^4x_2^4)$ in $\mathbb B$, so no member is primitive (Propositions 11.3, 11.4 and Theorem 11.16). A member of $\mathbb M$, $\mathbb N$, $\mathbb O$, $\mathbb P$, $\mathbb W$, $\mathbb Z$ is primitive exactly when the scalars $u,v$ of Theorem 7.2 are not roots of unity (Theorem 11.16).

In $\mathbb D$ and $\mathbb G$ put $q=hp$. The center of the quotient division ring has transcendence degree $4$, $2$ or $0$ according as both, one or none of $p,q$ are roots of unity (Theorem 11.6), so a non-PI member with $p$ or $q$ in $\mu_\infty$ is not primitive; when $q\in\mu_\infty$ and $p\notin\mu_\infty$ there is even a central element of positive degree in $B$. If neither is a root of unity, the member is primitive (Theorem 11.13). The proof goes through an algebra $A$ on $x_1^{-1}x_2,y_1,y_2$ in which every nonzero graded ideal contains a power of a central element $\Omega$.

In \[ZZ08, Corollary 4.8\], $B(h)$ is PI exactly when $\det\sigma$ has finite order. On the whole list PI still forces finite order, but the converse fails for $\mathbb Y$, for $\mathbb D(\pm1,h)$ and for generic members of $\mathbb O$ (Remark 10.2).

Except for two scalars quoted from \[ZZ09, Section 5\] in the proof of Theorem 6.5, every computation is done by hand from the relations in \[ZZ09, Section 4\].

# 2 Preliminaries

Throughout, $\Bbbk$ is algebraically closed of characteristic zero.

## 2.1 The families

We follow \[ZZ09, Sections 3 and 4\]. A trimmed double extension $B=(\Bbbk_Q[x_1,x_2])_P[y_1,y_2;\sigma]$ is generated by $x_1,x_2,y_1,y_2$ with $$\begin{aligned}
x_2x_1&=q_{12}x_1x_2+q_{11}x_1^2, & y_2y_1&=p_{12}y_1y_2+p_{11}y_1^2,
\end{aligned}$$ where $Q=(q_{12},q_{11})$ and $P=(p_{12},p_{11})$, and four mixed relations $$\begin{pmatrix}y_1x_1\\ y_1x_2\\ y_2x_1\\ y_2x_2\end{pmatrix}=\Sigma\begin{pmatrix}x_1y_1\\ x_2y_1\\ x_1y_2\\ x_2y_2\end{pmatrix}.$$ In every family except $\mathbb N$, $\Sigma=h\Sigma_0$ with $\Sigma_0$ the matrix of \[ZZ09, Section 4\] and $h\in\Bbbk^\times$; in $\mathbb N$ the matrix depends on $f,g$ and a twist only rescales them. We write $A_x=\Bbbk_Q[x_1,x_2]$ and $A_y=\Bbbk_P[y_1,y_2]$ for the two planes, and $\Bbbk_q[x_1,x_2]$ for the quantum plane with $x_2x_1=qx_1x_2$.

By \[ZZ09, Theorem 0.1\] every $B$ in the list is a strongly noetherian, Auslander regular and Cohen--Macaulay domain, AS-regular of global dimension four, and the ordered monomials $x_1^ax_2^by_1^cy_2^d$ form a basis. It is $\mathbb Z^2$-graded with $\deg x_i=(1,0)$ and $\deg y_i=(0,1)$. Multiplication gives an isomorphism of vector spaces $A_x\otimes A_y\to B$; in particular, if $a_1,\dots,a_r\in A_x$ and $b_1,\dots,b_t\in A_y$ are linearly independent, so are the products $a_jb_l$.

**Lemma 2.1.** Let $a\in A_x$ and $b\in A_y$ be homogeneous of positive degree with $ba=\mu\,ab$ for some $\mu\in\Bbbk^\times$. If $B$ is PI, then $\mu\in\mu_\infty$.

*Proof.* The powers of $a$ have different degrees, so they are linearly independent, and the same holds for $b$. So the monomials $a^jb^l$ are independent and $\Bbbk\langle a,b\rangle$ is the quantum plane $\Bbbk_\mu[a,b]$, which is PI only when $\mu\in\mu_\infty$. $\square$

## 2.2 Three general lemmas

**Lemma 2.2.** Let $R$ be a domain and $S\subseteq R$ a subring which is a right noetherian PI domain, with $R$ finitely generated as a right $S$-module. Then $R$ is PI. If $d=\dim_{Q(S)}R\otimes_SQ(S)$, then $\operatorname{PIdeg}R\le d\,\operatorname{PIdeg}S$. The same holds with right replaced by left.

*Proof.* $S$ is a right Ore domain, and by Posner's theorem $Q(S)$ is a division algebra of dimension $m^2$ over its center $Z$, with $m=\operatorname{PIdeg}S$ \[MR, Theorem 13.6.5\]. Since $R$ is a domain it is torsion-free over $S$, so $R\to V=R\otimes_SQ(S)$ is injective, and $V$ is a right $Q(S)$-space of dimension $d<\infty$. Left multiplication gives a ring homomorphism $R\to\operatorname{End}_{Q(S)}(V)\cong M_d(Q(S))$, injective because $b\otimes1\ne0$ for $b\ne0$. If $L$ is a splitting field of $Q(S)$ over $Z$, then $M_d(Q(S))\subseteq M_{dm}(L)$, so $R$ satisfies the identities of $M_{dm}(L)$. $\square$

**Lemma 2.3.** Let $D$ be a division algebra finite-dimensional over its center $F\supseteq\Bbbk$, and let $t,w\in D$, $t\notin\Bbbk$, $w\ne0$, with $wtw^{-1}=\mu t+c$, $\mu\in\Bbbk^\times$, $c\in\Bbbk$. Then the affine map $X\mapsto\mu X+c$ has finite order. In particular, if $c=0$ then $\mu\in\mu_\infty$.

*Proof.* $t$ is algebraic over $F$; let $m\in F[X]$ be its minimal polynomial. Conjugation by $w^n$ fixes $F$, so $m$ vanishes at $w^ntw^{-n}=\mu^nt+c_n$ for every $n$, and these elements lie in the field $F(t)$. If the affine map had infinite order, each nontrivial power would have at most one fixed point, which lies in $\Bbbk$, while $t\notin\Bbbk$. So the elements $\mu^nt+c_n$ would be pairwise distinct, and $m$ would have infinitely many roots in a field. $\square$

**Lemma 2.4.** Let $D$ be as in Lemma 2.3, $t\in D\setminus\Bbbk$, and $y_1,y_2\in D$ linearly independent over $\Bbbk$ with $$y_jt=a\,ty_j+\sum_{l=1}^2C_{jl}y_l\qquad(j=1,2),\qquad a\in\{1,-1\},\quad C\in M_2(\Bbbk).$$ If $a=1$ and $C$ has a nonzero eigenvalue, or $a=-1$ and $C$ has two distinct eigenvalues, we reach a contradiction.

*Proof.* If $vC=\lambda v$, then $z=v_1y_1+v_2y_2\ne0$ satisfies $zt=(at+\lambda)z$. For $a=1$ and $\lambda\ne0$ this is a translation of infinite order, against Lemma 2.3. For $a=-1$, take $z_1,z_2$ with eigenvalues $\lambda_1\ne\lambda_2$ and put $t_0=t-\lambda_2/2$, $\delta=(\lambda_1-\lambda_2)/2$. Then $z_2t_0=-t_0z_2$, $z_1t_0=(-t_0+2\delta)z_1$, and $w=z_1z_2$ satisfies $wt_0=(t_0-2\delta)w$, again a translation. $\square$

## 2.3 Twisting

Fix $\Sigma_0$ and write $B(h)$ for the algebra with mixed matrix $h\Sigma_0$. For bihomogeneous $a,b\in B(1)$ of bidegrees $(k_a,j_a)$ and $(k_b,j_b)$ put $$a*_hb=h^{j_ak_b}\,ab .$$ The exponent is bilinear in the bidegrees, so $*_h$ is associative, and the plane relations do not change while $y*_hx=h\,yx$. Hence $(B(1),*_h)\cong B(h)$; this is \[ZZ09, Lemma 3.2\]. Two consequences are used often. Let $N,N'\in B(1)$ have bidegrees $(0,d)$ and $(d,0)$. Then $$\qquad(2.1)
\begin{aligned}
Nx_i&=c\,x_iN\ \text{ in }B(1) &&\Longrightarrow && Nx_i&=c\,h^d\,x_iN\ \text{ in }B(h),\\
y_jN'&=c'N'y_j\ \text{ in }B(1) &&\Longrightarrow && y_jN'&=c'h^{d}N'y_j\ \text{ in }B(h).
\end{aligned}$$ And composing twists, $(B(h),*_\zeta)\cong B(h\zeta)$.

**Theorem 2.5.** Let $\zeta\in\Bbbk^\times$ be a root of unity of order $m$. Then $B(h)$ is PI if and only if $B(h\zeta)$ is PI.

*Proof.* Let $S=\bigoplus_{m\mid k}B(h)_{(k,j)}$, the span of the bihomogeneous components whose $x$-degree is a multiple of $m$. If $b\in S$, then $a*_\zeta b=\zeta^{j_ak_b}ab=ab$. So $S$ is a subalgebra of $B(h)$ and of $(B(h),*_\zeta)\cong B(h\zeta)$, with the same multiplication.

Every basis monomial $x_1^ax_2^by_1^cy_2^d$ can be written as $x^{\alpha'}\cdot(x^{\alpha''}y_1^cy_2^d)$, where $x^{\alpha'}$ consists of the first $l$ letters of $x_1^ax_2^b$, $l$ is the residue of $a+b$ modulo $m$, and the second factor lies in $S$. So $B(h)=\sum_{\alpha'}x^{\alpha'}S$, with $x^{\alpha'}$ running over the finitely many monomials of degree less than $m$. Since $x^{\alpha'}$ has $y$-degree zero, $x^{\alpha'}*_\zeta c=x^{\alpha'}c$, and the same finite set generates $B(h\zeta)$ as a right $S$-module.

$S$ is right noetherian: for a right ideal $I$ of $S$, the component of $IB(h)$ in $x$-degrees divisible by $m$ is $IS=I$, so $I\mapsto IB(h)$ is injective on right ideals and preserves inclusions. It is a domain, being inside $B(h)$. If $B(h)$ is PI, so is $S$, and Lemma 2.2 shows that the domain $B(h\zeta)$ is PI. The converse is the same argument with $\zeta^{-1}$. $\square$

So in each family the values of $h$ giving PI members form a union of cosets of $\mu_\infty$.

# 3 Members that are never PI

**Proposition 3.1.** The families $\mathbb A$ and $\mathbb H$ are never PI.

*Proof.* For $\mathbb A$, $P=(1,1)$, and for $\mathbb H$, $Q=(1,1)$. So $B$ contains a Jordan plane, which is not PI in characteristic zero. $\square$

**Proposition 3.2.** The following algebras are not PI, for any value of the remaining parameters: $$\mathbb X(h),\quad\mathbb Y(f,h),\quad\mathbb D(\pm1,h),\quad\mathbb M(0,h),\quad\mathbb O(0,h),\quad\mathbb P(0,h),\quad\mathbb W(0,h).$$ Moreover $\mathbb O(0,h)=\mathbb D(-1,h)$.

*Proof.* For $f=0$ the matrices of $\mathbb O$ and of $\mathbb D$ at $p=-1$ are both $$h\begin{pmatrix}1&0&0&0\\0&-1&1&0\\0&0&-1&0\\1&0&0&1\end{pmatrix},$$ with $P=Q=(-1,0)$ in both cases. This proves the last claim.

In each of the six remaining cases one generator $n\in\{x_1,x_2\}$ is normal. Reading off the mixed relations, $$\begin{array}{c|c|c}
 & n & \text{relations of }n\text{ with }y_1,y_2\\\hline
\mathbb X & x_1 & y_1x_1=h\,x_1y_2,\ \ y_2x_1=h\,x_1y_1\\
\mathbb Y & x_1 & y_1x_1=h\,x_1y_1,\ \ y_2x_1=h\,x_1y_2\\
\mathbb D(p,h) & x_1 & y_1x_1=-hp\,x_1y_1,\ \ y_2x_1=hp\,x_1y_2\\
\mathbb M(0,h) & x_2 & y_1x_2=-h\,x_2y_2,\ \ y_2x_2=-h\,x_2y_1\\
\mathbb P(0,h) & x_1 & y_1x_1=h\,x_1y_2,\ \ y_2x_1=h\,x_1y_1\\
\mathbb W(0,h) & x_1 & y_1x_1=h\,x_1y_2,\ \ y_2x_1=h\,x_1y_1
\end{array}$$ and $n$ $q$-commutes with the other $x$-generator. Let $x'$ be that generator and $t=n^{-1}x'$ in the quotient division ring $Q(B)$; $t$ is transcendental over $\Bbbk$. Using the remaining mixed relations, $h$ cancels and $$\begin{array}{c|c|c|c}
 & y_1t & y_2t & (a,C)\\\hline
\mathbb X & ty_1+y_1 & ty_2+y_2 & (1,\ I)\\
\mathbb Y & -ty_1+fy_1+y_2 & -ty_2+y_1+fy_2 & \bigl(-1,\ \left(\begin{smallmatrix}f&1\\1&f\end{smallmatrix}\right)\bigr)\\
\mathbb D(\pm1,h) & p\,ty_1-p^{-1}y_2 & p^{-1}ty_2+p^{-1}y_1 & \bigl(p,\ p\left(\begin{smallmatrix}0&-1\\1&0\end{smallmatrix}\right)\bigr)\\
\mathbb M(0,h) & -ty_1+y_2 & -ty_2-y_1 & \bigl(-1,\ \left(\begin{smallmatrix}0&1\\-1&0\end{smallmatrix}\right)\bigr)\\
\mathbb P(0,h) & ty_1-y_1 & ty_2+y_2 & \bigl(1,\ \left(\begin{smallmatrix}-1&0\\0&1\end{smallmatrix}\right)\bigr)\\
\mathbb W(0,h) & -ty_1+y_2 & -ty_2+y_1 & \bigl(-1,\ \left(\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right)\bigr)
\end{array}$$ As an example take $\mathbb W(0,h)$, where $y_1x_2=h(x_1y_1-x_2y_2)$ and $y_2x_2=h(x_1y_2-x_2y_1)$. From $y_2x_1=hx_1y_1$ we get $y_1x_1^{-1}=h^{-1}x_1^{-1}y_2$, so $$\begin{aligned}
y_1t&=h^{-1}x_1^{-1}y_2x_2=x_1^{-1}(x_1y_2-x_2y_1)=y_2-ty_1,
\end{aligned}$$ and $y_2t=y_1-ty_2$ in the same way. In every row, $C$ has a nonzero eigenvalue when $a=1$, and two distinct eigenvalues when $a=-1$: they are $f\pm1$ for $\mathbb Y$, $\pm ip$ for $\mathbb D$, $\pm i$ for $\mathbb M$ and $\pm1$ for $\mathbb W$. If $B$ were PI, $Q(B)$ would be finite-dimensional over its center by Posner's theorem, and Lemma 2.4 gives a contradiction. $\square$

In [3]:
# Proposition 3.2: the table of relations y_j t = a t y_j + sum_l C_jl y_l, checked in B
# through the equivalent identity  c_k^{-1} y_k x' = a x' y_j + sum_l C_jl n y_l,  where y_k n = c_k n y_j.
def check_row(name, A, n, xp, a, C):
    x1, x2, y1, y2 = gens(A); ys = [y1, y2]
    ok = True
    for j in range(2):
        found = None
        for k in range(2):
            L, R = A.mul(ys[k], n), A.mul(n, ys[j])
            c = ratio(L, R)
            if c is not None and not zero(c) and is_zero(add((1, L), (-c, R))):
                found = (k, c)
        k, c = found
        lhs = add((1 / c, A.mul(ys[k], xp)))
        rhs = add((a, A.mul(xp, ys[j])), (C[j][0], A.mul(n, y1)), (C[j][1], A.mul(n, y2)))
        ok &= is_zero(add((1, lhs), (-1, rhs)))
    return check(name, ok)

for name, par, ni, xi, a, C in [
        ('X', {}, 0, 1, 1, [[1, 0], [0, 1]]),
        ('Y', {}, 0, 1, -1, [[f, 1], [1, f]]),
        ('D', {'p': 1}, 0, 1, 1, [[0, -1], [1, 0]]),
        ('D', {'p': -1}, 0, 1, -1, [[0, 1], [-1, 0]]),
        ('M', {'f': 0}, 1, 0, -1, [[0, 1], [-1, 0]]),
        ('P', {'f': 0}, 0, 1, 1, [[-1, 0], [0, 1]]),
        ('W', {'f': 0}, 0, 1, -1, [[0, 1], [1, 0]])]:
    A = family(name, **par); G = gens(A)
    check_row(f'{name}{par}', A, G[ni], G[xi], a, C)

check('O(0,h) = D(-1,h)', family('O', f=0).rules == family('D', p=-1).rules)

  ok   X{}
  ok   Y{}
  ok   D{'p': 1}
  ok   D{'p': -1}
  ok   M{'f': 0}
  ok   P{'f': 0}
  ok   W{'f': 0}
  ok   O(0,h) = D(-1,h)


True

# 4 The monomial criterion

## 4.1 Monomial data

**Definition 4.1.** A *monomial datum* for $B$ consists of bases $z_1,z_2$ of $\Bbbk y_1\oplus\Bbbk y_2$ and $e_1,e_2$ of $\Bbbk x_1\oplus\Bbbk x_2$, permutations $\sigma,\tau$ of $\{1,2\}$ and scalars $\lambda_{ab}\in\Bbbk^\times$ with $$\tag{4.1}
z_ae_b=\lambda_{ab}\,e_{\sigma(b)}z_{\tau(a)}\qquad(a,b\in\{1,2\}).$$ It is of type $0$, I, II or III when neither, both, only $\tau$, or only $\sigma$ is the transposition. We call it *admissible* if each plane is $\Bbbk_{\pm1}$ (any basis), or the chosen basis of that plane is a coordinate basis of a quantum plane $\Bbbk_q$.

Put $$\begin{aligned}
\mathcal I_0&=\{\lambda_{11},\lambda_{12},\lambda_{21},\lambda_{22}\}, & \mathcal I_{\rm I}&=\{\lambda_{11}\lambda_{22},\ \lambda_{12}\lambda_{21}\},\\
\mathcal I_{\rm II}&=\Bigl\{\lambda_{11}\lambda_{21},\ \lambda_{12}\lambda_{22},\ \frac{\lambda_{11}}{\lambda_{12}},\ \frac{\lambda_{21}}{\lambda_{22}}\Bigr\}, & \mathcal I_{\rm III}&=\Bigl\{\lambda_{11}\lambda_{12},\ \lambda_{21}\lambda_{22},\ \frac{\lambda_{11}}{\lambda_{21}},\ \frac{\lambda_{12}}{\lambda_{22}}\Bigr\}.
\end{aligned}$$

**Lemma 4.2.** Under $z_a\mapsto d_az_a$, $e_b\mapsto c_be_b$ the scalars become $\lambda'_{ab}=\lambda_{ab}\,d_ac_b/(d_{\tau(a)}c_{\sigma(b)})$, and the set $\mathcal I$ of the corresponding type does not change. One can choose $d_a,c_b$ so that every $\lambda'_{ab}$ lies in the subgroup generated by $\mathcal I$, up to a square root.

*Proof.* The formula follows from (4.1). In type II put $r=d_1/d_2$, so $\lambda'_{1b}=r\lambda_{1b}$ and $\lambda'_{2b}=r^{-1}\lambda_{2b}$; with $r=\lambda_{11}^{-1}$ the new scalars are $1$, $\lambda_{12}/\lambda_{11}$, $\lambda_{11}\lambda_{21}$ and $\lambda_{12}\lambda_{22}\cdot\lambda_{11}/\lambda_{12}$. In type I put also $u=c_1/c_2$; then $\lambda'_{11}=ru\lambda_{11}$, $\lambda'_{12}=ru^{-1}\lambda_{12}$, $\lambda'_{21}=r^{-1}u\lambda_{21}$, $\lambda'_{22}=(ru)^{-1}\lambda_{22}$, and choosing $ru=\lambda_{11}^{-1}$, $ru^{-1}=\lambda_{12}^{-1}$ (a square root is needed) gives $1,1,\lambda_{12}\lambda_{21},\lambda_{11}\lambda_{22}$. Type III is type II with the roles of the planes exchanged, and type $0$ needs nothing. $\square$

**Lemma 4.3.** Let $X=\Bbbk_\varepsilon[x_1,x_2]$, $\varepsilon=\pm1$, let $e_1,e_2$ be a basis of $X_1$ and $u=e_1^2$, $v=e_2^2$, $a=e_1e_2$, $a'=e_2e_1$. Then $u,v,a+a'$ and $aa'$ are central in $X$, $a$ and $a'$ commute, $\Bbbk[x_1^2,x_2^2]=\Bbbk[u,v,a+a']$, and for every even $n$ the algebra $X$ is a finite module over $\Bbbk[u^n+v^n,\ (uv)^n,\ a^n+(a')^n]$, which is central in $X$.

*Proof.* For $\varepsilon=1$ everything is commutative. Let $\varepsilon=-1$ and $e_b=\alpha_bx_1+\beta_bx_2$. Squares of linear forms are central, $u=\alpha_1^2x_1^2+\beta_1^2x_2^2$, $v=\alpha_2^2x_1^2+\beta_2^2x_2^2$, and $a+a'=2(\alpha_1\alpha_2x_1^2+\beta_1\beta_2x_2^2)$. These three elements span $\Bbbk x_1^2+\Bbbk x_2^2$. Indeed, if the vectors $(\alpha_1^2,\beta_1^2)$, $(\alpha_2^2,\beta_2^2)$, $(\alpha_1\alpha_2,\beta_1\beta_2)$ lay on a line, comparing the third with the first two would give $\alpha_1\beta_1(\alpha_1\beta_2-\alpha_2\beta_1)=\alpha_2\beta_2(\alpha_1\beta_2-\alpha_2\beta_1)=0$, so $\alpha_1\beta_1=\alpha_2\beta_2=0$; then $e_1,e_2$ are multiples of $x_1,x_2$ in some order, and $u,v$ are independent. So $\Bbbk[u,v,a+a']=\Bbbk[x_1^2,x_2^2]$. Next $aa'=e_1v e_1=uv=a'a$. Write $a=m+w$, $a'=m-w$ with $m$ central and $w$ a multiple of $x_1x_2$; then $a^n+(a')^n$ only contains even powers of $w$, and $w^2$ is central. Finally $u^n,v^n$ are roots of $T^2-(u^n+v^n)T+(uv)^n$, and since $a^n+(a')^n$ is a monic polynomial of degree $n$ in $a+a'$ with coefficients in $\Bbbk[aa']=\Bbbk[uv]$, the element $a+a'$ is integral over $\Bbbk[u^n+v^n,(uv)^n,a^n+(a')^n]$. $\square$

**Theorem 4.4.** Let $B$ have an admissible monomial datum. Then $B$ is PI if and only if the parameters of the two planes are roots of unity and $\mathcal I\subseteq\mu_\infty$, where $\mathcal I$ is the set of the corresponding type.

*Proof.* The planes are subalgebras, so their parameters must be roots of unity.

*Necessity, products.* From (4.1), $z_a^2e_b=\lambda_{ab}\lambda_{a,\sigma(b)}\,e_bz_{\tau(a)}^2$ in types $0$ and I, and $z_a^2e_b=\lambda_{ab}^2\,e_bz_{\tau(a)}^2$ in type II. The squares $z_1^2,z_2^2$ commute up to a root of unity $\gamma$ (they are central in $\Bbbk_{\pm1}$, and in a coordinate basis of $\Bbbk_q$, $\gamma=q^{\pm4}$). So $$(z_1^2z_2^2)\,e_b=\gamma\,\Lambda_b\,e_b\,(z_1^2z_2^2),\qquad \Lambda_b=\begin{cases}\lambda_{11}\lambda_{12}\lambda_{21}\lambda_{22}&\text{type I},\\ (\lambda_{1b}\lambda_{2b})^2&\text{types }0,\ \mathrm{II},\end{cases}$$ and Lemma 2.1 gives $\Lambda_b\in\mu_\infty$. In type $0$, Lemma 2.1 applied to $e_b$ and $z_a$ gives $\lambda_{ab}\in\mu_\infty$ directly.

*Necessity, ratios.* Work in $Q(B)$. In type II, the relation $z_{\tau(a)}e_1=\lambda_{\tau(a),1}e_1z_a$ gives $z_ae_1^{-1}=\lambda_{\tau(a),1}^{-1}e_1^{-1}z_{\tau(a)}$, so for $t=e_1^{-1}e_2$, $$z_atz_a^{-1}=\frac{\lambda_{\tau(a),2}}{\lambda_{\tau(a),1}}\,t .$$ In type I the same computation with $t'=e_2^{-1}e_1$ gives $z_at'z_a^{-1}=c_a\,t'^{-1}$ with $c_1=\lambda_{21}/\lambda_{22}$ and $c_2=\lambda_{11}/\lambda_{12}$, hence $$(z_1z_2)\,t'\,(z_1z_2)^{-1}=\frac{c_2}{c_1}\,t'=\frac{\lambda_{11}\lambda_{22}}{\lambda_{12}\lambda_{21}}\,t' .$$ Both $t$ and $t'$ lie outside $\Bbbk$, and Lemma 2.3 shows that these ratios are roots of unity. Together with the products this gives $\mathcal I\subseteq\mu_\infty$ in types I and II; type III follows by exchanging the roles of the planes, that is, by using the relations $e_cz_d=\lambda_{\tau(d),\sigma(c)}^{-1}z_{\tau(d)}e_{\sigma(c)}$ obtained from (4.1).

*Sufficiency.* By Lemma 4.2 we may assume all $\lambda_{ab}\in\mu_\infty$. Choose a positive even integer $n$ such that $n/2$ is divisible by the orders of all $\lambda_{ab}$, of all products of two of them, and of the plane parameters. From (4.1), $$z_ae_b^n=(\lambda_{ab}\lambda_{\tau(a),b})^{n/2}\,e_{\sigma(b)}^n\,z_a=e_{\sigma(b)}^nz_a,\qquad z_a\,e_1e_2=\kappa_a\,e_{\sigma(1)}e_{\sigma(2)}\,z_a,\quad\kappa_a=\lambda_{a1}\lambda_{\tau(a),2}\in\mu_\infty,$$ and enlarging $n$ we may assume $\kappa_a^n=1$, so that $z_a(e_1e_2)^n=(e_{\sigma(1)}e_{\sigma(2)})^nz_a$. So conjugation by each $z_a$ permutes $\{e_1^n,e_2^n\}$ and $\{(e_1e_2)^n,(e_2e_1)^n\}$. If $A_x=\Bbbk_{\pm1}$ put $$Z_x=\Bbbk\bigl[e_1^{2n}+e_2^{2n},\ (e_1e_2)^{n}+(e_2e_1)^{n},\ e_1^{2n}e_2^{2n}\bigr],$$ and if $e_1,e_2$ is a coordinate basis of $\Bbbk_q$ put $Z_x=\Bbbk[e_1^n+e_2^n,\ e_1^ne_2^n]$. In both cases $Z_x$ commutes with $A_x$ and with $z_1,z_2$, so it is central in $B$, and $A_x$ is a finite $Z_x$-module, by Lemma 4.3 in the first case and clearly in the second. Rewriting (4.1) as $e_cz_d=\lambda_{\tau(d),\sigma(c)}^{-1}z_{\tau(d)}e_{\sigma(c)}$, the same construction on the other side gives a central $Z_y$ over which $A_y$ is finite. Since $B=A_xA_y$, the algebra $B$ is a finite module over the commutative central subalgebra $Z_xZ_y$, hence PI. $\square$

## 4.2 The fourteen monomial families

**Corollary 4.5.** The families $\mathbb B$, $\mathbb E$, $\mathbb J$, $\mathbb K$, $\mathbb L$, $\mathbb N$, $\mathbb S$, $\mathbb T$, $\mathbb U$, $\mathbb Z$, and $\mathbb M$, $\mathbb O$, $\mathbb P$, $\mathbb W$ for $f\ne0$, have the admissible monomial data of Table 2, and satisfy the criteria of Table 1.

|             |      $z_1,z_2$      |      $e_1,e_2$      | type | $(\lambda_{11},\lambda_{12},\lambda_{21},\lambda_{22})$ | $\mathcal I\subseteq\mu_\infty$ iff |
|:-----------:|:-------------------:|:-------------------:|:----:|:-------------------------------------------------------:|:-----------------------------------:|
| $\mathbb B$ |     $y_1,\ y_2$     |     $x_1,\ x_2$     |  I   |                      $(1,1,-1,1)$                       |          $h\in\mu_\infty$           |
| $\mathbb E$ |     $y_1,\ y_2$     |    $x_1\pm ix_2$    |  I   |               $(1+i,\ 1-i,\ -1+i,\ -1-i)$               |         $2h^2\in\mu_\infty$         |
| $\mathbb J$ |    $y_1\pm iy_2$    |     $x_1,\ x_2$     |  I   |               $(1+i,\ -1+i,\ 1-i,\ -1-i)$               |         $2h^2\in\mu_\infty$         |
| $\mathbb K$ | $y_1\pm s^{-1}y_2$  |     $x_1,\ x_2$     |  0   |                   $(1,\ s,\ 1,\ -s)$                    |         $f,h\in\mu_\infty$          |
| $\mathbb L$ |     $y_1,\ y_2$     |     $x_1,\ x_2$     |  II  |                    $(f,\ 1,\ f,\ 1)$                    |         $f,h\in\mu_\infty$          |
| $\mathbb M$ |    $y_1\pm iy_2$    | $x_1\pm is^{-1}x_2$ |  I   |         $(i(s+1),\ -i(s-1),\ i(s-1),\ -i(s+1))$         |       $h(s\pm1)\in\mu_\infty$       |
| $\mathbb N$ |    $y_1\pm y_2$     |     $x_1,\ x_2$     | III  |               $(f-g,\ f+g,\ -f-g,\ g-f)$                |        $f\pm g\in\mu_\infty$        |
| $\mathbb O$ |    $y_1\pm iy_2$    |   $x_1\pm isx_2$    |  I   |                $(1-s,\ 1+s,\ 1+s,\ 1-s)$                |      $h(1\pm s)\in\mu_\infty$       |
| $\mathbb P$ |     $y_1,\ y_2$     |    $x_1\pm sx_2$    |  II  |                $(1+s,\ 1-s,\ 1-s,\ 1+s)$                |      $h(1\pm s)\in\mu_\infty$       |
| $\mathbb S$ |    $y_1\pm y_2$     |    $x_1\pm x_2$     |  0   |                     $(2,-2,-2,-2)$                      |          $2h\in\mu_\infty$          |
| $\mathbb T$ |    $y_1\pm iy_2$    |    $x_1\pm x_2$     |  II  |                 $(2i,\ -2,\ -2i,\ -2)$                  |          $2h\in\mu_\infty$          |
| $\mathbb U$ |    $y_1\pm y_2$     |    $x_1\pm ix_2$    | III  |                 $(2i,\ -2i,\ -2,\ -2)$                  |          $2h\in\mu_\infty$          |
| $\mathbb W$ |    $y_1\pm y_2$     |   $x_1\pm isx_2$    | III  |         $(i(s-i),\ -i(s+i),\ i(s+i),\ -i(s-i))$         |      $h(s\pm i)\in\mu_\infty$       |
| $\mathbb Z$ | $y_1\pm is^{-1}y_2$ |    $x_1\pm x_2$     |  II  |         $(i(s-i),\ -i(s+i),\ -i(s+i),\ i(s-i))$         |      $h(s\pm i)\in\mu_\infty$       |

*Table 2. Monomial data; $s^2=f$, the first sign is index $1$, and the scalars are divided by $h$ (except for $\mathbb N$). For $\mathbb B$, $\mathbb E$, $\mathbb J$ the data are the same for both values of $p$ or $q$.*

*Proof.* Each entry is a direct substitution into the mixed relations. For instance, in $\mathbb O$ the relations are $y_1x_1=h(x_1y_1+fx_2y_2)$, $y_1x_2=h(-x_2y_1+x_1y_2)$, $y_2x_1=h(fx_2y_1-x_1y_2)$, $y_2x_2=h(x_1y_1+x_2y_2)$, and with $z_1=y_1+iy_2$, $e_1=x_1+isx_2$, $$\begin{aligned}
z_1e_1&=y_1x_1+is\,y_1x_2+i\,y_2x_1-s\,y_2x_2\\
&=h\bigl((1-s)x_1y_1+(is^2-is)x_2y_1+(is-i)x_1y_2+(s^2-s)x_2y_2\bigr)\\
&=h(1-s)\,(x_1-isx_2)(y_1-iy_2)=h(1-s)\,e_2z_2 .
\end{aligned}$$ The planes are $\Bbbk_{\pm1}$ except in $\mathbb B$ ($A_x=\Bbbk_p$, $A_y=\Bbbk_p$, coordinate bases), $\mathbb E$ ($A_y=\Bbbk_p$, coordinate), $\mathbb J$ ($A_x=\Bbbk_q$, coordinate) and $\mathbb K$, $\mathbb L$ ($A_x=\Bbbk_q$, coordinate), so the data are admissible. The bases are bases because $s\ne0$, which is where $f\ne0$ is needed. For the last column compute $\mathcal I$: in $\mathbb O$, for example, $\lambda_{11}\lambda_{22}=h^2(1-s)^2$ and $\lambda_{12}\lambda_{21}=h^2(1+s)^2$; in $\mathbb E$, $\lambda_{11}\lambda_{22}=-2ih^2$ and $\lambda_{12}\lambda_{21}=2ih^2$; in $\mathbb P$, $\lambda_{11}\lambda_{21}=h^2(1-f)$ and $\lambda_{11}/\lambda_{12}=(1+s)/(1-s)$. Theorem 4.4 finishes the proof. The plane parameters are $\pm1$ or $\pm i$ except in $\mathbb K$ and $\mathbb L$, where the condition $q\in\mu_\infty$ has to be added to the last column. $\square$

For $\mathbb K$ the criterion, with the PI degree, is in \[GV-K\]. The isomorphism $\mathbb E\cong\mathbb J$ of \[ZZ09, Remark 4.5\], exchanging the $x_i$ and the $y_i$, matches the table.

In [4]:
# Table 2: z_a e_b = lambda_ab e_sigma(b) z_tau(a), for symbolic h and s (f = s^2), and both values of p, q.
def check_table2(name, **par):
    zc, ec, ty, lam = table2(name, p=par.get('p', p))
    fam_par = dict(par)
    if name in 'KLMOPWZ': fam_par['f'] = s**2
    A = family(name, **fam_par)
    hh = 1 if name == 'N' else h
    sg, ta = TYPES[ty]
    z = [lin(A, c[0], c[1], 2, 3) for c in zc]
    e = [lin(A, c[0], c[1], 0, 1) for c in ec]
    ok = all(is_zero(add((1, A.mul(z[a], e[b])), (-hh * lam[2*a+b], A.mul(e[sg[b]], z[ta[a]]))))
             for a in range(2) for b in range(2))
    return check(f'{name} {par}  type {ty}', ok)

for name, par in [('B', {'p': I}), ('B', {'p': -I}), ('E', {'p': I}), ('E', {'p': -I}),
                  ('J', {'q': I}), ('J', {'q': -I}), ('K', {}), ('L', {}), ('M', {}), ('N', {}),
                  ('O', {}), ('P', {}), ('S', {}), ('T', {}), ('U', {}), ('W', {}), ('Z', {})]:
    check_table2(name, **par)

  ok   B {'p': I}  type I
  ok   B {'p': -I}  type I
  ok   E {'p': I}  type I
  ok   E {'p': -I}  type I
  ok   J {'q': I}  type I
  ok   J {'q': -I}  type I
  ok   K {}  type 0
  ok   L {}  type II
  ok   M {}  type I
  ok   N {}  type III
  ok   O {}  type I
  ok   P {}  type II
  ok   S {}  type 0
  ok   T {}  type II
  ok   U {}  type III
  ok   W {}  type III
  ok   Z {}  type II


# 5 Families with a normal generator

Each of the families $\mathbb D$, $\mathbb G$, $\mathbb K$, $\mathbb L$, $\mathbb Q$, $\mathbb V$ has a normal generator of degree one, as one checks on the relations of \[ZZ09, Section 4\]. The families $\mathbb K$ and $\mathbb L$ were treated in Corollary 4.5. For the other four, let $n$ be the normal generator, $x'$ the other $x$-generator and $t=n^{-1}x'$. As in Proposition 3.2, the twist cancels and $$\tag{5.1}
\begin{pmatrix}y_1\\y_2\end{pmatrix}t=M(t)\begin{pmatrix}y_1\\y_2\end{pmatrix},\qquad\text{hence}\qquad \begin{pmatrix}y_1\\y_2\end{pmatrix}F(t)=F\bigl(M(t)\bigr)\begin{pmatrix}y_1\\y_2\end{pmatrix}\quad\text{for all }F\in\Bbbk[t],$$ where $M(t)$ is a $2\times2$ matrix with entries in $\Bbbk[t]$. Central elements come from polynomials $F$ with $F(M(t))=F(t)I$.

## 5.1 The families $\mathbb D$ and $\mathbb G$

The relations of $\mathbb D(p,h)$ are $x_2x_1=-x_1x_2$, $y_2y_1=p\,y_1y_2$ and $$\begin{aligned}
y_1x_1&=-hp\,x_1y_1, & y_1x_2&=h(-p^2x_2y_1+x_1y_2),\\
y_2x_1&=hp\,x_1y_2, & y_2x_2&=h(x_1y_1+x_2y_2).
\end{aligned}$$ In $\mathbb G(p,f,h)$ we replace the generator $x_2$ of \[ZZ09\] by $x_2+\frac1{p-1}x_1$, as in \[GV-G\]; then the relations are $x_2x_1=x_1x_2$, $y_2y_1=p\,y_1y_2$ and $$\begin{aligned}
y_1x_1&=hp\,x_1y_1, & y_1x_2&=h(p^2x_2y_1+x_1y_2),\\
y_2x_1&=hp\,x_1y_2, & y_2x_2&=h(f\,x_1y_1+x_2y_2).
\end{aligned}$$ In both, $n=x_1$, $t=x_1^{-1}x_2$ and $$\tag{5.2}
M_{\mathbb D}(t)=\begin{pmatrix}pt&-p^{-1}\\ p^{-1}&p^{-1}t\end{pmatrix},\qquad M_{\mathbb G}(t)=\begin{pmatrix}pt&p^{-1}\\ fp^{-1}&p^{-1}t\end{pmatrix}.$$

**Lemma 5.1.** Let $y_2y_1=p\,y_1y_2$ and $y_1t=p\,ty_1+c_{12}y_2$, $y_2t=p^{-1}ty_2+c_{21}y_1$ with $c_{12},c_{21}\in\Bbbk$. Then for $k\ge1$ $$\begin{aligned}
y_1^kt&=p^k\,ty_1^k+c_{12}[k]_{p^2}\,y_1^{k-1}y_2, & y_2^kt&=p^{-k}\,ty_2^k+c_{21}\,p^{k-1}[k]_{p^{-2}}\,y_1y_2^{k-1},
\end{aligned}$$ where $[k]_q=1+q+\dots+q^{k-1}$. In particular, if $p^N=1$ and $p^2\ne1$, then $y_1^N$ and $y_2^N$ commute with $t$.

*Proof.* Induction on $k$. Using $y_2y_1^k=p^ky_1^ky_2$, $$\begin{aligned}
y_1^{k+1}t&=y_1\bigl(p^kty_1^k+c_{12}[k]_{p^2}y_1^{k-1}y_2\bigr)=p^k(p\,ty_1+c_{12}y_2)y_1^k+c_{12}[k]_{p^2}y_1^ky_2\\
&=p^{k+1}ty_1^{k+1}+c_{12}\bigl(p^{2k}+[k]_{p^2}\bigr)y_1^ky_2,
\end{aligned}$$ and the other formula is proved in the same way. If $p^N=1$ and $p^2\ne1$ then $[N]_{p^{\pm2}}=0$. $\square$

For $\alpha\in\Bbbk$ let $D_n(X,\alpha)\in\Bbbk[X]$ be the Dickson polynomial, determined by $D_n(w+\alpha w^{-1},\alpha)=w^n+\alpha^nw^{-n}$. It is monic of degree $n$ and has the parity of $n$.

**Lemma 5.2.** Let $M(t)=\left(\begin{smallmatrix}pt&c_{12}\\ c_{21}&p^{-1}t\end{smallmatrix}\right)$ with $p^n=1$, $p^2\ne1$, and $\alpha=-c_{12}c_{21}\,p^2(p^2-1)^{-2}$. Then $D_n(M(t),\alpha)=D_n(t,\alpha)I$.

*Proof.* Write $t=w+\alpha w^{-1}$ in an extension of $\Bbbk(t)$. The elements $\mu_1=pw+\alpha(pw)^{-1}$ and $\mu_2=p^{-1}w+\alpha p\,w^{-1}$ satisfy $$\begin{aligned}
\mu_1+\mu_2&=(p+p^{-1})t, & \mu_1\mu_2&=t^2+\alpha(p-p^{-1})^2=t^2-c_{12}c_{21},
\end{aligned}$$ so they are the eigenvalues of $M(t)$. They are distinct because $p^2\ne1$, and $M(t)$ is diagonalizable. Since $p^n=1$, $D_n(\mu_1,\alpha)=(pw)^n+\alpha^n(pw)^{-n}=D_n(t,\alpha)$, and the same holds for $\mu_2$. $\square$

**Theorem 5.3.** $\mathbb D(p,h)$ is PI if and only if $p,h\in\mu_\infty$ and $p\ne\pm1$. $\mathbb G(p,f,h)$ is PI if and only if $p,h\in\mu_\infty$.

*Proof.* Necessity. The plane $A_y=\Bbbk_p[y_1,y_2]$ forces $p\in\mu_\infty$, Lemma 2.1 applied to $x_1$ and $y_1$ gives $hp\in\mu_\infty$, and $\mathbb D(\pm1,h)$ is excluded by Proposition 3.2.

Sufficiency. By Theorem 2.5 we may take $h=1$. Let $N=\operatorname{ord}p$, so $N\ge3$.

In $\mathbb D(p,1)$ let $\ell=\operatorname{lcm}(2,N)$; by (5.2), Lemmas 5.1 and 5.2 apply with $c_{12}c_{21}=-p^{-2}$, so $\alpha=(p^2-1)^{-2}$. The elements $y_1^\ell$ and $y_2^N$ commute with $t$ by Lemma 5.1, with $x_1$ because $(-p)^\ell=p^N=1$, hence with $x_2=x_1t$, and they are central in $A_y$. The element $x_1^\ell$ commutes with $y_1,y_2$ and, $\ell$ being even, with $x_2$. Finally let $$\Phi=x_1^\ell\,D_N(t,\alpha)^{\ell/N}.$$ The polynomial $D_N(t,\alpha)^{\ell/N}$ is even in $t$, so it commutes with $x_1$, which anticommutes with $t$; it commutes with $y_1,y_2$ by (5.1) and Lemma 5.2. So $\Phi$ is central. It lies in $\Bbbk[x_1^2,x_2^2]$ and has the form $\pm x_2^\ell+{}$terms of lower degree in $x_2$. Hence $A_x$ is a finite module over $\Bbbk[x_1^\ell,\Phi]$, $A_y$ is finite over $\Bbbk[y_1^\ell,y_2^N]$, and $B=A_xA_y$ is a finite module over a central subalgebra.

In $\mathbb G(p,f,1)$, (5.2) gives $c_{12}c_{21}=fp^{-2}$ and $\alpha=-f(p^2-1)^{-2}$. In the same way $x_1^N$, $y_1^N$, $y_2^N$ and $x_1^ND_N(t,\alpha)=D_N(x_2,\alpha x_1^2)$ are central, and $B$ is a finite module over the subalgebra they generate. $\square$

The PI degrees of $\mathbb D$ and $\mathbb G$ are in \[GV-D, GV-G\]; they come from a quantum torus with the same quotient division ring (Lemma 11.8). For $\mathbb K$, a crossed product of index two over a quantum torus, see \[GV-K, Theorem 2.5\].

In [5]:
# Lemma 5.1, symbolically in p, c12, c21 (algebra on t, y1, y2).
c12, c21 = sp.symbols('c12 c21')
T = Alg(['t', 'y1', 'y2'], {(1, 0): [(p, (0, 1)), (c12, (2,))],
                            (2, 0): [(1 / p, (0, 2)), (c21, (1,))],
                            (2, 1): [(p, (1, 2))]})
t_, y1_, y2_ = T.gen(0), T.gen(1), T.gen(2)
qint = lambda k, x: sum(x**i for i in range(k))
for k in range(1, 7):
    Y1 = T.mul(*[y1_] * k); Y2 = T.mul(*[y2_] * k)
    ok1 = is_zero(add((1, T.mul(Y1, t_)), (-p**k, T.mul(t_, Y1)),
                      (-c12 * qint(k, p**2), T.mul(*([y1_] * (k - 1) + [y2_])))))
    ok2 = is_zero(add((1, T.mul(Y2, t_)), (-p**(-k), T.mul(t_, Y2)),
                      (-c21 * p**(k - 1) * qint(k, p**(-2)), T.mul(*([y1_] + [y2_] * (k - 1))))))
    check(f'Lemma 5.1, k = {k}', ok1 and ok2)

# Lemma 5.2 (Dickson): D_n(M(t), alpha) = D_n(t, alpha) I, numerically for n = 3..9 at random t.
import random
def dickson(n, X, a):
    D0, D1 = 2 * sp.eye(X.shape[0]) if hasattr(X, 'shape') else 2, X
    for _ in range(n - 1):
        D0, D1 = D1, X * D1 - a * D0
    return D1 if n >= 1 else D0
for n in range(3, 10):
    pp = complex(sp.N(root(n))); cc12, cc21 = 0.7 - 0.2j, -1.3 + 0.4j
    al = -cc12 * cc21 * pp**2 / (pp**2 - 1)**2
    tt = random.uniform(-2, 2) + 1j * random.uniform(-2, 2)
    Mt = sp.Matrix([[pp * tt, cc12], [cc21, tt / pp]])
    lhs = sp.N(dickson(n, Mt, al)); rhs = complex(sp.N(dickson(n, tt, al)))
    check(f'Lemma 5.2, n = {n}', max(abs(complex(lhs[i, j]) - (rhs if i == j else 0)) for i in range(2) for j in range(2)) < 1e-8)

  ok   Lemma 5.1, k = 1
  ok   Lemma 5.1, k = 2
  ok   Lemma 5.1, k = 3
  ok   Lemma 5.1, k = 4
  ok   Lemma 5.1, k = 5
  ok   Lemma 5.1, k = 6
  ok   Lemma 5.2, n = 3
  ok   Lemma 5.2, n = 4


  ok   Lemma 5.2, n = 5


  ok   Lemma 5.2, n = 6


  ok   Lemma 5.2, n = 7


  ok   Lemma 5.2, n = 8


  ok   Lemma 5.2, n = 9


In [6]:
# Theorem 5.3: the central elements used for h = 1, with p a primitive N-th root of unity (numeric).
def num_alg(A):
    A.rules = {k: [(complex(sp.N(c)), w) for c, w in v] for k, v in A.rules.items()}; return A
def central(A, el):
    return all(is_zero(add((1, A.mul(G_, el)), (-1, A.mul(el, G_)))) for G_ in gens(A))
def power(A, el, k):
    return A.mul(*[el] * k) if k else {(): 1}
for N in (3, 4, 5, 6):
    pp = complex(sp.N(root(N)))
    # D(p,1)
    A = num_alg(family('D', h=1, p=pp)); x1, x2, y1, y2 = gens(A)
    l = math.lcm(2, N); al = (pp**2 - 1)**(-2)
    poly = sp.Poly(sp.expand(dickson(N, sp.Symbol('X'), al)**(l // N)), sp.Symbol('X'))
    Phi = {}
    for (deg,), c in poly.terms():           # x1^l t^(2k) = (-1)^k x1^(l-2k) x2^(2k)
        kk = deg // 2
        Phi = add((1, Phi), (complex(c) * (-1)**kk, A.mul(power(A, x1, l - 2 * kk), power(A, x2, 2 * kk))))
    ok = all(central(A, el) for el in (power(A, y1, l), power(A, y2, N), power(A, x1, l), Phi))
    check(f'D: y1^l, y2^N, x1^l and Phi central, N = {N}', ok)
    # G(p,f,1)
    ff = 0.6 + 0.3j
    A = num_alg(family('G', h=1, p=pp, f=ff)); x1, x2, y1, y2 = gens(A)
    al = -ff * (pp**2 - 1)**(-2)
    Xs, a_ = sp.Symbol('X'), sp.Symbol('a')
    poly = sp.Poly(sp.expand(dickson(N, Xs, a_)), Xs, a_)
    Phi = {}
    for (dx, da), c in poly.terms():          # D_N(x2, alpha x1^2)
        Phi = add((1, Phi), (complex(c) * al**da, A.mul(power(A, x1, 2 * da), power(A, x2, dx))))
    ok = all(central(A, el) for el in (power(A, x1, N), power(A, y1, N), power(A, y2, N), Phi))
    check(f'G: x1^N, y1^N, y2^N and D_N(x2, alpha x1^2) central, N = {N}', ok)

  ok   D: y1^l, y2^N, x1^l and Phi central, N = 3


  ok   G: x1^N, y1^N, y2^N and D_N(x2, alpha x1^2) central, N = 3
  ok   D: y1^l, y2^N, x1^l and Phi central, N = 4
  ok   G: x1^N, y1^N, y2^N and D_N(x2, alpha x1^2) central, N = 4


  ok   D: y1^l, y2^N, x1^l and Phi central, N = 5
  ok   G: x1^N, y1^N, y2^N and D_N(x2, alpha x1^2) central, N = 5


  ok   D: y1^l, y2^N, x1^l and Phi central, N = 6
  ok   G: x1^N, y1^N, y2^N and D_N(x2, alpha x1^2) central, N = 6


## 5.2 The families $\mathbb Q$ and $\mathbb V$

For $h=1$ the relations are $$\begin{aligned}
\mathbb Q:\quad & y_1x_1=x_1y_2, && y_1x_2=x_1y_1+x_2y_1+x_1y_2,\\
& y_2x_1=-x_1y_1, && y_2x_2=x_1y_1-x_1y_2+x_2y_2,\\
\mathbb V:\quad & y_1x_1=x_2y_1+x_1y_2, && y_1x_2=x_2y_1,\\
& y_2x_1=-x_1y_1+x_2y_1, && y_2x_2=x_2y_2,
\end{aligned}$$ with $y_2y_1=-y_1y_2$ in both, $x_2x_1=-x_1x_2$ in $\mathbb Q$ and $x_2x_1=x_1x_2$ in $\mathbb V$.

**Proposition 5.4.** $\mathbb Q(h)$ and $\mathbb V(h)$ are PI if and only if $h\in\mu_\infty$. For $h=1$ the elements $$\begin{aligned}
\mathbb Q(1):&\quad y_1^2+y_2^2,\quad y_1^2y_2^2,\quad x_1^2+x_2^2,\quad x_1^2x_2^2+\tfrac12x_2^4,\\
\mathbb V(1):&\quad y_1^2+y_2^2,\quad y_1^2y_2^2,\quad x_2,\quad x_1^2(x_1-x_2)^2
\end{aligned}$$ are central, and the algebra is a finite module over the subalgebra they generate.

*Proof.* Necessity. In $\mathbb Q(h)$, $y_1x_1=hx_1y_2$ and $y_2x_1=-hx_1y_1$, so $y_1x_1^2=-h^2x_1^2y_1$; in $\mathbb V(h)$, $y_1x_2=hx_2y_1$. Lemma 2.1 gives $h^2\in\mu_\infty$, respectively $h\in\mu_\infty$.

Sufficiency. By Theorem 2.5 it suffices to treat $h=1$. Two applications of the relations give, in $\mathbb Q(1)$, $$\begin{aligned}
y_1^2x_1&=x_1y_2^2, & y_1^2x_2&=x_1y_1^2-2x_1y_1y_2+x_1y_2^2+x_2y_1^2,\\
y_2^2x_1&=x_1y_1^2, & y_2^2x_2&=-x_1y_1^2+2x_1y_1y_2-x_1y_2^2+x_2y_2^2,
\end{aligned}$$ and in $\mathbb V(1)$, $$\begin{aligned}
y_1^2x_1&=x_1y_2^2+x_2y_1^2+x_2y_1y_2, & y_1^2x_2&=x_2y_1^2,\\
y_2^2x_1&=x_1y_1^2-x_2y_1^2-x_2y_1y_2, & y_2^2x_2&=x_2y_2^2 .
\end{aligned}$$ Adding, $y_1^2+y_2^2$ commutes with $x_1,x_2$; it commutes with $y_1,y_2$ because squares are central in $\Bbbk_{-1}[y_1,y_2]$. For $y_1^2y_2^2$ one uses the same formulas twice; for instance in $\mathbb V(1)$, since $y_1^2$ is central in $A_y$, $$\begin{aligned}
y_1^2y_2^2x_1&=y_1^2\bigl(x_1y_1^2-x_2y_1^2-x_2y_1y_2\bigr)=(x_1y_2^2+x_2y_1^2+x_2y_1y_2)y_1^2-x_2y_1^4-x_2y_1^3y_2=x_1y_1^2y_2^2 .
\end{aligned}$$

For the $x$-elements let $n=x_1$, $t=x_1^{-1}x_2$ in $\mathbb Q$ and $n=x_2$, $t=x_2^{-1}x_1$ in $\mathbb V$. In (5.1), $$M_{\mathbb Q}(t)=\begin{pmatrix}-1&1-t\\1+t&1\end{pmatrix},\qquad M_{\mathbb V}(t)=\begin{pmatrix}1&t\\1-t&0\end{pmatrix}.$$ By Cayley--Hamilton, $M_{\mathbb Q}^2=(2-t^2)I$ and $M_{\mathbb V}^2-M_{\mathbb V}=(t-t^2)I$. So $F_{\mathbb Q}(t)=t^2(2-t^2)$ and $F_{\mathbb V}(t)=(t^2-t)^2$ satisfy $F(M(t))=F(t)I$, that is, they commute with $y_1,y_2$. In $\mathbb Q$, $t$ anticommutes with $x_1$, $x_1^4$ commutes with $y_1,y_2$, and $$x_1^4F_{\mathbb Q}(t)=-2x_1^2x_2^2-x_2^4,$$ which is central since it lies in $\Bbbk[x_1^2,x_2^2]$. In $\mathbb V$, $x_2$ is central and $x_2^4F_{\mathbb V}(t)=x_1^2(x_1-x_2)^2$.

Finiteness: $\Bbbk[x_1^2,x_2^2]$ is integral over $\Bbbk[x_1^2+x_2^2,\ x_1^2x_2^2+\frac12x_2^4]$, since $x_2^2$ is a root of $\frac12T^2-(x_1^2+x_2^2)T+(x_1^2x_2^2+\frac12x_2^4)$, and $\Bbbk[x_1,x_2]$ is integral over $\Bbbk[x_2,x_1^2(x_1-x_2)^2]$. With the $y$-elements and the basis this gives a finite module over a central subalgebra. $\square$

In [7]:
# Proposition 5.4: central elements of Q(1) and V(1), exactly.
def central_exact(A, el):
    return all(is_zero(add((1, A.mul(G_, el)), (-1, A.mul(el, G_)))) for G_ in gens(A))
A = family('Q', h=1); x1, x2, y1, y2 = gens(A)
for lab, el in [('y1^2+y2^2', add((1, A.mul(y1, y1)), (1, A.mul(y2, y2)))), ('y1^2 y2^2', A.mul(y1, y1, y2, y2)),
                ('x1^2+x2^2', add((1, A.mul(x1, x1)), (1, A.mul(x2, x2)))),
                ('x1^2x2^2 + x2^4/2', add((1, A.mul(x1, x1, x2, x2)), (sp.Rational(1, 2), A.mul(x2, x2, x2, x2))))]:
    check('Q(1): ' + lab + ' central', central_exact(A, el))
A = family('V', h=1); x1, x2, y1, y2 = gens(A); d = add((1, x1), (-1, x2))
for lab, el in [('y1^2+y2^2', add((1, A.mul(y1, y1)), (1, A.mul(y2, y2)))), ('y1^2 y2^2', A.mul(y1, y1, y2, y2)),
                ('x2', x2), ('x1^2 (x1-x2)^2', A.mul(x1, x1, d, d))]:
    check('V(1): ' + lab + ' central', central_exact(A, el))

  ok   Q(1): y1^2+y2^2 central
  ok   Q(1): y1^2 y2^2 central
  ok   Q(1): x1^2+x2^2 central
  ok   Q(1): x1^2x2^2 + x2^4/2 central
  ok   V(1): y1^2+y2^2 central
  ok   V(1): y1^2 y2^2 central
  ok   V(1): x2 central
  ok   V(1): x1^2 (x1-x2)^2 central


## 5.3 The family $\mathbb L$ and its PI degree

The relations of $\mathbb L(q,f,h)$ are $$\begin{aligned}
x_2x_1&=q\,x_1x_2, & y_1x_1&=hf\,x_1y_2, & y_1x_2&=h\,x_2y_2,\\
y_2y_1&=-y_1y_2, & y_2x_1&=hf\,x_1y_1, & y_2x_2&=h\,x_2y_1 .
\end{aligned}$$ Both $x_1$ and $x_2$ are normal and exchange $y_1$ and $y_2$ up to scalars.

**Theorem 5.5.** $\mathbb L(q,f,h)$ is PI if and only if $q,f,h\in\mu_\infty$. In that case let $N=|\langle-1,q,f,h\rangle|$ and $N_0=|\langle-1,q,f\rangle|$. Then $$\operatorname{PIdeg}\mathbb L(q,f,h)=\begin{cases}2N, & N/N_0\ \text{odd},\\ N, & N/N_0\ \text{even}.\end{cases}$$

*Proof.* The criterion is Corollary 4.5. Put $u=x_1^2$ and $w=x_1x_2$. From the relations, $$\begin{aligned}
y_au&=(hf)^2\,uy_a, & y_aw&=h^2f\,wy_a, & wu&=q^2\,uw, & y_2y_1&=-y_1y_2 ,\qquad(5.3)
\end{aligned}$$ and the monomials $y_1^cy_2^du^jw^l$ are independent because $u^jw^l$ is a nonzero multiple of $x_1^{2j+l}x_2^l$. So $y_1,y_2,u,w$ generate a quantum affine space; let $T$ be its quantum torus inside $Q(\mathbb L)$. Conjugation by $x_1$ sends $y_1\mapsto(hf)^{-1}y_2$, $y_2\mapsto(hf)^{-1}y_1$, $u\mapsto u$, $w\mapsto q^{-1}w$, so $x_1$ normalizes $Q(T)$, and $x_1^2=u\in T$, $x_2=x_1^{-1}w$. Hence $Q(\mathbb L)=Q(T)+Q(T)x_1$.

The element $y_1^N$ is central in $Q(T)$ by (5.3), and $x_1y_1^Nx_1^{-1}=y_2^N\ne y_1^N$. So $x_1\notin Q(T)$, the sum is direct, and $\operatorname{Ad}_{x_1}$ acts on $Z(Q(T))$ with order exactly two, its square being $\operatorname{Ad}_u$. If $t_0+t_1x_1$ is central, commuting it with $y_1^N$ gives $t_1(y_2^N-y_1^N)=0$, so $t_1=0$; hence $Z(Q(\mathbb L))=Z(Q(T))^{\langle\operatorname{Ad}_{x_1}\rangle}$ and $$[Q(\mathbb L):Z(Q(\mathbb L))]=2\cdot(\operatorname{PIdeg}T)^2\cdot 2 .$$ It remains to compute $\operatorname{PIdeg}T$. Let $\zeta$ be a primitive $N$th root of unity with $q=\zeta^\alpha$, $f=\zeta^\varphi$, $h=\zeta^\eta$, so that $\zeta^{N/2}=-1$ and $\gcd(\alpha,\varphi,\eta,N/2)=1$. In the basis $w,y_1,y_1^{-1}y_2,u$ of the lattice of monomials, (5.3) gives the commutation matrix with upper entries $$2\eta+\varphi,\quad 0,\quad 2\alpha,\quad N/2,\quad 2\eta+2\varphi,\quad 0,$$ whose Pfaffian is $\alpha N$. Let $g$ be the gcd of the entries; it divides $N/2$. By the normal form of alternating integer matrices \[New, Chapter IV\], as in \[GV-K, Lemma 2.4\], $T$ has PI degree $$\frac Ng\cdot\frac{N}{\gcd(N,\alpha N/g)}=\frac Ng\cdot\frac{g}{\gcd(g,\alpha)}=\frac{N}{\gcd(\alpha,N/2,\varphi,2\eta)} .$$ The last gcd is $1$ or $2$, and it is $2$ exactly when $\alpha$, $\varphi$ and $N/2$ are even, that is, when $-1,q,f$ lie in the subgroup of index two of $\langle-1,q,f,h\rangle$. This is the statement. $\square$

For instance $\operatorname{PIdeg}\mathbb L(1,1,1)=4$, $\operatorname{PIdeg}\mathbb L(1,1,i)=4$ and $\operatorname{PIdeg}\mathbb L(i,1,1)=8$. In $\mathbb K$, with the same parameters, the PI degree is always $2N$ \[GV-K, Theorem 2.5\]; in $\mathbb L$ the exchange of $y_1,y_2$ by both $x$-generators can halve it.

In [8]:
# Theorem 5.5: the gcd formula for PIdeg T against a direct lattice computation (2 PIdeg T = PIdeg L).
bad = 0; cases = 0
for kq, kf, kh in itertools.product((1, 2, 3, 4, 6), (1, 2, 3, 4), (1, 2, 3, 4, 6, 8)):
    N = math.lcm(2, kq, kf, kh); N0 = math.lcm(2, kq, kf)
    a, ph, eta = N // kq, N // kf, N // kh          # exponents: q = z^a, f = z^ph, h = z^eta
    # generators y1, y2, u, w ;  y_a u = (hf)^2 u y_a ; y_a w = h^2 f w y_a ; w u = q^2 u w ; y2 y1 = - y1 y2
    E = [[0] * 4 for _ in range(4)]
    def setp(i, j, e): E[i][j] = e % N; E[j][i] = (-e) % N
    for ya in (0, 1):
        setp(ya, 2, 2 * eta + 2 * ph); setp(ya, 3, 2 * eta + ph)
    setp(3, 2, 2 * a); setp(1, 0, N // 2)
    formula = 2 * N if (N // N0) % 2 else N
    cases += 1; bad += (2 * torus_pideg(E, N) != formula)
check(f'Theorem 5.5 on {cases} parameter values', bad == 0)

  ok   Theorem 5.5 on 120 parameter values


True

# 6 Normal subalgebras: the families $\mathbb F$, $\mathbb I$ and $\mathbb C$

Neither method applies here. Zhang and Zhang proved these three families strongly noetherian with normal elements of degree two and three \[ZZ09, Section 5\], and the same elements decide the PI property.

**Lemma 6.1.** Let $R=\Bbbk_{-1}[a,b]$, $\omega_1=ab$ and $\omega_2=a^2+b^2$. Then $\omega_1$ and $\omega_2$ commute and are algebraically independent, and $R$ is free as a left and as a right $\Bbbk[\omega_1,\omega_2]$-module with basis $1,a,b,a^2$.

*Proof.* $a^2,b^2$ are central, so $\omega_1$ commutes with $\omega_2$, and $\omega_1^2=-a^2b^2$. Since $\omega_2$ and $\omega_1^2$ are algebraically independent in $\Bbbk[a^2,b^2]$, so are $\omega_1$ and $\omega_2$. From $a^3=a\omega_2+b\omega_1$, $b^3=b\omega_2-a\omega_1$ and $a^4=\omega_2a^2+\omega_1^2$, together with $ab=\omega_1$ and $ba=-\omega_1$, the four elements generate $R$ on either side, and the surjection $\Bbbk[\omega_1,\omega_2]\oplus\Bbbk[\omega_1,\omega_2](-1)^2\oplus\Bbbk[\omega_1,\omega_2](-2)\to R$ is an isomorphism because both sides have Hilbert series $(1-t)^{-2}$. $\square$

## 6.1 The family $\mathbb F$

The relations of $\mathbb F(p,h)$, $p^2=-1$, are $x_2x_1=-x_1x_2$, $y_2y_1=py_1y_2$ and $$\begin{aligned}
y_1x_1&=h(-x_1y_1-px_2y_1+x_1y_2-x_2y_2), & y_1x_2&=h(-px_1y_1+x_2y_1+x_1y_2+x_2y_2),\\
y_2x_1&=h(-px_1y_1+px_2y_1+px_1y_2+x_2y_2), & y_2x_2&=h(-px_1y_1-px_2y_1+x_1y_2-px_2y_2).
\end{aligned}$$

**Lemma 6.2.** Let $\omega_1=x_1x_2$, $\omega_2=x_1^2+x_2^2$ and $\lambda=-2ph^2$. Then, for $j=1,2$, $$\tag{6.1}
y_j\omega_1=\lambda\,\omega_1y_j,\qquad y_j\omega_2=\lambda\,\omega_2y_j .$$

*Proof.* Apply the mixed relations twice and use $x_2x_1=-x_1x_2$, $p^2=-1$. For $y_1\omega_1$ the four terms of $y_1x_1$ give $$\begin{aligned}
h^{-2}y_1x_1x_2&=\bigl(p\,x_1^2y_1-x_1x_2y_1-x_1^2y_2-x_1x_2y_2\bigr)\\
&\quad+\bigl(x_1x_2y_1-p\,x_2^2y_1+p\,x_1x_2y_2-p\,x_2^2y_2\bigr)\\
&\quad+\bigl(-p\,x_1^2y_1-p\,x_1x_2y_1+x_1^2y_2-p\,x_1x_2y_2\bigr)\\
&\quad+\bigl(-p\,x_1x_2y_1+p\,x_2^2y_1+x_1x_2y_2+p\,x_2^2y_2\bigr)\\
&=-2p\,x_1x_2y_1 .
\end{aligned}$$ In the same way $$\begin{aligned}
h^{-2}y_1x_1^2&=(1-p)x_1^2y_1+(p-1)x_1^2y_2+2(p+1)x_1x_2y_2-(p+1)x_2^2y_1+(p-1)x_2^2y_2,\\
h^{-2}y_1x_2^2&=-(p+1)x_1^2y_1-(p-1)x_1^2y_2-2(p+1)x_1x_2y_2-(p-1)x_2^2y_1-(p-1)x_2^2y_2,\\
h^{-2}y_2x_1^2&=(p+1)x_1^2y_1-(p+1)x_1^2y_2+2(p-1)x_1x_2y_1+(p+1)x_2^2y_1-(p-1)x_2^2y_2,\\
h^{-2}y_2x_2^2&=-(p+1)x_1^2y_1-(p-1)x_1^2y_2-2(p-1)x_1x_2y_1-(p+1)x_2^2y_1-(p+1)x_2^2y_2,\\
h^{-2}y_2x_1x_2&=-2p\,x_1x_2y_2,
\end{aligned}$$ and adding the pairs gives $y_j\omega_2=-2ph^2\,\omega_2y_j$. $\square$

**Theorem 6.3.** $\mathbb F(p,h)$ is PI if and only if $2h^2\in\mu_\infty$.

*Proof.* Let $S=\Bbbk\langle \omega_1,\omega_2,y_1,y_2\rangle$. By (6.1), $\omega_1\omega_2=\omega_2\omega_1$ and $y_2y_1=py_1y_2$, $S$ is a quotient of the quantum affine space with these relations, and it is that quantum affine space because the monomials $\omega_1^a\omega_2^by_1^cy_2^d$ are independent (Lemma 6.1 and the basis of $B$). If $\mathbb F$ is PI, Lemma 2.1 applied to $\omega_1$ and $y_1$ gives $\lambda\in\mu_\infty$, that is, $2h^2\in\mu_\infty$ because $p\in\mu_\infty$. Conversely, Lemma 6.1 and the basis give $$\tag{6.2}
\mathbb F(p,h)=S\oplus x_1S\oplus x_2S\oplus x_1^2S ,$$ a free right $S$-module of rank four. If $\lambda\in\mu_\infty$ then $S$ is PI, and Lemma 2.2 applies. $\square$

In [9]:
# Lemma 6.2 (exact, symbolic h), for p = i and p = -i.
for pp in (I, -I):
    A = family('F', p=pp); x1, x2, y1, y2 = gens(A)
    lam = -2 * pp * h**2
    w1 = A.mul(x1, x2); w2 = add((1, A.mul(x1, x1)), (1, A.mul(x2, x2)))
    check(f'F, p = {pp}: y_j omega_k = lambda omega_k y_j',
          all(skew(A, y, w, lam) for y in (y1, y2) for w in (w1, w2)))

  ok   F, p = I: y_j omega_k = lambda omega_k y_j
  ok   F, p = -I: y_j omega_k = lambda omega_k y_j


## 6.2 The family $\mathbb I$

**Theorem 6.4.** $\mathbb I(q,h)$ is PI if and only if $2h^2\in\mu_\infty$.

*Proof.* Let $\omega_1=y_1y_2$, $\omega_2=y_1^2+y_2^2$ and $\lambda'=-2qh^2$. We claim that $$\tag{6.3}
\omega_kx_i=\lambda'\,x_i\omega_k\qquad(i,k\in\{1,2\}).$$ Take $w_1,w_2$, $Z_1,Z_2$ and $c=h(1-q)$ as in Lemma 8.8 below, whose proof does not use this section; note that $c^2=h^2(1-2q+q^2)=\lambda'$. By that lemma, conjugation by $Z_j^{-1}$ sends $y_1,y_2$ to $c\,y_1,-c\,y_2$ for $j=1$ and to $c\,y_2,c\,y_1$ for $j=2$, so it multiplies $\omega_1$ by $-c^2$ and $\omega_2$ by $c^2$. Since $Z_1-Z_2=(1-q)x_2w_2$, we have $x_2=(1-q)^{-1}(Z_1-Z_2)w_2^{-1}$ and $x_1=(Z_1-x_2w_2)w_1^{-1}$, so each $x_i$ is a sum $Z_1d_1+Z_2d_2$ with $d_j\in Q(A_y)$ of degree $-1$. In $Q(A_y)$, $\omega_2$ is central and $\omega_1$ anticommutes with the elements of odd degree. Hence $\omega_1Z_jd_j=-c^2Z_j\omega_1d_j=c^2Z_jd_j\omega_1$ and $\omega_2Z_jd_j=c^2Z_jd_j\omega_2$, which is (6.3). Lemma 2.1 applied to $x_1$ and $\omega_1$ shows that $\lambda'\in\mu_\infty$ if $\mathbb I$ is PI, that is, $2h^2\in\mu_\infty$. Now argue as for $\mathbb F$, with $S'=\Bbbk\langle x_1,x_2,\omega_1,\omega_2\rangle$ and $\mathbb I=S'\oplus S'y_1\oplus S'y_2\oplus S'y_1^2$ as a left $S'$-module. $\square$

In [10]:
# Equation (6.3): omega_k x_i = -2 q h^2 x_i omega_k (exact).
for qq in (I, -I):
    A = family('I', q=qq); x1, x2, y1, y2 = gens(A)
    w1 = A.mul(y1, y2); w2 = add((1, A.mul(y1, y1)), (1, A.mul(y2, y2)))
    check(f'I, q = {qq}', all(skew(A, w, x, -2 * qq * h**2) for w in (w1, w2) for x in (x1, x2)))

  ok   I, q = I
  ok   I, q = -I


## 6.3 The family $\mathbb C$

**Theorem 6.5.** $\mathbb C(p,h)$ is PI if and only if $3h^2\in\mu_\infty$.

*Proof.* Let $a=y_1^2y_2$ and $b=y_1^3-y_2^3$. In \[ZZ09, Section 5\] it is shown that, for $h=1$, $a$ and $b$ skew-commute with $\Bbbk x_1+\Bbbk x_2$ with the scalars $(p-1)^2(1-p^2)$ and $(1-p^2)^3$. With $p^2+p+1=0$ these are $3(1-p)$ and $3(2p+1)$, and by (2.1), in $\mathbb C(p,h)$, $$\begin{aligned}
ax_i&=\alpha\,x_ia, & bx_i&=\beta\,x_ib, & \alpha&=3(1-p)h^3, & \beta&=3(2p+1)h^3 .\qquad(6.4)
\end{aligned}$$ Then $\alpha=p^2\beta$, $\alpha^2=-p(3h^2)^3$ and $\beta^2=-(3h^2)^3$, so $\alpha,\beta\in\mu_\infty$ if and only if $3h^2\in\mu_\infty$.

Since $p^3=1$, $Y_1=y_1^3$ and $Y_2=y_2^3$ are central in $A_y$, so $a$ and $b$ commute. We have $b=Y_1-Y_2$ and $a^3=Y_1^2Y_2$, so $Y_1$ is a root of $T^3-bT^2-a^3$, and $A_y$, which is free of rank nine over $\Bbbk[Y_1,Y_2]$, is a finite left module over $\Bbbk[a,b]$; in particular $a,b$ are algebraically independent. Hence $S=\Bbbk\langle x_1,x_2,a,b\rangle$ is a quantum affine space with parameters $p,\alpha,\beta$, and $\mathbb C=\sum_wS\,w$ for a finite set of $w\in A_y$. If $\mathbb C$ is PI, Lemma 2.1 for $x_1$ and $a$ gives $\alpha\in\mu_\infty$. If $3h^2\in\mu_\infty$, $S$ is PI and Lemma 2.2 applies. $\square$

In [11]:
# Theorem 6.5: the scalars alpha, beta (exact, p = exp(2 pi i/3), symbolic h).
pp = sp.Rational(-1, 2) + sp.sqrt(3) * I / 2
A = family('C', p=pp); x1, x2, y1, y2 = gens(A)
a_ = A.mul(y1, y1, y2); b_ = add((1, A.mul(y1, y1, y1)), (-1, A.mul(y2, y2, y2)))
al, be = 3 * (1 - pp) * h**3, 3 * (2 * pp + 1) * h**3
check('a x_i = alpha x_i a', all(skew(A, a_, x, al) for x in (x1, x2)))
check('b x_i = beta x_i b', all(skew(A, b_, x, be) for x in (x1, x2)))
check('alpha = p^2 beta,  beta^2 = -(3h^2)^3', zero(al - pp**2 * be) and zero(be**2 + 27 * h**6))

  ok   a x_i = alpha x_i a
  ok   b x_i = beta x_i b
  ok   alpha = p^2 beta,  beta^2 = -(3h^2)^3


True

# 7 The PI degree of the monomial families

Here $B$ is one of $\mathbb B$, $\mathbb E$, $\mathbb J$, $\mathbb M$, $\mathbb N$, $\mathbb O$, $\mathbb P$, $\mathbb S$, $\mathbb T$, $\mathbb U$, $\mathbb W$, $\mathbb Z$ (with $f\ne0$ where it applies), with the datum of Table 2. The same method gives back the PI degrees of $\mathbb K$ \[GV-K\] and $\mathbb L$ (Theorem 5.5).

## 7.1 The three kinds of plane

Call a basis $v_1,v_2$ of a plane *balanced* if the plane is $\Bbbk_{-1}$ and $v_1^2=v_2^2$; every basis of the form $v_1=w_1+cw_2$, $v_2=w_1-cw_2$ in a coordinate basis $w_1,w_2$ of $\Bbbk_{-1}$ is balanced. Inspecting Table 2, each side of each datum is of one of three kinds:

<div class="center">

|    kind     |                                                                            families (side)                                                                            | rule for $w=v_1^{-1}v_2$ |
|:-----------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------:|:------------------------:|
|  balanced   | $x$: $\mathbb E,\mathbb M,\mathbb O,\mathbb P,\mathbb S,\mathbb T,\mathbb U,\mathbb Z$;  $y$: $\mathbb M,\mathbb N,\mathbb O,\mathbb S,\mathbb T,\mathbb U,\mathbb W$ |  $v_1wv_1^{-1}=w^{-1}$   |
| commutative |                                                                  $x$: $\mathbb W$;  $y$: $\mathbb Z$                                                                  |     $v_1wv_1^{-1}=w$     |
| coordinate  |                             $x$: $\mathbb B$ ($q=p$), $\mathbb N$ ($q=-1$);  $y$: $\mathbb B$, $\mathbb E$ ($q=p$), $\mathbb P$ ($q=-1$)                              |  $v_1wv_1^{-1}=q^{-1}w$  |

</div>

In the balanced case $v_1^2$ is central in the plane and $v_2v_1^{-1}=w^{-1}$, because $v_1^{-1}=v_1/v_1^2$ and $v_2^{-1}=v_2/v_1^2$.

In $Q(B)$ put $$\begin{aligned}
e&=e_1, & r&=e_1^{-1}e_2, & X&=e^2, & z&=z_1, & \rho&=z_1^{-1}z_2, & Y&=z^2 .
\end{aligned}$$ For elements $a,b$ we write $a\triangleright b=aba^{-1}$.

**Proposition 7.1.** Let $T'$ be the subalgebra of $Q(B)$ generated by $r^{\pm1},\rho^{\pm1},X^{\pm1},Y^{\pm1}$. Then:

1.  $T'$ is a quantum torus, and $e$ and $z$ normalize it.

2.  $Q(B)=Q(T')\oplus Q(T')e\oplus Q(T')z\oplus Q(T')ez$.

3.  Let $G\subseteq\operatorname{Aut}(K)$, $K=Z(Q(T'))$, be the group generated by $e\triangleright$ and $z\triangleright$, and $n'=\operatorname{PIdeg}T'$. If $G$ has order four, then $\operatorname{PIdeg}B=4n'$. If $G$ has order two, then $\operatorname{PIdeg}B=2n'$.

*Proof.* (i) The relations of the next subsection (proved in Appendix A) show that $e$ and $z$ send each of $r,\rho,X,Y$ to a scalar multiple of a monomial in them, and that these four elements commute up to scalars. On the $x$-side, $X^cr^a$ is a nonzero multiple of $X^{c-a}(e_1e_2)^a$ in the balanced and commutative cases and of $x_1^{2c-a}x_2^a$ in the coordinate case, and these elements of $Q(A_x)$ are linearly independent; the same holds on the $y$-side. Multiplying a relation $\sum c\,r^a\rho^bX^cY^d=0$ by powers of $X$ and $Y$ and using the basis of $B$ shows that the monomials are independent.

\(ii\) The sum $R$ of the four spaces is closed under multiplication, because $e^2=X$, $z^2=Y$, and $ze$ is a multiple of $e\,z$ times a monomial of $T'$ by (4.1). It contains $e_1$, $e_2=er$, $z_1$ and $z_2=z\rho$, so $B\subseteq R$; it is a domain of dimension four over $Q(T')$, hence a division ring, and $R=Q(B)$. The sum is direct: the automorphisms of $B$ given by $x_i\mapsto-x_i$ and by $y_j\mapsto-y_j$ extend to $Q(B)$ and fix $r,\rho,X,Y$, hence $Q(T')$, while they multiply $e$, $z$, $ez$ by the pairs of signs $(-1,1)$, $(1,-1)$, $(-1,-1)$. So the four summands lie in different joint eigenspaces.

\(iii\) This is the usual computation for crossed products. If $\sum_ga_gw_g$, with $w_g\in\{1,e,z,ez\}$, commutes with $Q(T')$, then $a_g\,g(c)=c\,a_g$ for $c\in K$, so $a_g=0$ unless $g$ acts trivially on $K$. If $G$ has order four, the centralizer of $Q(T')$ is $K$, the center of $Q(B)$ is $K^G$, and $[Q(B):K^G]=4\cdot n'^2\cdot4$. If $G$ has order two, let $w$ be the element of $\{e,z,ez\}$ acting trivially on $K$. By Skolem--Noether $w\triangleright$ agrees on $Q(T')$ with conjugation by some $t\in Q(T')$, so $c=t^{-1}w$ centralizes $Q(T')$, $c^2\in K$, and $K'=K(c)$ is a field of degree two over $K$. Then $Q(T')K'$ is central simple of degree $n'$ over $K'$, the other generator acts faithfully on $K'$, and $[Q(B):K'^{G}]=2\cdot n'^2\cdot2$. $\square$

## 7.2 The relations

All formulas below follow from (4.1) and the rules of the table above; the proofs are in Appendix A. We write $\kappa$ for the scalar with $\rho\triangleright r=\kappa r$; $X$ commutes with $r$ except in $\mathbb B$, and $Y$ with $\rho$ except in $\mathbb B$ and $\mathbb E$.

*Type I, both sides balanced* ($\mathbb M$, $\mathbb O$): $$\begin{aligned}
e\triangleright r&=r^{-1}, & e\triangleright\rho&=\tfrac{\lambda_{21}}{\lambda_{11}}\rho^{-1}, & z\triangleright r&=\tfrac{\lambda_{22}}{\lambda_{21}}r^{-1}, & z\triangleright\rho&=\rho^{-1},\\
\kappa&=\tfrac{\lambda_{12}\lambda_{21}}{\lambda_{11}\lambda_{22}}, & z\triangleright X&=\lambda_{11}\lambda_{21}X, & e\triangleright Y&=\tfrac{1}{\lambda_{21}\lambda_{22}}Y, & Y\triangleright X&=(\lambda_{11}\lambda_{21})^2X,
\end{aligned}$$ and $X\rho=\rho X$, $Yr=rY$.

*Type I, both sides coordinate with $q=p$* ($\mathbb B$): $e\triangleright r=p^{-1}r$, $z\triangleright\rho=p^{-1}\rho$, $z\triangleright r=-r^{-1}$, $e\triangleright\rho=-\rho^{-1}$, $\kappa=-1$, $z\triangleright X=-ph^2Xr^2$, $e\triangleright Y=-ph^{-2}Y\rho^2$, $X\triangleright r=-r$, $Y\triangleright\rho=-\rho$, $Y\triangleright X=-h^4X$, and $X\rho=\rho X$, $Yr=rY$.

*Type I, $x$ balanced, $y$ coordinate* ($\mathbb E$): the same, except $z\triangleright\rho=p^{-1}\rho$, $Y\triangleright\rho=p^{-2}\rho=-\rho$ and $e\triangleright Y=\frac{p}{\lambda_{21}\lambda_{22}}Y\rho^2$.

*Type $0$, both balanced* ($\mathbb S$): $e\triangleright r=r^{-1}$, $z\triangleright\rho=\rho^{-1}$, $z\triangleright r=\frac{\lambda_{12}}{\lambda_{11}}r$, $e\triangleright\rho=\frac{\lambda_{11}}{\lambda_{21}}\rho$, $\kappa=\frac{\lambda_{11}\lambda_{22}}{\lambda_{12}\lambda_{21}}$, $X\triangleright\rho=(\frac{\lambda_{11}}{\lambda_{21}})^2\rho$, $Y\triangleright r=(\frac{\lambda_{12}}{\lambda_{11}})^2r$, $Y\triangleright X=\lambda_{11}^4X$.

*Type II* ($\mathbb P$, $\mathbb T$, $\mathbb Z$; $x$ balanced): $e\triangleright r=r^{-1}$, $z\triangleright r=\frac{\lambda_{22}}{\lambda_{21}}r$, $e\triangleright\rho=\frac{\lambda_{21}}{\lambda_{11}}\rho^{-1}$, $\kappa=\frac{\lambda_{12}\lambda_{21}}{\lambda_{11}\lambda_{22}}$, $z\triangleright X=\lambda_{11}\lambda_{21}X$, $Y\triangleright r=(\frac{\lambda_{22}}{\lambda_{21}})^2r$, $Y\triangleright X=(\lambda_{11}\lambda_{21})^2X$, $X\rho=\rho X$; and $z\triangleright\rho$ is $\rho^{-1}$, $-\rho$ or $\rho$ for $\mathbb T$, $\mathbb P$, $\mathbb Z$.

*Type III* ($\mathbb N$, $\mathbb U$, $\mathbb W$; $y$ balanced): $z\triangleright\rho=\rho^{-1}$, $z\triangleright r=\frac{\lambda_{12}}{\lambda_{11}}r^{-1}$, $\kappa=\frac{\lambda_{11}\lambda_{22}}{\lambda_{12}\lambda_{21}}$, $z\triangleright X$ equal to $\lambda_{11}^2X$, $-\lambda_{11}^2Xr^2$, $\lambda_{11}^2Xr^2$ for $\mathbb U$, $\mathbb N$, $\mathbb W$, $Y\triangleright X=(\lambda_{11}\lambda_{12})^2X$ for $\mathbb N$, $\mathbb W$ and $\lambda_{11}^4X$ for $\mathbb U$, $e\triangleright Y=(\lambda_{11}\lambda_{12})^{-1}Y$, $e\triangleright\rho=\frac{\lambda_{11}}{\lambda_{21}}\rho$; and $$e\triangleright r=\begin{cases}r^{-1}&(\mathbb U)\\ -r&(\mathbb N)\\ r&(\mathbb W)\end{cases}.$$

In [12]:
# Section 7.2 / Appendix A: every relation of the lists, in the monomial model, for symbolic h and s.
e, z, r, X, rho, Y = mono_e, mono_z, mono_r, mono_X, mono_rho, mono_Y
def lists(name, pp=None):
    M, l = mono_model(name, p=pp if pp is not None else p)
    l11, l12, l21, l22 = l
    nf, cj, inv, mul = M.nf, M.conj, M.inv, M.mul
    ri, rhoi = inv(r), inv(rho)
    R = {}
    if name == 'B':
        R = {'e>r': (e, r, sc(1 / pp, r)), 'z>rho': (z, rho, sc(1 / pp, rho)), 'z>r': (z, r, sc(-1, ri)),
             'e>rho': (e, rho, sc(-1, rhoi)), 'kappa': (rho, r, sc(-1, r)),
             'z>X': (z, X, sc(-pp * h**2, mul(X, mul(r, r)))), 'e>Y': (e, Y, sc(-pp / h**2, mul(Y, mul(rho, rho)))),
             'X>r': (X, r, sc(-1, r)), 'Y>rho': (Y, rho, sc(-1, rho)), 'Y>X': (Y, X, sc(-h**4, X)),
             'X>rho': (X, rho, rho), 'Y>r': (Y, r, r)}
    if name in 'MOE':
        R = {'e>r': (e, r, ri), 'e>rho': (e, rho, sc(l21 / l11, rhoi)), 'z>r': (z, r, sc(l22 / l21, ri)),
             'kappa': (rho, r, sc(l12 * l21 / (l11 * l22), r)), 'z>X': (z, X, sc(l11 * l21, X)),
             'Y>X': (Y, X, sc((l11 * l21)**2, X)), 'X>rho': (X, rho, rho), 'Y>r': (Y, r, r)}
        if name == 'E':
            R.update({'z>rho': (z, rho, sc(1 / pp, rho)), 'Y>rho': (Y, rho, sc(-1, rho)),
                      'e>Y': (e, Y, sc(pp / (l21 * l22), mul(Y, mul(rho, rho))))})
        else:
            R.update({'z>rho': (z, rho, rhoi), 'e>Y': (e, Y, sc(1 / (l21 * l22), Y))})
    if name == 'S':
        R = {'e>r': (e, r, ri), 'z>rho': (z, rho, rhoi), 'z>r': (z, r, sc(l12 / l11, r)),
             'e>rho': (e, rho, sc(l11 / l21, rho)), 'kappa': (rho, r, sc(l11 * l22 / (l12 * l21), r)),
             'X>rho': (X, rho, sc((l11 / l21)**2, rho)), 'Y>r': (Y, r, sc((l12 / l11)**2, r)),
             'Y>X': (Y, X, sc(l11**4, X))}
    if name in 'PTZ':
        R = {'e>r': (e, r, ri), 'z>r': (z, r, sc(l22 / l21, r)), 'e>rho': (e, rho, sc(l21 / l11, rhoi)),
             'kappa': (rho, r, sc(l12 * l21 / (l11 * l22), r)), 'z>X': (z, X, sc(l11 * l21, X)),
             'Y>r': (Y, r, sc((l22 / l21)**2, r)), 'Y>X': (Y, X, sc((l11 * l21)**2, X)), 'X>rho': (X, rho, rho),
             'z>rho': (z, rho, {'T': rhoi, 'P': sc(-1, rho), 'Z': rho}[name])}
    if name in 'NUW':
        R = {'z>rho': (z, rho, rhoi), 'z>r': (z, r, sc(l12 / l11, ri)),
             'kappa': (rho, r, sc(l11 * l22 / (l12 * l21), r)), 'e>rho': (e, rho, sc(l11 / l21, rho)),
             'e>Y': (e, Y, sc(1 / (l11 * l12), Y)),
             'z>X': (z, X, {'U': sc(l11**2, X), 'N': sc(-l11**2, mul(X, mul(r, r))), 'W': sc(l11**2, mul(X, mul(r, r)))}[name]),
             'Y>X': (Y, X, sc({'U': l11**4}.get(name, (l11 * l12)**2), X)),
             'e>r': (e, r, {'U': ri, 'N': sc(-1, r), 'W': r}[name])}
    bad = [k for k, (a_, b_, c_) in R.items() if not M.same(cj(a_, b_), nf(c_))]
    return check(f'{name}{"" if pp is None else " p=" + str(pp)}: {len(R)} relations' + (f'  FAILED {bad}' if bad else ''), not bad)
for name in 'MOSPTZNUW':
    lists(name)
for pp in (I, -I):
    lists('E', pp); lists('B', pp)

  ok   M: 10 relations
  ok   O: 10 relations
  ok   S: 8 relations
  ok   P: 9 relations
  ok   T: 9 relations
  ok   Z: 9 relations
  ok   N: 8 relations
  ok   U: 8 relations
  ok   W: 8 relations
  ok   E p=I: 11 relations
  ok   B p=I: 12 relations
  ok   E p=-I: 11 relations
  ok   B p=-I: 12 relations


## 7.3 The formulas

**Theorem 7.2.** Let $\varepsilon(u,v)=2$ if $u$ and $v$ both have odd order, and $\varepsilon(u,v)=4$ otherwise. For the PI members of the families below, $$\begin{array}{|c|c|c|}
\hline
\text{family} & \operatorname{PIdeg}& u,\ v\\\hline
\mathbb B & 8\operatorname{ord}(h^4) & \\
\mathbb S,\ \mathbb T,\ \mathbb U & 8\operatorname{ord}(16h^4) & \\
\mathbb E,\ \mathbb J & 8\operatorname{ord}(4h^4) & \\
\mathbb M & 4\operatorname{ord}(u^2)\operatorname{ord}(v^2) & u=\frac{s+1}{s-1},\ v=h^2(f-1)\\
\mathbb N & 4\operatorname{ord}(u^2)\operatorname{ord}(v^2) & u=\frac{f-g}{f+g},\ v=f^2-g^2\\
\mathbb O & \varepsilon(u,v)\operatorname{ord}(u^2)\operatorname{ord}(v^2) & u=\frac{1+s}{1-s},\ v=h^2(1-f)\\
\mathbb P & 4\operatorname{ord}(u^2)\operatorname{ord}(v^2) & u=\frac{1+s}{1-s},\ v=h^2(1-f)\\
\mathbb W & \varepsilon(u,v)\operatorname{ord}(u^2)\operatorname{ord}(v^2) & u=\frac{s-i}{s+i},\ v=h^2(f+1)\\
\mathbb Z & \varepsilon(u,v)\operatorname{ord}(u^2)\operatorname{ord}(v^2) & u=\frac{i-s}{i+s},\ v=h^2(f+1)\\\hline
\end{array}$$

*Proof.* In each case we first split $T'$ as a tensor product of two quantum tori of rank two, then decide whether $G$ has order four or two, and apply Proposition 7.1.

$\mathbb O$: here $(\lambda_{ab})=(hb',ha',ha',hb')$ with $a'=1+s$, $b'=1-s$, so $\kappa=u^2$ and $Y\triangleright X=v^2X$; $T'$ is the tensor product of the tori on $r,\rho$ and on $X,Y$, and $n'=\operatorname{ord}(u^2)\operatorname{ord}(v^2)$. The center $K$ is generated by $r^{\pm m},\rho^{\pm m},X^{\pm m'},Y^{\pm m'}$, $m=\operatorname{ord}(u^2)$, $m'=\operatorname{ord}(v^2)$. Both $e\triangleright$ and $z\triangleright$ invert $r^m$ or $\rho^m$, so they are not trivial on $K$, while $ez$ acts by $$r\mapsto u^{-1}r,\qquad \rho\mapsto u^{-1}\rho,\qquad X\mapsto vX,\qquad Y\mapsto v^{-1}Y ,$$ which is trivial on $K$ exactly when $u^m=v^{m'}=1$, that is, when $u$ and $v$ have odd order.

$\mathbb M$: now $(\lambda_{ab})=h(i(s+1),-i(s-1),i(s-1),-i(s+1))$, $\kappa=u^{-2}$ and $(\lambda_{11}\lambda_{21})^2=v^2$, so $n'$ is as before. But $ez$ sends $r\mapsto-ur$ and $\rho\mapsto u\rho$; triviality on $r^m$ and $\rho^m$ would need $(-1)^m=1$ and $u^m=1$, and $u^m=1$ with $m=\operatorname{ord}(u^2)$ forces $m$ odd. So $G$ has order four.

$\mathbb S$, $\mathbb T$, $\mathbb U$: $(\lambda_{ab})$ is $2h(1,-1,-1,-1)$, $h(2i,-2,-2i,-2)$, $h(2i,-2i,-2,-2)$. In all three $\kappa=-1$, and the remaining scalars are $Y\triangleright X=16h^4X$ together with $Y\triangleright r=-r$ in $\mathbb T$ and $X\triangleright\rho=-\rho$ in $\mathbb U$. Replacing $Y$ by $Y\rho$ in $\mathbb T$ and $X$ by $Xr$ in $\mathbb U$ gives the tensor product of the tori on $r,\rho$ (parameter $-1$) and on two generators with parameter $16h^4$, so $n'=2\operatorname{ord}(16h^4)$. Each nontrivial element of $G$ inverts $r$ or $\rho$, so $G$ has order four.

$\mathbb B$: here $(\lambda_{ab})=h(1,1,-1,1)$ and both bases are coordinate bases of $\Bbbk_p$. By the list of Section 7, $\kappa=-1$, $X\triangleright r=-r$, $Y\triangleright\rho=-\rho$ and $Y\triangleright X=-h^4X$, while $X$ commutes with $\rho$ and $Y$ with $r$. With $X'=X\rho$ and $Y'=Yr$ the torus splits into the tori on $r,\rho$ (parameter $-1$) and on $X',Y'$, where $Y'\triangleright X'=h^4X'$; so $n'=2\operatorname{ord}(h^4)$. The element $e$ inverts $\rho$, and $z$ and $ez$ invert $r$, so $G$ has order four.

$\mathbb E$: $\kappa=-1$, $Y\triangleright\rho=-\rho$, $Y\triangleright X=(\lambda_{11}\lambda_{21})^2X=4h^4X$. With $Y'=Yr$ the torus splits and $n'=2\operatorname{ord}(4h^4)$. $e$ and $z$ invert $r$, and $ez$ inverts $\rho$. For $\mathbb J$ use $\mathbb J\cong\mathbb E$ \[ZZ09, Remark 4.5\].

$\mathbb P$, $\mathbb Z$ (type II): $(\lambda_{ab})$ is $h(a',b',b',a')$ for $\mathbb P$ and $(A,B,B,A)$ with $A=ih(s-i)$, $B=-ih(s+i)$ for $\mathbb Z$, so $\kappa=u^{-2}$, $Y\triangleright r=u^2r$, $Y\triangleright X=v^2X$, and $Y'=Y\rho$ commutes with $r$ and $\rho$; hence $n'=\operatorname{ord}(u^2)\operatorname{ord}(v^2)$. The elements $e$ and $ez$ invert $r$. The element $z$ acts by $r\mapsto ur$, $X\mapsto vX$, and by $\rho\mapsto-\rho$ in $\mathbb P$, $\rho\mapsto\rho$ in $\mathbb Z$. In $\mathbb Z$ it is trivial on $K$ exactly when $u,v$ have odd order. In $\mathbb P$ it would need $u^m=1$ and $(-1)^m=1$, which is impossible as for $\mathbb M$.

$\mathbb N$, $\mathbb W$ (type III): $(\lambda_{ab})$ is $(a,b,-b,-a)$ with $a=f-g$, $b=f+g$ for $\mathbb N$, and $(A,B,-B,-A)$ for $\mathbb W$; so $\kappa=(\lambda_{11}/\lambda_{12})^2$, $X\triangleright\rho=\kappa\rho$ and $Y\triangleright X=(\lambda_{11}\lambda_{12})^2X$; hence $X'=Xr$ commutes with $\rho$. Hence $n'=\operatorname{ord}(u^2)\operatorname{ord}(v^2)$, where for $\mathbb W$ we note $(A/B)^2=u^2$ and $AB=v$. The elements $z$ and $ez$ invert $\rho$. The element $e$ acts by $\rho\mapsto(\lambda_{11}/\lambda_{21})\rho$, $Y\mapsto v^{-1}Y$, and $r\mapsto-r$, $X'\mapsto-X'$ in $\mathbb N$, $r\mapsto r$, $X'\mapsto X'$ in $\mathbb W$. In $\mathbb W$, $\lambda_{11}/\lambda_{21}=-A/B=u$ and $e$ is trivial on $K$ exactly when $u$ and $v$ have odd order. In $\mathbb N$, triviality needs $(-1)^m=1$ and $(-u)^m=1$ with $m=\operatorname{ord}(u^2)$ even, so $\operatorname{ord}u=2m$ and $u^m=-1$, a contradiction. $\square$

**Remark 7.3.** $\mathbb O$ and $\mathbb P$ share the criterion and the scalars $u,v$, yet for $u,v$ of odd order the PI degree of $\mathbb O$ is half that of $\mathbb P$. And $\mathbb W$, $\mathbb Z$ are $\Sigma$-$M$-dual \[ZZ09\] with different PI degrees: for $f=3$ and $h=-\tfrac12$, $\operatorname{PIdeg}\mathbb Z=6$ and $\operatorname{PIdeg}\mathbb W=12$. The duality involves a twist, and twists change PI degrees.

In [13]:
# Theorem 7.2 (and B, K, L): closed formulas against the PI degree of the crossed product
# computed directly from the model (central lattice of T' and action of e, z, ez), for random roots of unity.
import random
random.seed(1)
def eps(u, v): return 2 if order(u) % 2 and order(v) % 2 else 4
def ordr(c): return order(c)
bad, n = 0, 0
for a, b in [(3, 5), (4, 3), (6, 8), (5, 5), (3, 3), (8, 6), (12, 5), (7, 4)]:
    u, v = root(a, random.choice([1, 1, 2]) if a != 4 else 1), root(b)
    cases = []
    ss = (u - 1) / (u + 1); hh = sp.sqrt(v / (1 - ss**2))
    cases += [('O', hh, ss, lambda U, V: eps(U, V) * ordr(U**2) * ordr(V**2), lambda S: ((1 + S) / (1 - S), hh**2 * (1 - S**2))),
              ('P', hh, ss, lambda U, V: 4 * ordr(U**2) * ordr(V**2), lambda S: ((1 + S) / (1 - S), hh**2 * (1 - S**2)))]
    ss2 = (u + 1) / (u - 1); hh2 = sp.sqrt(v / (ss2**2 - 1))
    cases += [('M', hh2, ss2, lambda U, V: 4 * ordr(U**2) * ordr(V**2), lambda S: ((S + 1) / (S - 1), hh2**2 * (S**2 - 1)))]
    ss3 = I * (1 + u) / (1 - u); hh3 = sp.sqrt(v / (ss3**2 + 1))
    cases += [('W', hh3, ss3, lambda U, V: eps(U, V) * ordr(U**2) * ordr(V**2), lambda S: ((S - I) / (S + I), hh3**2 * (S**2 + 1)))]
    ss4 = I * (1 - u) / (1 + u); hh4 = sp.sqrt(v / (ss4**2 + 1))
    cases += [('Z', hh4, ss4, lambda U, V: eps(U, V) * ordr(U**2) * ordr(V**2), lambda S: ((I - S) / (I + S), hh4**2 * (S**2 + 1)))]
    for name, hv, sv, formula, uv in cases:
        M, _ = mono_model(name, h=hv)
        M.l = [complex(sp.N(c.subs(s, sv))) for c in M.l]
        U, V = uv(sv)
        n += 1; bad += (mono_pideg(M) != formula(U, V))
    fg = sp.sqrt(v / u); fv, gv = (fg + u * fg) / 2, (fg - u * fg) / 2
    M, _ = mono_model('N'); M.l = [complex(sp.N(c.subs({f: fv, g: gv}))) for c in M.l]
    n += 1; bad += (mono_pideg(M) != 4 * ordr(((fv - gv) / (fv + gv))**2) * ordr((fv**2 - gv**2)**2))
for k in (1, 2, 3, 4, 5, 6, 8):
    for name in 'STU':
        hv = (root(k) / 16)**sp.Rational(1, 4); M, _ = mono_model(name, h=hv)
        n += 1; bad += (mono_pideg(M) != 8 * ordr(16 * hv**4))
    for name in 'E':
        for pp in (I, -I):
            hv = (root(k) / 4)**sp.Rational(1, 4); M, _ = mono_model(name, h=hv, p=pp)
            n += 1; bad += (mono_pideg(M) != 8 * ordr(4 * hv**4))
    for pp in (I, -I):
        hv = root(k); M, _ = mono_model('B', h=hv, p=pp)
        n += 1; bad += (mono_pideg(M) != 8 * ordr(hv**4))
for kq, kf, kh in itertools.product((1, 2, 3), (1, 2, 4), (1, 2, 4, 6)):
    qv, fv, hv = root(kq), root(kf), root(kh)
    grp = math.lcm(2, kq, kf, kh)
    M, _ = mono_model('K', h=hv, q=qv); M.l = [complex(sp.N(c.subs(s, sp.sqrt(fv)))) for c in M.l]
    n += 1; bad += (mono_pideg(M) != 2 * grp)
    M, _ = mono_model('L', h=hv, q=qv); M.l = [complex(sp.N(c.subs(s, sp.sqrt(fv)))) for c in M.l]
    N0 = math.lcm(2, kq, kf)
    n += 1; bad += (mono_pideg(M) != (2 * grp if (grp // N0) % 2 else grp))
check(f'PI degree formulas on {n} members (B, E, K, L, M, N, O, P, S, T, U, W, Z)', bad == 0)

  ok   PI degree formulas on 169 members (B, E, K, L, M, N, O, P, S, T, U, W, Z)


True

# 8 The families $\mathbb F$, $\mathbb I$, $\mathbb Q$, $\mathbb R$ and $\mathbb V$

These five families have no monomial datum. They contain instead elements of bidegree $(1,1)$ or $(1,2)$ that pass through one plane the way a monomial would, and $Q(B)$ is again a crossed product over a quantum torus. Here a *quantum torus* in a division ring $D$ is a subalgebra $T'$ generated by units $v_1^{\pm1},\dots,v_4^{\pm1}$ that commute up to scalars and whose monomials are linearly independent; $Q(T')\subseteq D$ is its quotient division ring, $K=Z(Q(T'))$, and $a\triangleright b=aba^{-1}$.

## 8.1 Two lemmas

**Lemma 8.1.** Let $D$ be a division ring containing $\Bbbk$, $T'\subseteq D$ a quantum torus and $w_1=1,w_2,\dots,w_N\in D^\times$ such that

1.  each $w_g\triangleright$ sends every generator of $T'$ to a scalar multiple of a monomial of $T'$;

2.  for all $g,g'$ there is $g''$ with $w_gw_{g'}\in Q(T')\,w_{g''}$;

3.  the automorphisms of $K$ induced by $w_1\triangleright,\dots,w_N\triangleright$ are pairwise distinct;

4.  $D$ is generated as a division ring by $T'$ and the $w_g$.

Then $D=\bigoplus_gQ(T')w_g$, these automorphisms form a group $G$ of order $N$, and $Z(D)=K^G$. If $T'$ is PI of PI degree $n'$, then $D$ has dimension $(Nn')^2$ over its center; if $T'$ is not PI, neither is any subalgebra of $D$ with quotient division ring $D$.

*Proof.* By (i) each $w_g$ normalizes $Q(T')$, so by (ii) the sum $R=\sum_gQ(T')w_g$ is a subring, and the $N$ automorphisms of (iii) are closed under composition; being distinct, they form a group. Suppose $\sum_ga_gw_g=0$ with $a_g\in Q(T')$, not all zero and with the fewest nonzero terms, and let $a_{g_0}\ne0$. For $c\in K$, multiplying by $c$ on the right and by $g_0(c)$ on the left and subtracting gives $\sum_ga_g\bigl(g(c)-g_0(c)\bigr)w_g=0$, a shorter relation. So $g(c)=g_0(c)$ for all $c$ whenever $a_g\ne0$, and by (iii) only $a_{g_0}$ is nonzero, which is absurd. Hence the sum is direct, $R$ is a domain of finite dimension over $Q(T')$, so a division ring, and $R=D$ by (iv).

If $\sum_ga_gw_g$ commutes with $K$, then $a_g(g(c)-c)=0$ for all $c\in K$, so $a_g=0$ for $g\ne1$. Thus a central element of $D$ lies in $Q(T')$, hence in $K$, and it must be fixed by $G$; conversely $K^G$ is central. If $T'$ is PI, $[D:K^G]=N\cdot n'^2\cdot N$. The last claim is clear, since $Q(T')\subseteq D$. $\square$

When the lattice $\mathbb L$ of exponents of central monomials of $T'$ has finite index, $T'$ is free of rank $[\mathbb Z^4:\mathbb L]$ over its center (Lemma 11.5), so $n'^2=[\mathbb Z^4:\mathbb L]$.

**Lemma 8.2.** Let $E=Q(A_x)\subseteq Q(B)$ and let $Z_1,Z_2\in Ey_1+Ey_2$ be left linearly independent over $E$ with $Z_iE=EZ_i$. Then for every $k\ge0$ the words of length $k$ in $Z_1,Z_2$ span a left $E$-space of dimension $k+1$. In particular, $k+1$ words of length $k$ that span it are left independent over $E$. The same holds with the two planes and left and right exchanged.

*Proof.* Since $B$ is a free left $A_x$-module on the $y_1^cy_2^d$ and a domain, $E(A_y)_k=\sum_{c+d=k}Ey_1^cy_2^d$ has left dimension $k+1$. Put $M=Ey_1+Ey_2=EZ_1+EZ_2$. Then $ME=M$, because $Z_iE=EZ_i$. So $y_jE\subseteq M$, and $M^k$ is at the same time $\sum E\,y_{j_1}\cdots y_{j_k}=E(A_y)_k$ and the $E$-span of the words of length $k$ in $Z_1,Z_2$. $\square$

## 8.2 The families $\mathbb Q$ and $\mathbb V$

We keep the relations displayed before Proposition 5.4, multiplied by $h$. In $A_y=\Bbbk_{-1}[y_1,y_2]$ the squares $y_1^2,y_2^2$ are central, and we put $$\begin{aligned}
\Theta_{\mathbb Q}&=x_2(y_2^2-y_1^2)-x_1(y_1^2-2y_1y_2+y_2^2), & \Theta_{\mathbb V}&=x_1(y_1^2-y_2^2)-x_2(y_1^2+y_1y_2).\qquad(8.1)
\end{aligned}$$

**Lemma 8.3.** In $\mathbb Q(h)$ and $\mathbb V(h)$, $$\begin{aligned}
\mathbb Q:&\quad y_1\Theta=h\,\Theta y_1, & y_2\Theta&=h\,\Theta y_2, & \Theta x_1&=h^2x_1\Theta,\\
\mathbb V:&\quad y_1\Theta=h\,\Theta y_2, & y_2\Theta&=-h\,\Theta y_1, & \Theta x_2&=h^2x_2\Theta .
\end{aligned}$$

*Proof.* Write $a=y_1^2$, $b=y_2^2$, $w=y_1y_2$, so that $y_1w=ay_2$, $y_2w=-by_1$, $wy_1=-ay_2$, $wy_2=by_1$. It is enough to take $h=1$ in the first two columns, because $\Theta$ has $x$-degree one. In $\mathbb Q$, $$\begin{aligned}
y_1\Theta&=(x_1y_1+x_2y_1+x_1y_2)(b-a)-x_1y_2(a+b-2w)=x_2y_1(b-a)-x_1\bigl(y_1(a+b)+2ay_2\bigr),\\
\Theta y_1&=x_2(b-a)y_1-x_1\bigl((a+b)y_1-2wy_1\bigr)=x_2y_1(b-a)-x_1\bigl((a+b)y_1+2ay_2\bigr),\\
y_2\Theta&=(x_1y_1-x_1y_2+x_2y_2)(b-a)+x_1y_1(a+b-2w)=x_2y_2(b-a)+x_1\bigl(2by_1-(a+b)y_2\bigr),\\
\Theta y_2&=x_2(b-a)y_2-x_1\bigl((a+b)y_2-2by_1\bigr).
\end{aligned}$$ In $\mathbb V$, $$\begin{aligned}
y_1\Theta&=(x_2y_1+x_1y_2)(a-b)-x_2y_1(a+w)=x_1y_2(a-b)-x_2(by_1+ay_2)=\Theta y_2,\\
y_2\Theta&=(-x_1y_1+x_2y_1)(a-b)-x_2y_2(a+w)=-x_1y_1(a-b)+x_2(ay_1-ay_2)=-\Theta y_1 .
\end{aligned}$$ For the last column, in $\mathbb Q$ one has $ax_1=h^2x_1b$, $bx_1=h^2x_1a$ and $wx_1=h^2x_1w$, so $$\begin{aligned}
\Theta x_1&=x_2(b-a)x_1-x_1(a+b-2w)x_1=-h^2x_2x_1(b-a)-h^2x_1^2(a+b-2w)=h^2x_1\Theta .
\end{aligned}$$ In $\mathbb V$, $x_2$ commutes with $x_1$ and $cx_2=h^2x_2c$ for every $c\in(A_y)_2$. $\square$

Let $z_\pm=y_1\pm iy_2$, a balanced basis of $A_y$, and $$\begin{aligned}
z&=z_+, & \rho&=z_+^{-1}z_-, & Y&=z_+^2=y_1^2-y_2^2 ,
\end{aligned}$$ so that $z\triangleright\rho=\rho^{-1}$, and $Y$ is central in $A_y$. From the relations and Lemma 8.3, $$\begin{aligned}
\mathbb Q:&\quad z_\pm x_1=\mp ih\,x_1z_\pm, & z_\pm\Theta&=h\,\Theta z_\pm, & \Theta x_1&=h^2x_1\Theta,\qquad(8.2)\\
\mathbb V:&\quad z_\pm x_2=h\,x_2z_\pm, & z_\pm\Theta&=\mp ih\,\Theta z_\pm, & \Theta x_2&=h^2x_2\Theta .\qquad(8.3)
\end{aligned}$$

**Theorem 8.4.** Let $n$ be the normal generator ($x_1$ in $\mathbb Q$, $x_2$ in $\mathbb V$). The elements $n,\Theta,\rho,Y$ generate a quantum torus $T'$ in $Q(B)$, and $Q(B)=Q(T')\oplus Q(T')z$. If $h\in\mu_\infty$, then $$\operatorname{PIdeg}\mathbb Q(h)=\operatorname{PIdeg}\mathbb V(h)=4\operatorname{ord}(h^2).$$

*Proof.* From (8.2) and (8.3), $$\begin{aligned}
\mathbb Q:&\quad \rho x_1=-x_1\rho, & \rho\Theta&=\Theta\rho, & Yx_1&=-h^2x_1Y, & Y\Theta&=h^2\Theta Y,\\
\mathbb V:&\quad \rho x_2=x_2\rho, & \rho\Theta&=-\Theta\rho, & Yx_2&=h^2x_2Y, & Y\Theta&=-h^2\Theta Y,
\end{aligned}$$ and $Y\rho=\rho Y$. The monomials are independent. Indeed, let $x'$ be the generator other than $n$. Moving an element of $A_y$ past $x'$ produces $x'$ times an element of $A_y$ of the same degree, plus terms in $nA_y$. So by (8.1) and induction, $\Theta^b\in x'^bc_b+\sum_{j<b}n^{b-j}x'^jA_y$ with $0\ne c_b\in A_y$. Hence the $n^a\Theta^b$, $a\in\mathbb Z$, $b\ge0$, are right independent over $Q(A_y)$, and the $\rho^cY^d$ are independent in $Q(A_y)$ as in Proposition 7.1. Negative powers of $\Theta$ are removed by multiplying a relation by a power of $\Theta$.

Let $m=\operatorname{ord}(h^2)$. A monomial $n^a\Theta^b\rho^cY^d$ is central in $T'$ if and only if $$\begin{aligned}
\mathbb Q:&\quad a\ \text{even},\quad c\equiv d\ (2),\quad b\equiv-a,\ d\equiv a\ (m);\\
\mathbb V:&\quad b\ \text{even},\quad c\equiv d\ (2),\quad a\equiv-b,\ d\equiv-b\ (m).
\end{aligned}$$ In both cases the lattice has index $4m^2$, so $\operatorname{PIdeg}T'=2m$.

We apply Lemma 8.1 with $w\in\{1,z\}$. Conjugation by $z$ multiplies $n$ and $\Theta$ by scalars, fixes $Y$ and inverts $\rho$; and $z^2=Y$. Since $\rho^2\in K$ and $\rho^2\ne\rho^{-2}$, $z$ acts nontrivially on $K$. Finally $T'$ and $z$ generate $Q(B)$: $z_-=z\rho$, the elements $y_1^2,y_2^2,y_1y_2$ lie in the span of $Y$, $Y\rho$, $Y\rho^{-1}$, and $x'$ is recovered from (8.1). So $\operatorname{PIdeg}B=2\cdot2m$. $\square$

By Lemma 8.1, the same model gives back the criterion of Proposition 5.4.

In [14]:
# Lemma 8.3 (exact, symbolic h).
A = family('Q'); x1, x2, y1, y2 = gens(A)
a_, b_, w_ = A.mul(y1, y1), A.mul(y2, y2), A.mul(y1, y2)
Th = add((1, A.mul(x2, add((1, b_), (-1, a_)))), (-1, A.mul(x1, add((1, a_), (1, b_), (-2, w_)))))
check('Q: y_j Theta = h Theta y_j', skew(A, y1, Th, h) and skew(A, y2, Th, h))
check('Q: Theta x1 = h^2 x1 Theta', skew(A, Th, x1, h**2))
A = family('V'); x1, x2, y1, y2 = gens(A)
a_, b_, w_ = A.mul(y1, y1), A.mul(y2, y2), A.mul(y1, y2)
Th = add((1, A.mul(x1, add((1, a_), (-1, b_)))), (-1, A.mul(x2, add((1, a_), (1, w_)))))
check('V: y1 Theta = h Theta y2', is_zero(add((1, A.mul(y1, Th)), (-h, A.mul(Th, y2)))))
check('V: y2 Theta = -h Theta y1', is_zero(add((1, A.mul(y2, Th)), (h, A.mul(Th, y1)))))
check('V: Theta x2 = h^2 x2 Theta', skew(A, Th, x2, h**2))

  ok   Q: y_j Theta = h Theta y_j
  ok   Q: Theta x1 = h^2 x1 Theta
  ok   V: y1 Theta = h Theta y2
  ok   V: y2 Theta = -h Theta y1
  ok   V: Theta x2 = h^2 x2 Theta


True

## 8.3 The family $\mathbb R$

The relations of $\mathbb R(h)$ are $x_2x_1=-x_1x_2$, $y_2y_1=-y_1y_2$ and $$\begin{aligned}
y_1x_1&=h(x_1y_1+x_2y_1+x_1y_2), & y_1x_2&=h\,x_1y_2,\\
y_2x_1&=h\,x_2y_1, & y_2x_2&=h(-x_2y_1-x_1y_2+x_2y_2).
\end{aligned}$$ Let $e_\pm=x_1\pm x_2$ and $z_\pm=y_1\pm y_2$. Adding and subtracting the relations, $$\begin{aligned}
z_+x_1&=h(e_+z_++x_2z_-), & z_+x_2&=-h\,x_2z_-, & z_-x_1&=h\,x_1z_+, & z_-x_2&=h(x_1z_+-e_-z_-).\qquad(8.4)
\end{aligned}$$

**Lemma 8.5.** In $\mathbb R(h)$ let $$\begin{aligned}
Z_1&=x_1z_++x_2z_-=(x_1+x_2)y_1+(x_1-x_2)y_2, & Z_2&=x_1y_2+x_2y_1 .
\end{aligned}$$ Then $$\begin{aligned}
Z_1x_1&=h\,x_1Z_1, & Z_1x_2&=h\,x_2Z_1, & Z_2x_1&=h\,x_2Z_2, & Z_2x_2&=-h\,x_1Z_2, & Z_2Z_1&=Z_1Z_2 .
\end{aligned}$$

*Proof.* Write $a=x_1^2$, $b=x_2^2$, $u=x_1x_2$. By (8.4), $$\begin{aligned}
Z_1x_1&=h\bigl(x_1e_+z_++x_1x_2z_-+x_2x_1z_+\bigr)=h\bigl(a\,z_++u\,z_-\bigr)=h\,x_1Z_1,\\
Z_1x_2&=h\bigl(-x_1x_2z_-+x_2x_1z_+-x_2e_-z_-\bigr)=h\bigl(-u\,z_++b\,z_-\bigr)=h\,x_2Z_1 .
\end{aligned}$$ Note that $2Z_2=e_+z_+-e_-z_-$. Since $e_+^2-e_-x_1=x_2e_+$, $e_+x_2=-x_2e_-$, $e_-x_1=x_1e_+$ and $e_-^2-e_+x_2=x_1e_-$, $$\begin{aligned}
2Z_2x_1&=h\bigl(e_+^2z_++e_+x_2z_--e_-x_1z_+\bigr)=h\,x_2(e_+z_+-e_-z_-),\\
2Z_2x_2&=h\bigl(-e_+x_2z_--e_-x_1z_++e_-^2z_-\bigr)=-h\,x_1(e_+z_+-e_-z_-).
\end{aligned}$$ For the last relation put $Y=z_+^2=z_-^2$ and $v=a+b$. By (8.4), $z_+e_+=he_+z_+$, $z_-e_-=he_-z_-$, $z_+e_-=h(e_+z_++2x_2z_-)$ and $z_-e_+=h(2x_1z_+-e_-z_-)$. Hence $$\begin{aligned}
2Z_1Z_2&=x_1z_+e_+z_+-x_1z_+e_-z_-+x_2z_-e_+z_+-x_2z_-e_-z_-\\
&=h\bigl((x_1e_+-2x_1x_2+2x_2x_1-x_2e_-)Y-x_1e_+\,z_+z_--x_2e_-\,z_-z_+\bigr)\\
&=h\bigl((v-2u)Y-(a+u)z_+z_-+(b+u)z_-z_+\bigr),\\
2Z_2Z_1&=e_+z_+x_1z_++e_+z_+x_2z_--e_-z_-x_1z_+-e_-z_-x_2z_-\\
&=h\bigl((e_+^2-e_+x_2-e_-x_1+e_-^2)Y+e_+x_2\,z_-z_+-e_-x_1\,z_+z_-\bigr)\\
&=h\bigl((v-2u)Y+(b+u)z_-z_+-(a+u)z_+z_-\bigr),
\end{aligned}$$ because $x_1e_+=e_-x_1=a+u$, $-x_2e_-=e_+x_2=b+u$ and $e_\pm^2=v$. $\square$

**Theorem 8.6.** In $Q(\mathbb R(h))$ let $X=x_1^2$, $r=x_1^{-1}x_2$ and $W=Z_2^2$. Then $X,r,Z_1,W$ generate a quantum torus $T'$ whose only nontrivial relations are $Z_1X=h^2XZ_1$ and $WX=h^4XW$, and $$Q(\mathbb R)=Q(T')\oplus Q(T')x_1\oplus Q(T')Z_2\oplus Q(T')x_1Z_2 .$$ $\mathbb R(h)$ is PI if and only if $h\in\mu_\infty$, and then $\operatorname{PIdeg}\mathbb R(h)=4\operatorname{ord}(h^2)$.

*Proof.* $Z_1$ and $Z_2$ normalize $E=Q(A_x)$ by Lemma 8.5, and they are left independent over $E$, because they induce different automorphisms of the center $\Bbbk(x_1^2,x_2^2)$ of $E$. So Lemma 8.2 applies: the words $Z_1^aZ_2^b$ are left independent over $E$, and since the $X^ir^j$ are independent in $E$, the monomials of $T'$ are independent. The relations of $T'$ follow from Lemma 8.5: $Z_1\triangleright$ multiplies $x_1,x_2$ by $h$, and $Z_2^2\triangleright$ multiplies them by $-h^2$. A monomial $X^ir^jZ_1^kW^l$ is central if and only if $m\mid i$ and $m\mid k+2l$, where $m=\operatorname{ord}(h^2)$. So $\operatorname{PIdeg}T'=m$ when $h\in\mu_\infty$, and $T'$ is not PI otherwise.

The elements $x_1$ and $Z_2$ act on the generators by $$\begin{array}{c|cccc}
 & r & X & Z_1 & W\\\hline
x_1\triangleright & -r & X & h^{-1}Z_1 & -h^{-2}W\\
Z_2\triangleright & -r^{-1} & -h^2r^2X & Z_1 & W
\end{array}$$ Here $x_1\triangleright Z_2=h^{-1}Z_2r$ and $(Z_2r)^2=-W$. Moreover $x_1^2=X$, $Z_2^2=W$ and $Z_2x_1=h\,x_1rZ_2$. The element $r$ lies in $K$, and $x_1$, $Z_2$, $x_1Z_2$ send it to $-r$, $-r^{-1}$, $r^{-1}$. Since $x_2=x_1r$ and $y_1,y_2\in EZ_1+EZ_2$, Lemma 8.1 applies with $N=4$. $\square$

This recovers \[ZZ08, Corollary 4.8\] and adds the PI degree.

In [15]:
# Lemma 8.4 (exact, symbolic h).
A = family('R'); x1, x2, y1, y2 = gens(A)
Z1 = add((1, A.mul(add((1, x1), (1, x2)), y1)), (1, A.mul(add((1, x1), (-1, x2)), y2)))
Z2 = add((1, A.mul(x1, y2)), (1, A.mul(x2, y1)))
check('Z1 x_i = h x_i Z1', skew(A, Z1, x1, h) and skew(A, Z1, x2, h))
check('Z2 x1 = h x2 Z2', is_zero(add((1, A.mul(Z2, x1)), (-h, A.mul(x2, Z2)))))
check('Z2 x2 = -h x1 Z2', is_zero(add((1, A.mul(Z2, x2)), (h, A.mul(x1, Z2)))))
check('Z2 Z1 = Z1 Z2', skew(A, Z2, Z1, 1))

  ok   Z1 x_i = h x_i Z1
  ok   Z2 x1 = h x2 Z2
  ok   Z2 x2 = -h x1 Z2
  ok   Z2 Z1 = Z1 Z2


True

## 8.4 The families $\mathbb F$ and $\mathbb I$

In $\mathbb F(p,h)$, with the relations of Section 6, let $$\begin{aligned}
g&=x_1+px_2, & \bar g&=x_1-px_2, & c&=h(1-p),
\end{aligned}$$ so that $g^2=\bar g^2=x_1^2-x_2^2$, and $c^2=-2ph^2$ is the scalar $\lambda$ of Lemma 6.2. Using $(p-1)/(p+1)=p$, the mixed relations become $$\begin{aligned}
y_1g&=h(1+p)\,gy_2, & y_1\bar g&=h\bigl(-2gy_1+(1-p)\bar gy_2\bigr),\\
y_2\bar g&=-h(1+p)\,\bar gy_1, & y_2g&=h\bigl((1-p)gy_1+2p\,\bar gy_2\bigr).\qquad(8.5)
\end{aligned}$$

**Lemma 8.7.** In $\mathbb F(p,h)$ let $Z_1=gy_1-\bar gy_2$ and $Z_2=gy_1+p\,\bar gy_2$. Then $$\begin{aligned}
Z_1x_1&=-c\,x_1Z_1, & Z_1x_2&=c\,x_2Z_1, & Z_2x_1&=c\,x_2Z_2, & Z_2x_2&=c\,x_1Z_2, & Z_2^2&=-p\,Z_1^2 .
\end{aligned}$$

*Proof.* Let $\omega_0=g^2=\bar g^2$, which is central in $A_x$. By (8.5), $$\begin{aligned}
Z_1g&=h(1+p)\omega_0 y_2-h\bar g\bigl((1-p)gy_1+2p\bar gy_2\bigr)=-h(1-p)\,\bar g(gy_1-\bar gy_2)=-c\,\bar gZ_1,\\
Z_1\bar g&=hg\bigl(-2gy_1+(1-p)\bar gy_2\bigr)+h(1+p)\omega_0 y_1=-h(1-p)\,g(gy_1-\bar gy_2)=-c\,gZ_1,\\
Z_2g&=h(1+p)\omega_0 y_2+hp\,\bar g\bigl((1-p)gy_1+2p\bar gy_2\bigr)=hp(1-p)\,\bar g(gy_1+p\bar gy_2)=cp\,\bar gZ_2,\\
Z_2\bar g&=hg\bigl(-2gy_1+(1-p)\bar gy_2\bigr)-hp(1+p)\omega_0 y_1=-hp(1-p)\,g(gy_1+p\bar gy_2)=-cp\,gZ_2 .
\end{aligned}$$ The first four relations follow, since $2x_1=g+\bar g$ and $2px_2=g-\bar g$. For the last one put $G_1=gy_1$, $G_2=\bar gy_2$. By (8.5) and $y_2y_1=py_1y_2$, $$\begin{aligned}
G_1^2&=h(p-1)\,\omega_0\,y_1y_2, & G_1G_2&=h\bigl(-2\omega_0\,y_1y_2+(1-p)g\bar g\,y_2^2\bigr),\\
G_2^2&=-h(1+p)\,\omega_0\,y_1y_2, & G_2G_1&=h\bigl((1-p)\bar gg\,y_1^2-2\omega_0\,y_1y_2\bigr).
\end{aligned}$$ So $G_1^2-G_2^2=2hp\,\omega_0 y_1y_2=-p(G_1^2+G_2^2)$, and with $S=G_1G_2+G_2G_1$, $$\begin{aligned}
Z_2^2&=G_1^2-G_2^2+pS=-p\bigl(G_1^2+G_2^2-S\bigr)=-p\,Z_1^2 .
\end{aligned}$$ $\square$

The relations of $\mathbb I(q,h)$, $q^2=-1$, are $x_2x_1=q\,x_1x_2$, $y_2y_1=-y_1y_2$ and $$\begin{aligned}
y_1x_1&=h(-qx_1y_1-qx_2y_1+x_1y_2-qx_2y_2), & y_1x_2&=h(x_1y_1+x_2y_1+x_1y_2-qx_2y_2),\\
y_2x_1&=h(x_1y_1+qx_2y_1+qx_1y_2-qx_2y_2), & y_2x_2&=h(-x_1y_1-qx_2y_1+x_1y_2-x_2y_2).
\end{aligned}$$ With $w_1=y_2-qy_1$ and $w_2=-(y_2+qy_1)$, so that $w_1^2=w_2^2=y_2^2-y_1^2$, they become $$\begin{aligned}
w_1x_1&=-h(1+q)\,x_2w_1, & w_1x_2&=h\bigl((1-q)x_1w_1+2x_2w_2\bigr),\\
w_2x_2&=h(1+q)\,x_1w_2, & w_2x_1&=h\bigl(-2q\,x_1w_1+(1-q)x_2w_2\bigr).\qquad(8.6)
\end{aligned}$$

**Lemma 8.8.** In $\mathbb I(q,h)$ let $Z_1=x_1w_1+x_2w_2$, $Z_2=x_1w_1+q\,x_2w_2$ and $c=h(1-q)$. Then $$\begin{aligned}
y_1Z_1&=c\,Z_1y_1, & y_2Z_1&=-c\,Z_1y_2, & y_1Z_2&=c\,Z_2y_2, & y_2Z_2&=c\,Z_2y_1, & Z_2^2&=q\,Z_1^2 .
\end{aligned}$$

*Proof.* This is the computation of Lemma 8.7 with the two planes exchanged. Let $\omega_0=w_1^2=w_2^2$, $u=x_1x_2$, $H_1=x_1w_1$ and $H_2=x_2w_2$. From (8.6), $$\begin{aligned}
w_1Z_1&=-h(1+q)x_2\omega_0+h\bigl((1-q)x_1w_1+2x_2w_2\bigr)w_2=h(1-q)(x_1w_1+x_2w_2)w_2=c\,Z_1w_2,
\end{aligned}$$ and in the same way $w_2Z_1=c\,Z_1w_1$, $w_1Z_2=cq\,Z_2w_2$ and $w_2Z_2=-cq\,Z_2w_1$. These give the first four relations, because $2y_2=w_1-w_2$ and $2qy_1=-(w_1+w_2)$. For the last one, $$\begin{aligned}
H_1^2&=-h(1+q)\,u\omega_0, & H_2^2&=hq(1+q)\,u\omega_0,
\end{aligned}$$ so $H_1^2+q^2H_2^2=q(H_1^2+H_2^2)$, and $Z_2^2=H_1^2+q^2H_2^2+q(H_1H_2+H_2H_1)=q\,Z_1^2$. $\square$

**Theorem 8.9.** Let $B$ be $\mathbb F(p,h)$ and $\delta^2=-p$, or $\mathbb I(q,h)$ and $\delta^2=q$, and put $Z_2'=\delta^{-1}Z_2$, so that $Z_2'^2=Z_1^2$. In $\mathbb F$ let $$\begin{aligned}
X&=x_1^2, & r&=x_1^{-1}x_2, & Y&=Z_1^2, & \rho&=Z_1^{-1}Z_2', & P&=\rho^2 .
\end{aligned}$$ Then $X,r,Y,P$ generate a quantum torus $T'$ whose only nontrivial relation is $YX=c^4XY$, and $$Q(\mathbb F)=\bigoplus_{i,j,k\in\{0,1\}}Q(T')\,x_1^iZ_1^j\rho^k .$$ The same holds for $\mathbb I$ with $y_1,y_2$ in place of $x_1,x_2$ and left and right exchanged. If $2h^2\in\mu_\infty$, then $$\operatorname{PIdeg}\mathbb F(p,h)=\operatorname{PIdeg}\mathbb I(q,h)=8\operatorname{ord}(-4h^4).$$

*Proof.* We treat $\mathbb F$; for $\mathbb I$ apply the same argument to the opposite algebra. As in Theorem 8.6, $Z_1,Z_2$ normalize $E=Q(A_x)$ and are left independent over $E$. Since $Y$ commutes with $Z_1$ and $Z_2'$, every word of length $k$ in $Z_1,Z_2'$ is a power of $Y$ times an alternating word, and there are $k+1$ such products of length $k$. By Lemma 8.2 they are left independent over $E$. Now $Y^jP^l=Y^{j-2l}(Z_1Z_2')^{2l}$, because $\rho=Y^{-1}Z_1Z_2'$, so the monomials of $T'$ are independent.

By Lemma 8.7, $Z_1\triangleright\rho=\rho^{-1}$ and $$\begin{array}{c|cccc}
 & r & X & Y & P\\\hline
x_1\triangleright & -r & X & c^{-2}Y & -P\\
Z_1\triangleright & -r & c^2X & Y & P^{-1}\\
\rho\triangleright & -r^{-1} & -r^2X & Y & P
\end{array}$$ For the first row, $x_1\triangleright Z_1=-c^{-1}Z_1$ and $x_1\triangleright Z_2'=-c^{-1}Z_2'r$, so $x_1\triangleright\rho=\rho r$, and $(\rho r)^2=-P$ because $\rho r=-r^{-1}\rho$. For the third, $\rho\triangleright x_1=x_2$ and $\rho\triangleright x_2=-x_1$. In particular $P\triangleright$ fixes $r$ and $X$, while $Y\triangleright$ fixes $r$ and multiplies $X$ by $c^4$. These are the relations of $T'$, and since $c^4=-4h^4$ we get $\operatorname{PIdeg}T'=\operatorname{ord}(-4h^4)$.

The eight elements $x_1^iZ_1^j\rho^k$ satisfy (i) and (ii) of Lemma 8.1: $x_1^2=X$, $Z_1^2=Y$, $\rho^2=P$, and they commute with each other up to units of $T'$. The elements $r,P$ lie in $K$, and $$\begin{array}{c|ccccccc}
 & x_1 & Z_1 & \rho & x_1Z_1 & x_1\rho & Z_1\rho & x_1Z_1\rho\\\hline
r\mapsto & -r & -r & -r^{-1} & r & r^{-1} & r^{-1} & -r^{-1}\\
P\mapsto & -P & P^{-1} & P & -P^{-1} & -P & P^{-1} & -P^{-1}
\end{array}$$ so the eight automorphisms of $K$ are distinct. Since $x_2=x_1r$, $Z_2=\delta Z_1\rho$ and $y_1,y_2\in EZ_1+EZ_2$, Lemma 8.1 gives $\operatorname{PIdeg}\mathbb F=8\operatorname{ord}(-4h^4)$. $\square$

Since $-4h^4=-(2h^2)^2$, Lemma 8.1 also recovers Theorems 6.3 and 6.4.

**Theorem 8.10.** By Theorems 8.4, 8.6 and 8.9, for the PI members $$\begin{aligned}
\operatorname{PIdeg}\mathbb Q(h)&=\operatorname{PIdeg}\mathbb R(h)=\operatorname{PIdeg}\mathbb V(h)=4\operatorname{ord}(h^2),\\
\operatorname{PIdeg}\mathbb F(p,h)&=\operatorname{PIdeg}\mathbb I(q,h)=8\operatorname{ord}(-4h^4).
\end{aligned}$$

**Remark 8.11.** The quantum affine space $S$ of the proof of Theorem 6.3 gives less. In its quantum torus $\omega_1\omega_2^{-1}$ is central, and the remaining generators $\omega_2,y_1,y_2$ have commutation scalars $\lambda$ and $p$; so $\operatorname{PIdeg}S=|\langle\lambda,p\rangle|=\operatorname{lcm}(4,\operatorname{ord}(2h^2))$, and Lemma 2.2 with (6.2) gives $\operatorname{PIdeg}\mathbb F\le4\operatorname{lcm}(4,\operatorname{ord}(2h^2))$. This bound is attained for $2h^2=1$, where both sides are $16$, and not for $2h^2=\pm i$, where $\operatorname{PIdeg}\mathbb F=8$.

In [16]:
# Lemmas 8.7 and 8.8 (exact, symbolic h).
for pp in (I, -I):
    A = family('F', p=pp); x1, x2, y1, y2 = gens(A); c = h * (1 - pp)
    g_, gb = lin(A, 1, pp, 0, 1), lin(A, 1, -pp, 0, 1)
    Z1 = add((1, A.mul(g_, y1)), (-1, A.mul(gb, y2))); Z2 = add((1, A.mul(g_, y1)), (pp, A.mul(gb, y2)))
    ok = (is_zero(add((1, A.mul(Z1, x1)), (c, A.mul(x1, Z1)))) and is_zero(add((1, A.mul(Z1, x2)), (-c, A.mul(x2, Z1))))
          and is_zero(add((1, A.mul(Z2, x1)), (-c, A.mul(x2, Z2)))) and is_zero(add((1, A.mul(Z2, x2)), (-c, A.mul(x1, Z2))))
          and is_zero(add((1, A.mul(Z2, Z2)), (pp, A.mul(Z1, Z1)))))
    check(f'Lemma 8.7, p = {pp}', ok)
for qq in (I, -I):
    A = family('I', q=qq); x1, x2, y1, y2 = gens(A); c = h * (1 - qq)
    w1, w2 = lin(A, -qq, 1, 2, 3), lin(A, -qq, -1, 2, 3)
    Z1 = add((1, A.mul(x1, w1)), (1, A.mul(x2, w2))); Z2 = add((1, A.mul(x1, w1)), (qq, A.mul(x2, w2)))
    ok = (is_zero(add((1, A.mul(y1, Z1)), (-c, A.mul(Z1, y1)))) and is_zero(add((1, A.mul(y2, Z1)), (c, A.mul(Z1, y2))))
          and is_zero(add((1, A.mul(y1, Z2)), (-c, A.mul(Z2, y2)))) and is_zero(add((1, A.mul(y2, Z2)), (-c, A.mul(Z2, y1))))
          and is_zero(add((1, A.mul(Z2, Z2)), (-qq, A.mul(Z1, Z1)))))
    check(f'Lemma 8.8, q = {qq}', ok)

  ok   Lemma 8.7, p = I
  ok   Lemma 8.7, p = -I
  ok   Lemma 8.8, q = I
  ok   Lemma 8.8, q = -I


# 9 The family $\mathbb C$ and a summary of PI degrees

**Proposition 9.1.** If $\mathbb C(p,h)$ is PI, then $$\begin{aligned}
\operatorname{lcm}\bigl(3,\operatorname{ord}(\beta)\bigr)&\le\operatorname{PIdeg}\mathbb C(p,h)\le9\operatorname{lcm}\bigl(3,\operatorname{ord}(\beta)\bigr),\qquad\beta=3(2p+1)h^3 .
\end{aligned}$$

*Proof.* The lower bound is the PI degree of the quantum affine space $S$ of the proof of Theorem 6.5. Write $p=\zeta^{k}$, $\beta=\zeta^{l}$ and $\alpha=\zeta^{l+2k}$ for a root of unity $\zeta$. The commutation matrix of $S$ in the order $x_1,x_2,a,b$ has entries $-k,\ -(l+2k),\ -l,\ -(l+2k),\ -l,\ 0$ and Pfaffian $0$, so the PI degree is $|\langle p,\beta\rangle|$. The upper bound is Lemma 2.2: the rank of $\mathbb C$ over $S$ is nine, as one sees from the bigraded Hilbert series $(1-t_1)^{-2}(1-t_2)^{-2}$ of $\mathbb C$ and $(1-t_1)^{-2}(1-t_2^3)^{-2}$ of $S$. $\square$

Table 3 collects what is known. In the formulas for $\mathbb D$ and $\mathbb G$, $\zeta$ is a primitive $N$th root of unity, $p=\zeta^k$ and $h=\zeta^{k'}$.

|                                          Family                                           | $\operatorname{PIdeg}$                                                         | Reference       |
|:-----------------------------------------------------------------------------------------:|:-------------------------------------------------------------------------------|:----------------|
|                                        $\mathbb D$                                        | $N^2/\gcd\bigl(N,2k(k+k')\bigr)$, $N=|\langle-1,p,h\rangle|$                   | \[GV-D\]        |
|                                        $\mathbb G$                                        | $N^2/\gcd\bigl(N,2k(k+k')\bigr)$, $N=|\langle p,h\rangle|$, independent of $f$ | \[GV-G\]        |
|                                        $\mathbb K$                                        | $2|\langle-1,q,f,h\rangle|$                                                    | \[GV-K\]        |
|                                        $\mathbb L$                                        | $2N$ or $N$, $N=|\langle-1,q,f,h\rangle|$                                      | Theorem 5.5     |
| $\mathbb B$, $\mathbb E$, $\mathbb J$, $\mathbb M$, $\mathbb N$, $\mathbb O$, $\mathbb P$ | see Theorem 7.2                                                                | Theorem 7.2     |
|              $\mathbb S$, $\mathbb T$, $\mathbb U$, $\mathbb W$, $\mathbb Z$              | see Theorem 7.2                                                                | Theorem 7.2     |
|                                 $\mathbb F$, $\mathbb I$                                  | $8\operatorname{ord}(-4h^4)$                                                   | Theorem 8.10    |
|                           $\mathbb Q$, $\mathbb R$, $\mathbb V$                           | $4\operatorname{ord}(h^2)$                                                     | Theorem 8.10    |
|                                        $\mathbb C$                                        | bounds only                                                                    | Proposition 9.1 |

*Table 3. PI degrees of the PI members.*

**Question 9.2.** What is the PI degree of $\mathbb C(p,h)$?

As far as we can see, the method of Section 8 does not reach $\mathbb C$: we know no element of low bidegree normalizing one of its planes, and both planes are $\Bbbk_p$ with $p$ of order three, so their quotient rings have degree three, not two.

In [17]:
# Cross-check of Theorem 8.10 with PI degrees obtained earlier, independently, by computing the
# centers numerically (Hilbert series and rank over a system of parameters, previous draft).
data = [('Q', -1, 4), ('Q', 1, 4), ('Q', I, 8), ('R', -1, 4), ('R', 1, 4), ('R', I, 8), ('V', -1, 4), ('V', 1, 4), ('V', I, 8)]
check('Q, R, V: 4 ord(h^2)', all(4 * order(hv**2) == d for _, hv, d in data))
check('F(i, h^2 = i/2): 8 ord(-4h^4) = 8', 8 * order(-4 * (I / 2)**2) == 8)
check('I(i, h^2 = 1/2): 8 ord(-4h^4) = 16', 8 * order(-4 * sp.Rational(1, 4)) == 16)

  ok   Q, R, V: 4 ord(h^2)
  ok   F(i, h^2 = i/2): 8 ord(-4h^4) = 8
  ok   I(i, h^2 = 1/2): 8 ord(-4h^4) = 16


True

# 10 The determinant of $\sigma$

Zhang and Zhang attach to a double extension the automorphism $\det\sigma=-p_{11}\sigma_{12}\sigma_{11}+\sigma_{22}\sigma_{11}-p_{12}\sigma_{12}\sigma_{21}$ of $A_x$ \[ZZ08, Section 3\], and for family $\mathbb R$ they prove that $B$ is PI if and only if $\det\sigma$ has finite order \[ZZ08, Corollary 4.8\]. On degree one, $\det\sigma$ is the matrix listed family by family in \[ZZ09, Section 4\]; we reproduce the list in Table 4.

|   Family    |                      $\det\sigma$                      |   Family    |                     $\det\sigma$                     |
|:-----------:|:------------------------------------------------------:|:-----------:|:----------------------------------------------------:|
| $\mathbb A$ |                         $h^2I$                         | $\mathbb N$ |                     $(f^2-g^2)I$                     |
| $\mathbb B$ |            $ph^2\operatorname{diag}(1,-1)$             | $\mathbb O$ |                     $(f-1)h^2I$                      |
| $\mathbb C$ |            $-3h^2\operatorname{diag}(p,1)$             | $\mathbb P$ |                     $(1-f)h^2I$                      |
| $\mathbb D$ |                       $-p^2h^2I$                       | $\mathbb Q$ |            $h^2\operatorname{diag}(-1,1)$            |
| $\mathbb E$ | $2ph^2\left(\begin{smallmatrix}0&1\\-1&0\end{smallmatrix}\right)$ | $\mathbb R$ | $h^2\left(\begin{smallmatrix}0&1\\-1&0\end{smallmatrix}\right)$ |
| $\mathbb F$ |                       $-2ph^2I$                        | $\mathbb S$ |                       $4h^2I$                        |
| $\mathbb G$ |                       $p^2h^2I$                        | $\mathbb T$ | $4h^2\left(\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right)$ |
| $\mathbb H$ |  $h^2\left(\begin{smallmatrix}1&0\\2f&1\end{smallmatrix}\right)$  | $\mathbb U$ |                       $4h^2I$                        |
| $\mathbb I$ |            $2h^2\operatorname{diag}(1,-1)$             | $\mathbb V$ | $h^2\left(\begin{smallmatrix}-1&1\\0&1\end{smallmatrix}\right)$ |
| $\mathbb J$ |                        $2h^2I$                         | $\mathbb W$ |                     $(f+1)h^2I$                      |
| $\mathbb K$ |             $h^2\operatorname{diag}(1,f)$              | $\mathbb X$ | $h^2\left(\begin{smallmatrix}1&0\\2&1\end{smallmatrix}\right)$  |
| $\mathbb L$ |            $h^2\operatorname{diag}(f^2,1)$             | $\mathbb Y$ |                        $h^2I$                        |
| $\mathbb M$ |                      $(1-f)h^2I$                       | $\mathbb Z$ |                     $-(f+1)h^2I$                     |

*Table 4. The automorphism $\det\sigma$ in degree one \[ZZ09, Section 4\].*

**Corollary 10.1.** If a member of the list is PI, then $\det\sigma$ has finite order.

*Proof.* Compare Tables 1 and 4. In every family that has PI members the matrix is a scalar $\gamma$ times a matrix of finite order, or a diagonal matrix, and the scalars involved are roots of unity when the PI criterion holds: $\gamma$ is $\pm h^2$, $\pm p^2h^2$ or $ph^2$ in $\mathbb B$, $\mathbb D$, $\mathbb G$, $\mathbb Q$, $\mathbb R$, $\mathbb V$; it is $\pm2h^2$ or $\pm2ph^2$ in $\mathbb E$, $\mathbb F$, $\mathbb I$, $\mathbb J$; it is $4h^2=(2h)^2$ in $\mathbb S$, $\mathbb T$, $\mathbb U$; the entries are $-3h^2p$, $-3h^2$ in $\mathbb C$ and $h^2$, $h^2f$, $h^2f^2$ in $\mathbb K$, $\mathbb L$; and $$\begin{aligned}
(f-1)h^2&=-h(1-s)\cdot h(1+s), & (f+1)h^2&=h(s+i)\cdot h(s-i), & f^2-g^2&=(f-g)(f+g)
\end{aligned}$$ in $\mathbb M$, $\mathbb O$, $\mathbb P$, in $\mathbb W$, $\mathbb Z$ and in $\mathbb N$. $\square$

**Remark 10.2.** The converse fails, so \[ZZ08, Corollary 4.8\] does not extend to the list. For $h\in\mu_\infty$, $\det\sigma=h^2I$ in $\mathbb Y(f,h)$ and $\det\sigma=-h^2I$ in $\mathbb D(\pm1,h)$, but these algebras are not PI (Proposition 3.2). For a member that is not degenerate, take $\mathbb O$ with $s=3$ and $h=(2\sqrt2)^{-1}$: then $\det\sigma=h^2(f-1)I=I$, while $h(1+s)=\sqrt2\notin\mu_\infty$. The twist is not to blame: in $\mathbb N$, where there is no $h$, the member with $f=2$, $g=\sqrt3$ has $\det\sigma=(f^2-g^2)I=I$ and is not PI, because $(2-\sqrt3)/(2+\sqrt3)$ is not a root of unity. The determinant sees the products in $\mathcal I$ but not the ratios.

In [18]:
# Remark 10.3: the two counterexamples.
sv, hv = 3, 1 / (2 * sp.sqrt(2))
check('O(9, 1/(2 sqrt 2)): det sigma = (f-1)h^2 = 1', zero((sv**2 - 1) * hv**2 - 1))
check('   but h(1+s) = sqrt 2 is not a root of unity', zero(hv * (1 + sv) - sp.sqrt(2)))
check('N(2, sqrt 3): f^2 - g^2 = 1, f + g = 2 + sqrt 3 not of modulus 1', 2**2 - 3 == 1 and abs(float(2 + sp.sqrt(3))) != 1)

  ok   O(9, 1/(2 sqrt 2)): det sigma = (f-1)h^2 = 1
  ok      but h(1+s) = sqrt 2 is not a root of unity
  ok   N(2, sqrt 3): f^2 - g^2 = 1, f + g = 2 + sqrt 3 not of modulus 1


True

# 11 Primitivity

**Proposition 11.1.** Let $B=\bigoplus_{n\ge0}B_n$ be a connected graded PI algebra with $B_1\ne0$. Then $B$ is not primitive. In particular no PI member of the list is primitive.

*Proof.* By Kaplansky's theorem a primitive PI ring is simple artinian \[MR, Theorem 13.3.8\], while $B_{\ge1}$ is a proper nonzero ideal. $\square$

Connected graded algebras can be primitive, so the non-PI members need another argument. The obstruction we use is this.

**Proposition 11.2.** Let $B$ be a member of the list. If the center of $Q(B)$ is not algebraic over $\Bbbk$, then $B$ is not primitive.

*Proof.* $B$ is a noetherian domain, strongly noetherian by \[ZZ09, Theorem 0.1\], and connected graded with finite-dimensional components. So it is a Jacobson algebra satisfying the Nullstellensatz \[ASZ, Theorem 0.4\]. For such algebras a primitive ideal is rational \[BG, Lemma II.7.15\]; in particular, if $(0)$ is primitive, then $Z(Q(B))$ is algebraic over $\Bbbk$. $\square$

## 11.1 The family $\mathbb B$

The relations of $\mathbb B(p,h)$, $p^2=-1$, are $x_2x_1=p\,x_1x_2$, $y_2y_1=p\,y_1y_2$ and $$y_1x_1=h\,x_2y_2,\qquad y_1x_2=h\,x_1y_2,\qquad y_2x_1=-h\,x_2y_1,\qquad y_2x_2=h\,x_1y_1 .$$

**Proposition 11.3.** For every $h\in\Bbbk^\times$ the element $$w=(x_1^4+x_2^4)^2\,(x_1^4x_2^4)^{-1}\in Q(\mathbb B)$$ is central and transcendental over $\Bbbk$. In particular no member of $\mathbb B$ is primitive.

*Proof.* Applying the mixed relations twice, $$\begin{aligned}
y_1x_1^2&=h\,x_2y_2x_1=-h^2x_2^2y_1, & y_2x_1^2&=-h\,x_2y_1x_1=-h^2x_2^2y_2,\\
y_1x_2^2&=h\,x_1y_2x_2=h^2x_1^2y_1, & y_2x_2^2&=h\,x_1y_1x_2=h^2x_1^2y_2,
\end{aligned}$$ so $yx_1^2=-h^2x_2^2y$ and $yx_2^2=h^2x_1^2y$ for $y\in\{y_1,y_2\}$. Applying each rule twice, $$\tag{11.1}
y\,x_1^4=h^4\,x_2^4\,y,\qquad y\,x_2^4=h^4\,x_1^4\,y .$$ Since $p^4=1$, $x_1^4$ and $x_2^4$ are central in $A_x$. By (11.1), $y(x_1^4+x_2^4)^2=h^8(x_1^4+x_2^4)^2y$ and $y\,x_1^4x_2^4=h^8\,x_2^4x_1^4y=h^8\,x_1^4x_2^4y$, so $w$ commutes with $y_1,y_2$, and also with $x_1,x_2$. By the PBW basis $x_1^4$ and $x_2^4$ are algebraically independent, so $w$ is a nonconstant element of $\Bbbk(x_1^4,x_2^4)$, hence transcendental over $\Bbbk$. Proposition 11.2 finishes the proof. $\square$

In [19]:
# Proposition 11.3 (exact, symbolic h, p = i and -i).
for pp in (I, -I):
    A = family('B', p=pp); x1, x2, y1, y2 = gens(A)
    X1, X2 = A.mul(x1, x1), A.mul(x2, x2)
    ok = all(is_zero(add((1, A.mul(y, X1)), (h**2, A.mul(X2, y)))) and is_zero(add((1, A.mul(y, X2)), (-h**2, A.mul(X1, y))))
             for y in (y1, y2))
    check(f'B, p = {pp}: y x1^2 = -h^2 x2^2 y,  y x2^2 = h^2 x1^2 y', ok)

  ok   B, p = I: y x1^2 = -h^2 x2^2 y,  y x2^2 = h^2 x1^2 y
  ok   B, p = -I: y x1^2 = -h^2 x2^2 y,  y x2^2 = h^2 x1^2 y


## 11.2 The families $\mathbb C$, $\mathbb F$, $\mathbb I$, $\mathbb Q$, $\mathbb R$, $\mathbb V$

**Proposition 11.4.** For every value of the twist the following elements are central in $Q(B)$ and transcendental over $\Bbbk$: $$\begin{aligned}
\mathbb F,\ \mathbb R:&\quad (x_1x_2)^2(x_1^2+x_2^2)^{-2}, & \mathbb I:&\quad (y_1y_2)^2(y_1^2+y_2^2)^{-2}, & \mathbb C:&\quad (y_1^2y_2)^3(y_1^3-y_2^3)^{-3},\\
\mathbb Q:&\quad t^2(2-t^2),\ \ t=x_1^{-1}x_2, & \mathbb V:&\quad (t^2-t)^2,\ \ t=x_2^{-1}x_1 . &&
\end{aligned}$$ In particular no member of these six families is primitive.

*Proof.* $\mathbb F$: by Lemma 6.2, $\xi=\omega_1\omega_2^{-1}$ commutes with $y_1,y_2$. Now $\omega_1^2=-x_1^2x_2^2$ and $\omega_2$ are central in $A_x$, so $\xi^2$ is central in $Q(\mathbb F)$. For $\mathbb I$ use the elements $\omega_1,\omega_2$ of the proof of Theorem 6.4, and for $\mathbb R$ the relations $y_j\omega_1=-h^2\omega_1y_j$, $y_j\omega_2=h^2\omega_2y_j$ for $\omega_1=x_1x_2$, $\omega_2=x_1^2+x_2^2$, which follow from Lemma 8.5 as (6.3) followed from Lemma 8.8: conjugation by $Z_1$ and by $Z_2$ multiplies both $\omega_1$ and $\omega_2$ by $h^2$, each $y_j$ lies in $Q(A_x)Z_1+Q(A_x)Z_2$ with coefficients of degree $-1$, and $\omega_1$ anticommutes with the elements of odd degree of $Q(A_x)$.

$\mathbb C$: by (6.4) and $\alpha^3=\beta^3$, the element $a^3b^{-3}$ commutes with $x_1,x_2$; and $a^3=y_1^6y_2^3$ and $b$ are central in $A_y$.

$\mathbb Q$, $\mathbb V$: the proof of Proposition 5.4 shows that $F_{\mathbb Q}(t)$ and $F_{\mathbb V}(t)$ commute with $y_1,y_2$, and (5.1) does not involve $h$. In $\mathbb Q$, $t$ anticommutes with $x_1$ and $F_{\mathbb Q}$ is even; in $\mathbb V$, $t$ commutes with $x_1,x_2$.

Each element is a nonconstant element of a purely transcendental extension of $\Bbbk$, namely $\Bbbk(x_1^2,x_2^2)$, $\Bbbk(y_1^2,y_2^2)$, $\Bbbk(y_1^3,y_2^3)$ or $\Bbbk(t)$. Proposition 11.2 finishes the proof. $\square$

In [20]:
# Proposition 11.4: numerator and denominator of each central fraction are moved past the other
# generators with the same scalar (exact; C through alpha^3 = beta^3).
def same_scalar(A, gs, N1, N2):
    out = True
    for G_ in gs:
        L1, R1 = A.mul(G_, N1), A.mul(N1, G_); c1 = ratio(L1, R1)
        L2, R2 = A.mul(G_, N2), A.mul(N2, G_); c2 = ratio(L2, R2)
        out &= is_zero(add((1, L1), (-c1, R1))) and is_zero(add((1, L2), (-c2, R2))) and zero(c1 - c2)
    return out
for name, par, side in [('F', {'p': I}, 'x'), ('F', {'p': -I}, 'x'), ('R', {}, 'x'), ('I', {'q': I}, 'y'), ('I', {'q': -I}, 'y')]:
    A = family(name, **par); x1, x2, y1, y2 = gens(A)
    a, b = (x1, x2) if side == 'x' else (y1, y2)
    w1 = A.mul(a, b); w2 = add((1, A.mul(a, a)), (1, A.mul(b, b)))
    others = (y1, y2) if side == 'x' else (x1, x2)
    check(f'{name} {par}: omega_1^2 and omega_2^2', same_scalar(A, others, A.mul(w1, w1), A.mul(w2, w2)))
pp = sp.Rational(-1, 2) + sp.sqrt(3) * I / 2
check('C: alpha^3 = beta^3', zero((3 * (1 - pp))**3 - (3 * (2 * pp + 1))**3))
for name in 'KL':
    A = family(name, f=s**2); x1, x2, y1, y2 = gens(A)
    num = add((s**4 if name == 'K' else 1, A.mul(y1, y1, y1, y1)), (1, A.mul(y2, y2, y2, y2)))
    check(f'{name} (Theorem 11.15): numerator and denominator', same_scalar(A, (x1, x2), num, A.mul(y1, y1, y2, y2)))
# Theorem 11.15, family S: r^2 + r^{-2} in terms of x1, x2 (a = x1^2 - x2^2, w = x1 x2 in k_{-1}[x1,x2]).
A = double_ext((-1, 0), (1, 0), [[0] * 4] * 4); x1, x2, _, _ = gens(A)
a_ = add((1, A.mul(x1, x1)), (-1, A.mul(x2, x2))); w_ = A.mul(x1, x2)
lhs = add((1, A.mul(add((1, a_), (-2, w_)), add((1, a_), (-2, w_)))), (1, A.mul(add((1, a_), (2, w_)), add((1, a_), (2, w_)))))
rhs = add((2, A.mul(x1, x1, x1, x1)), (-12, A.mul(x1, x1, x2, x2)), (2, A.mul(x2, x2, x2, x2)))
check('S: (a-2w)^2 + (a+2w)^2 = 2(x1^4 - 6x1^2x2^2 + x2^4)', is_zero(add((1, lhs), (-1, rhs))))

  ok   F {'p': I}: omega_1^2 and omega_2^2


  ok   F {'p': -I}: omega_1^2 and omega_2^2
  ok   R {}: omega_1^2 and omega_2^2


  ok   I {'q': I}: omega_1^2 and omega_2^2


  ok   I {'q': -I}: omega_1^2 and omega_2^2
  ok   C: alpha^3 = beta^3
  ok   K (Theorem 11.15): numerator and denominator
  ok   L (Theorem 11.15): numerator and denominator
  ok   S: (a-2w)^2 + (a+2w)^2 = 2(x1^4 - 6x1^2x2^2 + x2^4)


True

## 11.3 The families $\mathbb D$ and $\mathbb G$

We use the presentations of Section 5, as in \[GV-D, GV-G\]. Let $p^2\ne1$ and $q=hp$; in this section $q$ always denotes $hp$. The relations of $\mathbb D(p,h)$ are $$\begin{aligned}
x_2x_1&=-x_1x_2, & y_2y_1&=p\,y_1y_2, & y_1x_1&=-q\,x_1y_1,\\
y_2x_1&=q\,x_1y_2, & y_1x_2&=h(-p^2x_2y_1+x_1y_2), & y_2x_2&=h(x_2y_2+x_1y_1),
\end{aligned}$$ and those of $\mathbb G(p,f,h)$ are $$\begin{aligned}
x_2x_1&=x_1x_2, & y_ix_1&=q\,x_1y_i, & y_2y_1&=p\,y_1y_2,\\
y_1x_2&=h(p^2x_2y_1+x_1y_2), & y_2x_2&=h(x_2y_2+fx_1y_1). &&
\end{aligned}$$ For all $p,h$ with $p^2\ne1$ (and all $f$), $Q(\mathbb D)$ and $Q(\mathbb G)$ are the quotient division rings of the quantum tori on $x_1,\vartheta,y_1,y_2$ with $$\begin{aligned}
\mathbb D:&\quad \vartheta x_1=-x_1\vartheta,\ \ \vartheta y_1=p^{-1}y_1\vartheta,\ \ \vartheta y_2=py_2\vartheta,\qquad(11.2)\\
&\quad y_1x_1=-q\,x_1y_1,\ \ y_2x_1=q\,x_1y_2,\ \ y_2y_1=p\,y_1y_2,\\
\mathbb G:&\quad \vartheta x_1=x_1\vartheta,\ \ \vartheta y_1=p^{-1}y_1\vartheta,\ \ \vartheta y_2=py_2\vartheta,\ \ y_ix_1=q\,x_1y_i,\ \ y_2y_1=p\,y_1y_2 .\qquad(11.3)
\end{aligned}$$ Here $\vartheta$ is $x_1^{-1}x_2$ minus an explicit Laurent polynomial in $y_1y_2^{-1}$; it is called $T$ in \[GV-D, GV-G\], where this model was found \[GV-D, Theorem 2.7\], \[GV-G, Theorem 3.2\]. A proof is in Lemma 11.8 below.

**Lemma 11.5.** Let $\mathcal T$ be a quantum torus on $v_1^{\pm1},\dots,v_n^{\pm1}$, with $v_iv_j=c_{ij}v_jv_i$, and let $\mathbb L\subseteq\mathbb Z^n$ be the set of exponents of central monomials. Then $Z(Q(\mathcal T))$ is the fraction field of $Z(\mathcal T)=\Bbbk[v^{\mathbf a}:\mathbf a\in\mathbb L]$, and its transcendence degree over $\Bbbk$ is the rank of $\mathbb L$.

*Proof.* For $\mathbf b\in\mathbb Z^n$ let $\chi_{\mathbf b}$ be the character with $v^{\mathbf b}v^{\mathbf a}v^{-\mathbf b}=\chi_{\mathbf b}(\mathbf a)v^{\mathbf a}$. Two monomials $v^{\mathbf a},v^{\mathbf a'}$ have the same characters if and only if $\mathbf a-\mathbf a'\in\mathbb L$. So an element $\phi\in\mathcal T$ with $v^{\mathbf b}\phi v^{-\mathbf b}=\chi_{\mathbf b}(\mathbf a_0)\phi$ for all $\mathbf b$ is of the form $v^{\mathbf a_0}c$ with $c\in Z(\mathcal T)$.

Let $0\ne z\in Z(Q(\mathcal T))$ and $I=\{\phi\in\mathcal T:z\phi\in\mathcal T\}$. Since $z$ is central, $I$ is a two-sided ideal, and it is nonzero because $z=a\phi^{-1}$ for some $a,\phi\in\mathcal T$. It is stable under conjugation by monomials, since $z\,v^{\mathbf b}\phi v^{-\mathbf b}=v^{\mathbf b}(z\phi)v^{-\mathbf b}$. Pick $\phi\in I$ nonzero with the fewest monomials, and $\mathbf a_0$ in its support. Then $v^{\mathbf b}\phi v^{-\mathbf b}-\chi_{\mathbf b}(\mathbf a_0)\phi$ lies in $I$ and has fewer monomials, so it vanishes, and $\phi=v^{\mathbf a_0}c$ with $c\in Z(\mathcal T)$. The element $z\phi\in\mathcal T$ satisfies the same rule, so $z\phi=v^{\mathbf a_0}c'$ with $c'\in Z(\mathcal T)$. Hence $z=v^{\mathbf a_0}c'c^{-1}v^{-\mathbf a_0}=c'c^{-1}$. Finally $Z(\mathcal T)$ is a Laurent polynomial ring in $\operatorname{rk}\mathbb L$ variables, by choosing a basis of $\mathbb L$. $\square$

**Theorem 11.6.** Let $B$ be $\mathbb D(p,h)$ with $p^2\ne1$, or $\mathbb G(p,f,h)$, and $q=hp$.

1.  The transcendence degree of $Z(Q(B))$ over $\Bbbk$ is $4$ if $p,q\in\mu_\infty$, $2$ if exactly one of $p,q$ is in $\mu_\infty$, and $0$ if $p,q\notin\mu_\infty$. In the last case $Z(Q(B))=\Bbbk$.

2.  If $q\in\mu_\infty$, then $x_1^{m}$, $m=\operatorname{lcm}(2,\operatorname{ord}q)$, is central in $\mathbb D(p,h)$, and $x_1^{\operatorname{ord}q}$ is central in $\mathbb G(p,f,h)$.

3.  If $B$ is not PI and $p\in\mu_\infty$ or $q\in\mu_\infty$, then $B$ is not primitive.

*Proof.* (i) Commuting $x_1^a\vartheta^by_1^cy_2^d$ past $x_1,\vartheta,y_1,y_2$ with (11.2) multiplies it by the inverses of $$(-1)^{b+c}q^{-(c+d)},\qquad (-1)^ap^{d-c},\qquad (-q)^ap^{b-d},\qquad q^ap^{c-b},$$ and doing the same for $x_1^a\vartheta^by_1^cy_2^d$ with (11.3) gives $$q^{-(c+d)},\qquad p^{d-c},\qquad q^ap^{b-d},\qquad q^ap^{c-b}.$$ The lattice $\mathbb L$ is cut out by setting these four scalars equal to $1$. If $p,q\in\mu_\infty$, it has finite index and rank four. Suppose $p\notin\mu_\infty$. The second condition gives $d=c$, and in $\mathbb D$ also $a$ even. The last two give $q^ap^{c-b}=q^ap^{b-c}=1$, hence $p^{2(b-c)}=1$, $b=c$ and $q^a=1$. The first then reads $q^{2c}=1$. So $$\mathbb L=\{(a,c,c,c):\ q^a=q^{2c}=1,\ a\text{ even in }\mathbb D\},$$ which has rank two if $q\in\mu_\infty$ and is zero if $q\notin\mu_\infty$. Suppose $p\in\mu_\infty$ and $q\notin\mu_\infty$. The last condition forces $q^a\in\mu_\infty$, so $a=0$; the first forces $c+d=0$, and in $\mathbb D$ also $b+c$ even. What remains is $p^{b+c}=p^{c-b}=1$, so $$\mathbb L=\{(0,b,c,-c):\ p^{b+c}=p^{c-b}=1,\ b+c\text{ even in }\mathbb D\},$$ which contains $(0,2n,0,0)$ and $(0,0,2n,-2n)$, $n=\operatorname{ord}p$, and has rank two. Lemma 11.5 gives (i).

\(ii\) In $\mathbb D$, $m$ is even and $q^m=1$, so $x_1^m$ commutes with $x_2$, $y_1$ and $y_2$ by the first, third and fourth relations. In $\mathbb G$, $x_1$ commutes with $x_2$ and $y_ix_1^{\operatorname{ord}q}=q^{\operatorname{ord}q}x_1^{\operatorname{ord}q}y_i$.

\(iii\) If $B$ is not PI, then $p$ and $q$ are not both roots of unity, so by (i) $Z(Q(B))$ has transcendence degree two. Proposition 11.2 applies. $\square$

For $p\in\mu_\infty$, part (iii) can also be seen with Dickson polynomials: with $t=x_1^{-1}x_2$ and $d=\operatorname{ord}(p^2)$, the element $D_d(t,\kappa^2)$, $\kappa=(p^2-1)^{-1}$, is normal in $Q(\mathbb D)$ and its square is central \[GV-D\]; the computation does not involve $h$, and $\mathbb G$ is similar \[GV-G\]. When $q\in\mu_\infty$ and $p\notin\mu_\infty$, part (ii) gives an algebra that is neither PI nor primitive and still has central elements of positive degree.

In [21]:
# Theorem 11.6 (i): rank of the lattice of central monomials of the quantum torus (11.1),
# with p, q of the given orders ("inf" = not a root of unity), taken multiplicatively independent.
def rank_lattice(fam, op, oq, box=8):
    def ok(v):
        a, b, c, d = v
        if fam == 'D':   # (sign exponent, p exponent, q exponent) of the four scalars
            conds = [(b + c, 0, -(c + d)), (a, d - c, 0), (a, b - d, a), (0, c - b, a)]
        else:
            conds = [(0, 0, -(c + d)), (0, d - c, 0), (0, b - d, a), (0, c - b, a)]
        return all(sg % 2 == 0 and (ep % op == 0 if op else ep == 0) and (eq % oq == 0 if oq else eq == 0)
                   for sg, ep, eq in conds)
    pts = [v for v in itertools.product(range(-box, box + 1), repeat=4) if ok(v)]
    return sp.Matrix(pts).rank() if pts else 0
for fam in 'DG':
    for op, oq, expected in [(3, 4, 4), (0, 4, 2), (3, 0, 2), (0, 0, 0)]:
        check(f'{fam}: ord p = {op or "inf"}, ord q = {oq or "inf"}  ->  transcendence degree {expected}',
              rank_lattice(fam, op, oq) == expected)

  ok   D: ord p = 3, ord q = 4  ->  transcendence degree 4
  ok   D: ord p = inf, ord q = 4  ->  transcendence degree 2
  ok   D: ord p = 3, ord q = inf  ->  transcendence degree 2
  ok   D: ord p = inf, ord q = inf  ->  transcendence degree 0


  ok   G: ord p = 3, ord q = 4  ->  transcendence degree 4


  ok   G: ord p = inf, ord q = 4  ->  transcendence degree 2
  ok   G: ord p = 3, ord q = inf  ->  transcendence degree 2
  ok   G: ord p = inf, ord q = inf  ->  transcendence degree 0


## 11.4 Primitivity of $\mathbb D$ and $\mathbb G$ when $p,hp\notin\mu_\infty$

Now let $p,q\notin\mu_\infty$. Then $Z(Q(B))=\Bbbk$ and Proposition 11.2 gives nothing. We show that $B$ is primitive, treating $\mathbb D$ and $\mathbb G$ together through the part of $B$ over $\Bbbk[x_1^{-1}x_2]$.

Let $\beta_1,\beta_2\in\Bbbk^\times$ and $p^2\ne1$, and let $A$ be the algebra generated by $t,y_1,y_2$ with $$\tag{11.4}
y_2y_1=p\,y_1y_2,\qquad ty_1=p^{-1}y_1t+\beta_1y_2,\qquad ty_2=py_2t+\beta_2y_1 .$$ Put $$b=-\frac{\beta_2}{p^2-1},\qquad c=\frac{p^2\beta_1}{p^2-1},\qquad a=bc,\qquad \lambda(w)=w+aw^{-1},$$ and $$\Omega=ty_1y_2-p^{-1}b\,y_1^2-c\,y_2^2 .$$ The constants satisfy $$\tag{11.5}
a(p-p^{-1})^2=-\beta_1\beta_2,\qquad a(p^2-1)=p^2\beta_1b .$$ In both families let $t=x_1^{-1}x_2$. For $\mathbb D$ take $\beta_1=p^{-2}$, $\beta_2=-1$; then $b=c=\kappa$ and $\Omega=x_1^{-1}\Xi$ with $\Xi$ as in Remark 11.14 below. For $\mathbb G$ take $\beta_1=-p^{-2}$, $\beta_2=-f$; then $b,c$ are the constants of \[GV-G\] and $\Omega=x_1^{-1}\Xi$ with $\Xi=x_2y_1y_2-x_1(cy_2^2+p^{-1}by_1^2)$. In both cases (11.4) are the relations of \[GV-D, GV-G\] between $t$ and the $y$'s, and they do not involve $h$. By the PBW bases of $\mathbb D$ and $\mathbb G$, $A$ is the subalgebra of $B[x_1^{-1}]$ generated by $t,y_1,y_2$, it has basis $t^iy_1^jy_2^l$, and $$\tag{11.6}
B[x_1^{-1}]=A[x_1^{\pm1};\alpha],\qquad \alpha=x_1(-)x_1^{-1},$$ We grade $A$ by $\deg t=0$, $\deg y_i=1$; the relations are homogeneous, and $A_n$ is a free left $\Bbbk[t]$-module on the $y_1^jy_2^l$, $j+l=n$.

**Lemma 11.7.** $\Omega$ is central in $A$. In $A$ we also have $y_1t=p\,ty_1-p\beta_1y_2$ and $y_2t=p^{-1}ty_2-p^{-1}\beta_2y_1$.

*Proof.* The last two relations are (11.4) solved for $y_it$. In $Q(A)$ put $\varpi=y_1y_2^{-1}$ and $\vartheta=t-b\varpi-c\varpi^{-1}$. From $y_2^{-1}y_1=p^{-1}y_1y_2^{-1}$ one gets $\varpi y_1=p^{-1}y_1\varpi$, $\varpi^{-1}y_1=y_2$, $y_1\varpi^{-1}=p^{-1}y_2$, $\varpi y_2=y_1$, $y_2\varpi=py_1$, hence $$\begin{aligned}
(b\varpi+c\varpi^{-1})y_1-p^{-1}y_1(b\varpi+c\varpi^{-1})&=c(1-p^{-2})y_2=\beta_1y_2,\\
(b\varpi+c\varpi^{-1})y_2-py_2(b\varpi+c\varpi^{-1})&=b(1-p^2)y_1=\beta_2y_1,
\end{aligned}$$ and subtracting from (11.4) gives $\vartheta y_1=p^{-1}y_1\vartheta$ and $\vartheta y_2=py_2\vartheta$. Since $\varpi y_1y_2=p^{-1}y_1^2$ and $\varpi^{-1}y_1y_2=y_2^2$, we have $\Omega=\vartheta y_1y_2$. Now $(y_1y_2)y_1=p\,y_1(y_1y_2)$ and $(y_1y_2)y_2=p^{-1}y_2(y_1y_2)$, so $\vartheta y_1y_2$ commutes with $y_1$ and $y_2$; it commutes with $\vartheta$, and with $\varpi$ because $(y_1y_2)\varpi=p^2\varpi(y_1y_2)$ and $\vartheta\varpi=p^{-2}\varpi\vartheta$. Hence it commutes with $t=\vartheta+b\varpi+c\varpi^{-1}$. $\square$

**Lemma 11.8.** Let $B$ be $\mathbb D(p,h)$ with $p^2\ne1$, or $\mathbb G(p,f,h)$, and $\vartheta=t-b\varpi-c\varpi^{-1}\in Q(B)$ as in the proof of Lemma 11.7. Then $x_1,\vartheta,y_1,y_2$ satisfy (11.2), respectively (11.3), their monomials are linearly independent, and they generate $Q(B)$ as a division ring.

*Proof.* The relations of $\vartheta$ with $y_1,y_2$ are in the proof of Lemma 11.7. In $\mathbb D$, $$\begin{aligned}
x_1tx_1^{-1}&=-t, & x_1y_1x_1^{-1}&=-q^{-1}y_1, & x_1y_2x_1^{-1}&=q^{-1}y_2,
\end{aligned}$$ so $x_1\varpi x_1^{-1}=-\varpi$ and $x_1\vartheta x_1^{-1}=-\vartheta$. In $\mathbb G$, $x_1$ commutes with $t$ and $x_1y_ix_1^{-1}=q^{-1}y_i$, so $x_1$ commutes with $\varpi$ and with $\vartheta$. Since $t=\vartheta+b\varpi+c\varpi^{-1}$ and $x_2=x_1t$, the four elements generate $Q(B)$.

For the independence, note that $Q(A_y)\,t\subseteq t\,Q(A_y)+Q(A_y)$ by Lemma 11.7, so $\vartheta^k\in t^k+\sum_{i<k}t^iQ(A_y)$. The powers of $t$ are right independent over $Q(A_y)$, because the $t^iy_1^jy_2^l$ are a basis of $A$. Hence the elements $\vartheta^ky_1^jy_2^l$, $k\ge0$, $j,l\in\mathbb Z$, are independent, and by (11.6) so are the $x_1^a\vartheta^ky_1^jy_2^l$. Negative powers of $\vartheta$ are handled by multiplying a relation by a power of $\vartheta$, which commutes with the other three generators up to scalars. $\square$

From now on $p\notin\mu_\infty$.

**Lemma 11.9.** Let $M$ be a left $A$-module, $w\in\Bbbk^\times$ and $0\ne v\in M$ with $tv=\lambda(w)v$. Then $$\bigl(t-\lambda(pw)\bigr)\bigl(t-\lambda(w/p)\bigr)y_iv=0\qquad(i=1,2).$$ If $t-\lambda(pw)$ is injective on $M$, then $\Omega v=0$.

*Proof.* Write $\lambda=\lambda(w)$. By (11.4), $$t\,y_1v=p^{-1}\lambda\,y_1v+\beta_1\,y_2v,\qquad t\,y_2v=p\lambda\,y_2v+\beta_2\,y_1v .$$ The $2\times2$ matrix of this rule has trace $(p+p^{-1})\lambda$ and determinant $\lambda^2-\beta_1\beta_2$, and by (11.5) $$\begin{aligned}
\lambda(pw)+\lambda(w/p)&=(p+p^{-1})\lambda(w),\\
\lambda(pw)\lambda(w/p)&=w^2+a(p^2+p^{-2})+a^2w^{-2}=\lambda(w)^2-\beta_1\beta_2 .
\end{aligned}$$ Cayley--Hamilton gives the first claim. If $t-\lambda(pw)$ is injective, then $t\,y_iv=\lambda(w/p)\,y_iv$ for $i=1,2$, and the first displayed rule gives $y_2v=\gamma\,y_1v$ with $$\gamma=\frac{\lambda(w/p)-p^{-1}\lambda(w)}{\beta_1}=\frac{a(p^2-1)}{p\beta_1w}=\frac{pb}{w}.$$ Then $y_1y_2v=\gamma y_1^2v$, $y_2^2v=\gamma\,y_2y_1v=p\gamma^2y_1^2v$, and $$t\,y_1^2v=p^{-1}y_1\,t\,y_1v+\beta_1y_2y_1v=\bigl(p^{-1}\lambda(w/p)+p\beta_1\gamma\bigr)y_1^2v .$$ Since $p^2\beta_1-c\,p^2=-c$, we get $$\begin{aligned}
\Omega v&=\bigl(\gamma p^{-1}\lambda(w/p)+p\beta_1\gamma^2-p^{-1}b-cp\gamma^2\bigr)y_1^2v\\
&=p^{-1}\bigl(\gamma\lambda(w/p)-c\gamma^2-b\bigr)y_1^2v\\
&=p^{-1}\Bigl(b+\frac{ap^2b}{w^2}-\frac{cp^2b^2}{w^2}-b\Bigr)y_1^2v=0,
\end{aligned}$$ because $a=bc$. $\square$

**Corollary 11.10.** Let $\Bbbk$ be algebraically closed and $M\ne0$ a left $A$-module with $g(t)M=0$ for some nonzero $g\in\Bbbk[t]$. Then $\Omega v=0$ for some $0\ne v\in M$. Consequently, if $Q$ is a prime ideal of $A$ containing a nonzero element of $\Bbbk[t]$, then $\Omega\in Q$.

*Proof.* Let $\Lambda$ be the set of $\mu\in\Bbbk$ with $\ker(t-\mu)\ne0$ on $M$. It is a nonempty set of roots of $g$, and $t-\mu$ is injective for $\mu\notin\Lambda$. The set $W=\{w\in\Bbbk^\times:\lambda(w)\in\Lambda\}$ is finite and nonempty, and since $p\notin\mu_\infty$ there is $w_0\in W$ with $pw_0\notin W$. Take $v\ne0$ with $tv=\lambda(w_0)v$ and apply Lemma 11.9. For the second claim apply this to $M=A/Q$: there is $r\notin Q$ with $\Omega r\in Q$, so $(A\Omega A)(ArA)=A\Omega rA\subseteq Q$ and $\Omega\in Q$. $\square$

Next, the weights of the right action of $t$. Let $F=\Bbbk(t)$ and $F(w)$ the extension by a root of $w^2-tw+a$, so that $t=\lambda(w)$; since $a\ne0$ this polynomial is irreducible, and $w$ is transcendental over $\Bbbk$. Put $V=F(w)\otimes_{\Bbbk[t]}A$, using the left $\Bbbk[t]$-structure of $A$. Right multiplication by any element of $A$ is left $\Bbbk[t]$-linear, so it acts $F(w)$-linearly on $V=\bigoplus_nV_n$, and it is injective because $A$ is a domain and $F(w)$ is flat over $\Bbbk[t]$. Also $\dim_{F(w)}V_n=n+1$. The scalars $\lambda(p^{\nu}w)$, $\nu\in\mathbb Z$, are pairwise distinct: $\lambda(p^{\nu}w)=\lambda(p^{\nu'}w)$ forces $\nu=\nu'$ or $w^2\in\Bbbk$.

**Lemma 11.11.** Let $\eta\in F(w)^\times$ with $\lambda(p\eta)\ne\lambda(\eta/p)$, and $0\ne m\in V_n$ with $mt=\lambda(\eta)m$. Then $my_1,my_2\in V_{n+1}$ lie in the sum of the $\lambda(p\eta)$- and $\lambda(\eta/p)$-eigenspaces of right multiplication by $t$. The $\lambda(\eta/p)$-components of $my_1$ and $my_2$ are not both zero, and neither are their $\lambda(p\eta)$-components.

*Proof.* By Lemma 11.7, with $\mu=\lambda(\eta)$, $$(my_1)t=p\mu\,my_1-p\beta_1\,my_2,\qquad (my_2)t=p^{-1}\mu\,my_2-p^{-1}\beta_2\,my_1 ,$$ a matrix with trace $(p+p^{-1})\mu$ and determinant $\mu^2-\beta_1\beta_2$, hence with the distinct eigenvalues $\lambda(p\eta)$ and $\lambda(\eta/p)$, as in Lemma 11.9. Suppose the $\lambda(\eta/p)$-components of $my_1$ and $my_2$ vanish. Then the first equation gives $my_2=\gamma'my_1$ with $$\gamma'=\frac{p\lambda(\eta)-\lambda(p\eta)}{p\beta_1}=\frac{a(p^2-1)}{p^2\beta_1\eta}=\frac b\eta ,$$ and as before, using $y_1y_2=p^{-1}y_2y_1$, $$m\Omega=p^{-1}\bigl(\lambda(\eta)\gamma'-b-c\gamma'^2\bigr)my_1^2=p^{-1}\,\frac{b(a-bc)}{\eta^2}\,my_1^2=0,$$ which is impossible because right multiplication by $\Omega$ is injective on $V$. For the $\lambda(p\eta)$-components apply the same argument to $\eta'=a/\eta$, noting that $\lambda(\eta')=\lambda(\eta)$ and $\lambda(\eta'/p)=\lambda(p\eta)$. $\square$

**Proposition 11.12.**

1.  Right multiplication by $t$ on $V_n$ is diagonalizable, with the $n+1$ distinct eigenvalues $\lambda(p^{\nu}w)$, $\nu\in\{-n,-n+2,\dots,n\}$. For $n=2k$ the $\lambda(w)$-eigenspace is $F(w)\,\Omega^k$.

2.  Every nonzero graded ideal of $A$ contains a power of $\Omega$.

*Proof.* (i) Induction on $n$, with $V_0=F(w)$ and $1\cdot t=\lambda(w)$. Since $A_n=A_{n-1}y_1+A_{n-1}y_2$, Lemma 11.11, applied to $\eta=p^{\nu}w$, shows that $V_n$ is spanned by eigenvectors with eigenvalues $\lambda(p^{\nu}w)$, $|\nu|\le n$, $\nu\equiv n\pmod2$, and that each of these $n+1$ values occurs. As $\dim V_n=n+1$, each eigenspace is a line. Finally $\Omega^k\in A_{2k}$ commutes with $t$, so $\Omega^kt=\lambda(w)\Omega^k$.

\(ii\) Let $J\ne0$ be a graded ideal and $J_n\ne0$. Then $U_n=F(w)\otimes J_n\subseteq V_n$ is nonzero and stable under right multiplication by $t$, so it contains an eigenvector $m$, say with eigenvalue $\lambda(p^{\nu}w)$. Suppose $\nu>0$. By Lemma 11.11, one of $my_1,my_2\in U_{n+1}$ has a nonzero $\lambda(p^{\nu-1}w)$-component, and this component lies in $U_{n+1}$ because the eigenprojections are polynomials in right multiplication by $t$. After $|\nu|$ steps (lowering if $\nu>0$, raising if $\nu<0$) we find $U_N\ni\Omega^k$ with $N=2k$, by (i). So the class of $\Omega^k$ in $A_N/J_N$ dies in $F(w)\otimes(A_N/J_N)$, hence is $\Bbbk[t]$-torsion: $g(t)\Omega^k\in J$ for some nonzero $g$.

Now let $Q_1,\dots,Q_\ell$ be prime ideals containing $J$ with $Q_1\cdots Q_\ell\subseteq J$. Each $Q_i$ contains $g(t)\Omega^k$, hence $(Ag(t)A)(A\Omega^kA)$, so it contains $g(t)$ or $\Omega^k$; in both cases $\Omega\in Q_i$, by Corollary 11.10 in the first. Therefore $\Omega^\ell\in J$. $\square$

**Theorem 11.13.** Let $B$ be $\mathbb D(p,h)$ with $p^2\ne1$, or $\mathbb G(p,f,h)$, and suppose that $B$ is not PI. Then $B$ is primitive if and only if neither $p$ nor $hp$ is a root of unity, that is, if and only if $Z(Q(B))=\Bbbk$.

*Proof.* One direction is Theorem 11.6(iii). Let $p,q\notin\mu_\infty$, $q=hp$. The elements $x_1$ and $\Xi=x_1\Omega$ are normal in $B$ (Remark 11.14). We show that every nonzero prime $P$ of $B$ contains $x_1\Xi$; then $(0)$ is locally closed, and $B$ is primitive by \[BG, Lemma II.7.15\] and \[ASZ, Theorem 0.4\], as in the proof of Theorem 11.16 below.

Let $x_1\notin P$. Then $P'=P\,B[x_1^{-1}]$ is a nonzero ideal of $A[x_1^{\pm1};\alpha]$ with $P'\cap B=P$. Conjugation by $x_1$ multiplies $\Omega$ by $q^{-2}$, so $\Omega x_1^i=q^{2i}x_1^i\Omega$. Take $0\ne g=\sum_{i\in I}\theta_ix_1^i\in P'$, $\theta_i\in A$, with $|I|$ minimal, and $j\in I$. Then $$\Omega g-q^{2j}g\Omega=\sum_{i\in I}(q^{2i}-q^{2j})\,\theta_ix_1^i\,\Omega\in P'$$ has fewer terms, so it vanishes, and since $q\notin\mu_\infty$ and $A$ is a domain, $g=\theta_jx_1^j$. Hence $J=P'\cap A\ne0$. It is stable under $\alpha$, which acts on the basis by $$\alpha(t^iy_1^jy_2^l)=\varepsilon\,q^{-(j+l)}\,t^iy_1^jy_2^l,\qquad \varepsilon\in\{\pm1\},$$ ($\varepsilon=(-1)^{i+j}$ for $\mathbb D$, $\varepsilon=1$ for $\mathbb G$). Since $q\notin\mu_\infty$, elements of different degree have different eigenvalues, and a Vandermonde argument shows that $J$ is graded. By Proposition 11.12(ii), $\Omega^\ell\in J$ for some $\ell$. Then $\Xi^\ell$, a nonzero multiple of $x_1^\ell\Omega^\ell$, lies in $P'\cap B=P$, and since $\Xi$ is normal and $P$ is prime, $\Xi\in P$. $\square$

**Remark 11.14.** In $\mathbb D$ the element $$\Xi=x_2y_1y_2-\kappa x_1\bigl(p^{-1}y_1^2+y_2^2\bigr)=x_1\Omega$$ is normal: $\Xi x_1=q^2x_1\Xi$, $\Xi y_1=-q^{-1}y_1\Xi$, $\Xi y_2=q^{-1}y_2\Xi$ and $\Xi x_2=-q^2x_2\Xi$, because $\Omega$ is central in $A$ and $x_1$ normalizes $A$. In $\mathbb G$, $\Xi=x_1\Omega$ satisfies $\Xi x_i=q^2x_i\Xi$ and $\Xi y_i=q^{-1}y_i\Xi$ \[GV-G\]. For $h=1$ these are the elements called $C$ in \[GV-D, GV-G\]. Unlike in the monomial families, the normal elements only see the $x$-side: no monomial $y_1^ay_2^b\ne1$ is normal in $\mathbb D$, since conjugation by it multiplies $\varpi$ by $p^{-(a+b)}$ and $\varpi^{-1}$ by $p^{a+b}$. Proposition 11.12(ii) plays the role of a second normal element.

In [22]:
# Lemma 11.9 (Omega central in A) and Remark 11.12 (Xi normal in D and G), exactly.
b1, b2 = sp.symbols('beta1 beta2', nonzero=True)
A = Alg(['t', 'y1', 'y2'], {(1, 0): [(p, (0, 1)), (-p * b1, (2,))],
                            (2, 0): [(1 / p, (0, 2)), (-b2 / p, (1,))],
                            (2, 1): [(p, (1, 2))]})
t_, y1_, y2_ = A.gen(0), A.gen(1), A.gen(2)
bb, cc = -b2 / (p**2 - 1), p**2 * b1 / (p**2 - 1)
Om = add((1, A.mul(t_, y1_, y2_)), (-bb / p, A.mul(y1_, y1_)), (-cc, A.mul(y2_, y2_)))
check('Omega central in A', all(skew(A, G_, Om, 1) for G_ in (t_, y1_, y2_)))
q_ = h * p
A = family('D'); x1, x2, y1, y2 = gens(A); k_ = 1 / (p**2 - 1)
Xi = add((1, A.mul(x2, y1, y2)), (-k_ / p, A.mul(x1, y1, y1)), (-k_, A.mul(x1, y2, y2)))
check('D: Xi x1 = q^2 x1 Xi, Xi x2 = -q^2 x2 Xi, Xi y1 = -q^{-1} y1 Xi, Xi y2 = q^{-1} y2 Xi',
      skew(A, Xi, x1, q_**2) and skew(A, Xi, x2, -q_**2) and skew(A, Xi, y1, -1 / q_) and skew(A, Xi, y2, 1 / q_))
A = family('G'); x1, x2, y1, y2 = gens(A)
bb, cc = f / (p**2 - 1), -1 / (p**2 - 1)
Xi = add((1, A.mul(x2, y1, y2)), (-cc, A.mul(x1, y2, y2)), (-bb / p, A.mul(x1, y1, y1)))
check('G: Xi x_i = q^2 x_i Xi, Xi y_i = q^{-1} y_i Xi',
      all(skew(A, Xi, x, q_**2) for x in (x1, x2)) and all(skew(A, Xi, y, 1 / q_) for y in (y1, y2)))
# Lemmas 11.13 and 11.15: the scalar identities.
w = sp.Symbol('w', nonzero=True); a_ = bb * cc
lam = lambda u: u + a_ / u
b1v, b2v = -1 / p**2, -f
check('lambda(pw) + lambda(w/p) = (p + 1/p) lambda(w)', zero(lam(p * w) + lam(w / p) - (p + 1 / p) * lam(w)))
check('lambda(pw) lambda(w/p) = lambda(w)^2 - beta1 beta2', zero(lam(p * w) * lam(w / p) - lam(w)**2 + b1v * b2v))
gam = (lam(w / p) - lam(w) / p) / b1v
check('gamma = p b / w', zero(gam - p * bb / w))
check('gamma lambda(w/p) - c gamma^2 - b = 0', zero(gam * lam(w / p) - cc * gam**2 - bb))

  ok   Omega central in A


  ok   D: Xi x1 = q^2 x1 Xi, Xi x2 = -q^2 x2 Xi, Xi y1 = -q^{-1} y1 Xi, Xi y2 = q^{-1} y2 Xi


  ok   G: Xi x_i = q^2 x_i Xi, Xi y_i = q^{-1} y_i Xi
  ok   lambda(pw) + lambda(w/p) = (p + 1/p) lambda(w)


  ok   lambda(pw) lambda(w/p) = lambda(w)^2 - beta1 beta2
  ok   gamma = p b / w


  ok   gamma lambda(w/p) - c gamma^2 - b = 0


True

## 11.5 The monomial families

For the families of Section 7 the crossed product settles primitivity for all parameters: Proposition 7.1(i)--(ii), the relations of Section 7 and the splitting of $T'$ in the proof of Theorem 7.2 do not need the PI hypothesis. The two factors have parameters $$\tag{11.7}
\begin{array}{c|c|c}
 & \text{first factor} & \text{second factor}\\\hline
\mathbb S,\ \mathbb T,\ \mathbb U & -1 & 16h^4\\
\mathbb E,\ \mathbb J & -1 & 4h^4\\
\mathbb M,\ \mathbb N,\ \mathbb O,\ \mathbb P,\ \mathbb W,\ \mathbb Z & u^{\pm2} & v^2
\end{array}$$ with $u,v$ as in Theorem 7.2.

**Lemma 11.15.** Let $N_x\in A_x$ and $N_y\in A_y$ be the elements $$\begin{array}{c|cccccc}
 & \mathbb M,\ \mathbb O & \mathbb P & \mathbb Z & \mathbb W & \mathbb N\\\hline
N_x & e_1^2 & e_1^2 & e_1^2 & e_1e_2 & x_1x_2\\
N_y & z_1^2 & y_1y_2 & z_1z_2 & z_1^2 & z_1^2
\end{array}$$ They are normal in $B$, and the localization of $B$ at the multiplicative set they generate is the ring $$\Lambda=T'\oplus T'e\oplus T'z\oplus T'ez\subseteq Q(B).$$

*Proof.* In each case $N_x$ is normal in $A_x$: $e_1^2$ is central for a balanced basis, and $e_1e_2$, $x_1x_2$ are normal in a commutative or coordinate plane. By (4.1), $z_aN_x$ is a scalar multiple of $N_x'z_a$ with $N_x'\in\{e_1^2,e_2^2\}$ or $\{e_1e_2,e_2e_1\}$, and $N_x'$ is a scalar multiple of $N_x$ in the cases listed. So $N_x$ is normal in $B$, and the same argument applies to $N_y$. In $\Lambda$ both are units, so $B[N_x^{-1},N_y^{-1}]\subseteq\Lambda$. Conversely, inverting $N_x$ inverts $e_1$ and $e_2$ (for instance $e_1^{-1}=e_1N_x^{-1}$ when $N_x=e_1^2$ and the basis is balanced, and $e_1^{-1}=e_2N_x^{-1}$ when $N_x=e_1e_2$), so it contains $r^{\pm1}$ and $X^{\pm1}$; the same holds on the $y$-side, and $\Lambda\subseteq B[N_x^{-1},N_y^{-1}]$. $\square$

**Theorem 11.16.**

1.  The members of $\mathbb E$, $\mathbb J$, $\mathbb S$, $\mathbb T$, $\mathbb U$, $\mathbb K$ and $\mathbb L$ are never primitive.

2.  Let $B$ be a member of $\mathbb M$, $\mathbb N$, $\mathbb O$, $\mathbb P$, $\mathbb W$ or $\mathbb Z$, with $f\ne0$ in $\mathbb M,\mathbb O,\mathbb P,\mathbb W$, and let $u,v$ be as in Theorem 7.2. Then $B$ is primitive if and only if neither $u$ nor $v$ is a root of unity.

*Proof.* Suppose one of the two factors in (11.7) has a parameter in $\mu_\infty$; this always happens in the first two rows. Then $T'$ has a central monomial $c$ other than a scalar, for instance $r^2$ when the first factor has parameter $-1$. Put $$\prod_{g\in G}\bigl(t-g(c)\bigr)\in K[t].$$ The coefficients lie in $K^G$, which is central in $Q(B)$, and they are not all in $\Bbbk$, because $c$ is transcendental over $\Bbbk$ and is a root of this polynomial. So the center of $Q(B)$ is not algebraic over $\Bbbk$, and $B$ is not primitive by Proposition 11.2. For $\mathbb S$, for instance, $r^2+r^{-2}$ is already central in $Q(B)$; in terms of $x_1,x_2$, $$r^2+r^{-2}=\frac{2\bigl(x_1^4-6x_1^2x_2^2+x_2^4\bigr)}{(x_1^2+x_2^2)^2},$$ for every $h$. For $\mathbb K$ and $\mathbb L$ the elements $$\frac{f^2y_1^4+y_2^4}{y_1^2y_2^2}\in Q(\mathbb K),\qquad \frac{y_1^4+y_2^4}{y_1^2y_2^2}\in Q(\mathbb L)$$ are central for all $q,f,h$. Indeed, the mixed relations of $\mathbb K$ are $y_1x_1=hx_1y_1$, $y_2x_1=hx_1y_2$, $y_1x_2=hx_2y_2$, $y_2x_2=hfx_2y_1$, and those of $\mathbb L$ are in Section 5. Applying them twice, $$\begin{aligned}
\mathbb K:&\quad y_j^2x_1=h^2x_1y_j^2, & y_1^2x_2&=h^2x_2y_2^2, & y_2^2x_2&=h^2f^2x_2y_1^2,\\
\mathbb L:&\quad y_1^2x_1=h^2f^2x_1y_2^2, & y_2^2x_1&=h^2f^2x_1y_1^2, & y_1^2x_2&=h^2x_2y_2^2,\quad y_2^2x_2=h^2x_2y_1^2 .
\end{aligned}$$ So in $\mathbb K$ both $f^2y_1^4+y_2^4$ and $y_1^2y_2^2$ are multiplied by $h^4$ when moved past $x_1$ and by $h^4f^2$ past $x_2$, and in $\mathbb L$ both $y_1^4+y_2^4$ and $y_1^2y_2^2$ are multiplied by $h^4f^4$ past $x_1$ and by $h^4$ past $x_2$. Moreover $y_1^2,y_2^2$ are central in $\Bbbk_{-1}[y_1,y_2]$. This proves (i) and one direction of (ii).

Now let $u,v\notin\mu_\infty$ in (ii). The quantum torus $T'$ is a tensor product of two rank-two tori with parameters that are not roots of unity, so its center is $\Bbbk$, and $T'$ is simple \[MP, Proposition 1.3\]. Its units are the nonzero scalar multiples of monomials, so an inner automorphism of $T'$ multiplies each monomial by a scalar. Here $e$, $z$ and $ez$ act on $T'$ by conjugation, and $\Lambda$ is a crossed product of $T'$ by the group $(\mathbb Z/2)^2$ of these actions modulo inner automorphisms (recall $e^2=X$, $z^2=Y$ and $ze\in T'^{\times}ez$). We claim that the automorphisms of $T'$ induced by $e$, $z$ and $ez$ are outer. The elements that invert $r$ or $\rho$ are not of that form. The remaining nontrivial element is $ez$ in $\mathbb M$ and $\mathbb O$, $z$ in $\mathbb P$ and $\mathbb Z$, and $e$ in $\mathbb N$ and $\mathbb W$, and it multiplies $r$ or $\rho$ by one of $u^{\pm1}$, $-u$. Conjugation by a monomial of $T'$ multiplies $r$ and $\rho$ by elements of $\langle u^2\rangle$, by the relations of Section 7. If $\pm u^{\pm1}\in\langle u^2\rangle$ then $u^{2j\mp1}=\pm1$ for some $j$, and $u\in\mu_\infty$. This proves the claim.

Now $\Lambda$ is simple. Let $I\ne0$ be an ideal and $x=\sum_ga_gw_g\in I$ a nonzero element with the fewest nonzero terms, where $w_g\in\{1,e,z,ez\}$ and $a_g\in T'$. Multiplying by some $w_g^{-1}$ we may assume $a_1\ne0$. The coefficients $a_1$ of the elements of $I$ with support contained in that of $x$ form a nonzero ideal of $T'$, so we may take $a_1=1$. For $t\in T'$ the element $tx-xt=\sum_g\bigl(ta_g-a_g(w_g\triangleright t)\bigr)w_g$ has fewer terms, so $ta_g=a_g(w_g\triangleright t)$ for all $t$. If $a_g\ne0$ for some $g\ne1$, then $a_gT'=T'a_g$ is a nonzero ideal, so $a_g$ is a unit and $w_g\triangleright$ is conjugation by $a_g^{-1}$, against the claim. Hence $x=1$.

Let $P\ne0$ be a prime ideal of $B$. If $P$ missed the multiplicative set generated by $N_x$ and $N_y$, then $P\Lambda$ would be a proper nonzero ideal of $\Lambda$ by Lemma 11.15. So $P$ contains a product of copies of $N_x$ and $N_y$, and since these are normal and $P$ is prime, $P$ contains $N_x$ or $N_y$, hence $N_xN_y$. Therefore the intersection of the nonzero primes of $B$ contains $N_xN_y\ne0$, and $(0)$ is locally closed. For a noetherian algebra satisfying the Nullstellensatz, locally closed primes are primitive \[BG, Lemma II.7.15\], and $B$ satisfies the Nullstellensatz \[ASZ, Theorem 0.4\]. $\square$

**Remark.** In (ii), $u,v\in\mu_\infty$ gives a PI algebra that is not primitive, $u,v\notin\mu_\infty$ a primitive algebra that is not PI, and if exactly one of $u,v$ is a root of unity $B$ is neither. In the families of (i), and in $\mathbb B$, the non-PI members are neither PI nor primitive.

Table 5 sums up primitivity for the non-PI members.

| Family                                                                                                 | non-PI member is primitive iff | Reference             |
|:-------------------------------------------------------------------------------------------------------|:-------------------------------|:----------------------|
| $\mathbb B$, $\mathbb E$, $\mathbb J$, $\mathbb K$, $\mathbb L$, $\mathbb S$, $\mathbb T$, $\mathbb U$ | never                          | Prop. 11.3,Thm. 11.16 |
| $\mathbb C$, $\mathbb F$, $\mathbb I$, $\mathbb Q$, $\mathbb R$, $\mathbb V$                           | never                          | Prop. 11.4            |
| $\mathbb M$, $\mathbb N$, $\mathbb O$, $\mathbb P$, $\mathbb W$, $\mathbb Z$ ($f\ne0$)                 | $u,v\notin\mu_\infty$          | Thm. 11.16            |
| $\mathbb D$ ($p^2\ne1$), $\mathbb G$                                                                   | $p,hp\notin\mu_\infty$         | Thms. 11.6, 11.13     |
| $\mathbb A$, $\mathbb H$, $\mathbb X$, $\mathbb Y$, degenerate members                                 | open                           |                       |

*Table 5. Primitivity of the non-PI members.*

**Question.** Which members of $\mathbb A$, $\mathbb H$, $\mathbb X$, $\mathbb Y$ are primitive? And the degenerate members $\mathbb D(\pm1,h)$, $\mathbb M(0,h)$, $\mathbb P(0,h)$, $\mathbb W(0,h)$? None of them is PI.

# A The relations of Section 7

We prove the lists of Section 7 from the monomial relations (4.1) and the rules for the three kinds of plane. As there, $e=e_1$, $r=e_1^{-1}e_2$, $X=e_1^2$, $z=z_1$, $\rho=z_1^{-1}z_2$, $Y=z_1^2$ and $a\triangleright b=aba^{-1}$. Since $\sigma$ and $\tau$ are involutions, (4.1) can be solved for the products of $z_a^{\pm1}$ and $e_b^{\pm1}$ in any order.

**Lemma A.1.** For all $a,b,c,d\in\{1,2\}$, $$\begin{aligned}
z_ae_c^{-1}&=\lambda_{\tau(a),c}^{-1}\,e_{\sigma(c)}^{-1}z_{\tau(a)}, & e_bz_c^{-1}&=\lambda_{\tau(c),b}\,z_{\tau(c)}^{-1}e_{\sigma(b)},\qquad(A.1)
\end{aligned}$$ and consequently $$\begin{aligned}
z_a\triangleright(e_b^{-1}e_c)&=\frac{\lambda_{\tau(a),c}}{\lambda_{\tau(a),b}}\,e_{\sigma(b)}^{-1}e_{\sigma(c)}, & e_b\triangleright(z_c^{-1}z_d)&=\frac{\lambda_{\tau(c),b}}{\lambda_{\tau(d),b}}\,z_{\tau(c)}^{-1}z_{\tau(d)},\qquad(A.2)\\
z_a\triangleright e_b^2&=\lambda_{ab}\lambda_{\tau(a),b}\,e_{\sigma(b)}^2, & e_b\triangleright z_c^2&=\bigl(\lambda_{\tau(c),b}\lambda_{\tau(c),\sigma(b)}\bigr)^{-1}z_{\tau(c)}^2 .\qquad(A.3)
\end{aligned}$$

*Proof.* By (4.1), $z_{\tau(a)}e_c=\lambda_{\tau(a),c}\,e_{\sigma(c)}z_a$. Multiplying by $e_{\sigma(c)}^{-1}$ on the left and by $e_c^{-1}$ on the right gives the first formula of (A.1), and inverting both sides of it, with $(a,c)$ renamed $(c,b)$, gives the second.

For (A.2), the first formula of (A.1) and (4.1) give $$z_ae_b^{-1}e_cz_a^{-1}=\lambda_{\tau(a),b}^{-1}\,e_{\sigma(b)}^{-1}\bigl(z_{\tau(a)}e_c\bigr)z_a^{-1}=\frac{\lambda_{\tau(a),c}}{\lambda_{\tau(a),b}}\,e_{\sigma(b)}^{-1}e_{\sigma(c)},$$ and both formulas of (A.1) give $$e_bz_c^{-1}z_de_b^{-1}=\lambda_{\tau(c),b}\,z_{\tau(c)}^{-1}e_{\sigma(b)}\bigl(z_de_b^{-1}\bigr)=\frac{\lambda_{\tau(c),b}}{\lambda_{\tau(d),b}}\,z_{\tau(c)}^{-1}e_{\sigma(b)}e_{\sigma(b)}^{-1}z_{\tau(d)} .$$ For (A.3), $z_ae_be_b=\lambda_{ab}\,e_{\sigma(b)}z_{\tau(a)}e_b=\lambda_{ab}\lambda_{\tau(a),b}\,e_{\sigma(b)}^2z_a$. Applying the second formula of (A.1) twice, $$e_bz_c^{-2}=\lambda_{\tau(c),b}\,z_{\tau(c)}^{-1}e_{\sigma(b)}z_c^{-1}=\lambda_{\tau(c),b}\lambda_{\tau(c),\sigma(b)}\,z_{\tau(c)}^{-2}e_b,$$ and inverting gives the last formula. $\square$

**Lemma A.2.** Let $v_1,v_2$ be one of the two bases of a datum, $w=v_1^{-1}v_2$ and $V=v_1^2$. Then $v_2=v_1w$ and $v_2^{-1}v_1=w^{-1}$. Moreover:

1.  if the basis is balanced, $v_1\triangleright w=w^{-1}$, $v_2^2=V$, and $V$ commutes with $w$;

2.  if the plane is commutative, $v_1\triangleright w=w$, $v_2^2=Vw^2$, and $V$ commutes with $w$;

3.  if the basis is a coordinate basis with $v_2v_1=q\,v_1v_2$, then $v_1\triangleright w=q^{-1}w$, $V\triangleright w=q^{-2}w$ and $v_2^2=q\,Vw^2$.

*Proof.* In (i), $v_1^2=v_2^2$ is central in the plane, as is every square of a linear form in $\Bbbk_{-1}[x_1,x_2]$, and $$v_1wv_1^{-1}=v_2v_1^{-1}=v_2^{-1}v_2^2v_1^{-1}=v_2^{-1}v_1=w^{-1}.$$ Part (ii) is clear. In (iii), $v_1^{-1}v_2v_1=q\,v_2$ gives $v_2v_1^{-1}=q^{-1}v_1^{-1}v_2$, that is, $v_1\triangleright w=q^{-1}w$. Applying this twice, $V\triangleright w=q^{-2}w$, so $wV=q^2Vw$ and $$v_2^2=v_1wv_1w=(v_1\triangleright w)\,Vw=q^{-1}wVw=q\,Vw^2.$$ $\square$

Conjugation is multiplicative, so $Y\triangleright{}=(z\triangleright{})^2$, $X\triangleright{}=(e\triangleright{})^2$ and $\rho\triangleright r=z_1^{-1}\triangleright(z_2\triangleright r)$. Put $$\mu_a=\frac{\lambda_{\tau(a),2}}{\lambda_{\tau(a),1}}\qquad(a=1,2).$$

**Lemma A.3.** For every monomial datum:

1.  $z_a\triangleright r=\mu_a\,r$ if $\sigma$ is the identity, and $z_a\triangleright r=\mu_a\,r^{-1}$ otherwise;

2.  $\kappa=\mu_2/\mu_1$, where $\rho\triangleright r=\kappa r$;

3.  $e\triangleright\rho=\dfrac{\lambda_{\tau(1),1}}{\lambda_{\tau(2),1}}\,\rho^{\pm1}$, with the sign $+$ exactly when $\tau$ is the identity;

4.  $z\triangleright X=\lambda_{11}\lambda_{\tau(1),1}\,e_{\sigma(1)}^2$ and $e\triangleright Y=\bigl(\lambda_{\tau(1),1}\lambda_{\tau(1),\sigma(1)}\bigr)^{-1}z_{\tau(1)}^2$.

*Proof.* Part (i) is (A.2) with $b=1$, $c=2$, together with $e_2^{-1}e_1=r^{-1}$. For (ii), if $\sigma$ is the identity then $z_1^{-1}\triangleright r=\mu_1^{-1}r$ and $\rho\triangleright r=z_1^{-1}\triangleright(\mu_2r)=\mu_2\mu_1^{-1}r$. Otherwise $z_1\triangleright r^{-1}=\mu_1^{-1}r$, so $z_1^{-1}\triangleright r=\mu_1r^{-1}$ and $$\rho\triangleright r=z_1^{-1}\triangleright(\mu_2r^{-1})=\mu_2\,(\mu_1r^{-1})^{-1}=\mu_2\mu_1^{-1}r .$$ Part (iii) is (A.2) with $b=c=1$, $d=2$, since $z_{\tau(1)}^{-1}z_{\tau(2)}$ is $\rho$ or $z_2^{-1}z_1=\rho^{-1}$. Part (iv) is (A.3) with $a=b=1$ and $b=c=1$. $\square$

*Proof of the lists of Section 7.* When $z\triangleright r$ is a multiple of $r^{-1}$, say $\mu r^{-1}$, then $Y\triangleright r=z\triangleright(\mu r^{-1})=\mu(\mu r^{-1})^{-1}=r$; in the same way $X\triangleright\rho=\rho$ when $e\triangleright\rho$ is a multiple of $\rho^{-1}$. The relations $e\triangleright r$ and $z\triangleright\rho$ are Lemma A.2 applied to the two sides, and so is the action of $X$ on $r$ (in $\mathbb N$, Lemma A.2(iii) gives $X\triangleright r=q^{-2}r=r$ because $q=-1$).

*Type I* ($\sigma$ and $\tau$ both the transposition). Here $\mu_1=\lambda_{22}/\lambda_{21}$ and $\mu_2=\lambda_{12}/\lambda_{11}$, and Lemma A.3 gives $$\begin{aligned}
z\triangleright r&=\frac{\lambda_{22}}{\lambda_{21}}\,r^{-1}, & \kappa&=\frac{\lambda_{12}\lambda_{21}}{\lambda_{11}\lambda_{22}}, & e\triangleright\rho&=\frac{\lambda_{21}}{\lambda_{11}}\,\rho^{-1}, & z\triangleright X&=\lambda_{11}\lambda_{21}\,e_2^2, & e\triangleright Y&=\frac{z_2^2}{\lambda_{21}\lambda_{22}} .
\end{aligned}$$ In $\mathbb M$, $\mathbb O$ and $\mathbb E$ the $x$-side is balanced, so $e_2^2=X$ and $Y\triangleright X=(\lambda_{11}\lambda_{21})^2X$. In $\mathbb M$ and $\mathbb O$ the $y$-side is balanced as well, so $z_2^2=Y$. In $\mathbb E$ it is a coordinate basis with $q=p$, so $z\triangleright\rho=p^{-1}\rho$, $Y\triangleright\rho=p^{-2}\rho=-\rho$ and $z_2^2=pY\rho^2$.

In $\mathbb B$ both bases are coordinate bases with $q=p$ and $(\lambda_{ab})=h(1,1,-1,1)$, so $\mu_1=-1$, $\mu_2=1$. Lemma A.2(iii) gives $e\triangleright r=p^{-1}r$, $z\triangleright\rho=p^{-1}\rho$, $X\triangleright r=-r$, $Y\triangleright\rho=-\rho$, $e_2^2=pXr^2$ and $z_2^2=pY\rho^2$. Hence $z\triangleright X=-ph^2Xr^2$, $e\triangleright Y=-ph^{-2}Y\rho^2$ and $$Y\triangleright X=-ph^2\,(z\triangleright X)(z\triangleright r)^2=p^2h^4\,Xr^2r^{-2}=-h^4X .$$

*Type $0$* ($\mathbb S$). Here $\mu_1=\lambda_{12}/\lambda_{11}$ and $\mu_2=\lambda_{22}/\lambda_{21}$, so $$\begin{aligned}
z\triangleright r&=\frac{\lambda_{12}}{\lambda_{11}}\,r, & \kappa&=\frac{\lambda_{11}\lambda_{22}}{\lambda_{12}\lambda_{21}}, & e\triangleright\rho&=\frac{\lambda_{11}}{\lambda_{21}}\,\rho, & z\triangleright X&=\lambda_{11}^2X,
\end{aligned}$$ and squaring, $Y\triangleright r=(\lambda_{12}/\lambda_{11})^2r$, $X\triangleright\rho=(\lambda_{11}/\lambda_{21})^2\rho$ and $Y\triangleright X=\lambda_{11}^4X$.

*Type II* ($\tau$ the transposition, $\sigma$ the identity). Here $\mu_1=\lambda_{22}/\lambda_{21}$ and $\mu_2=\lambda_{12}/\lambda_{11}$, so $$\begin{aligned}
z\triangleright r&=\frac{\lambda_{22}}{\lambda_{21}}\,r, & \kappa&=\frac{\lambda_{12}\lambda_{21}}{\lambda_{11}\lambda_{22}}, & e\triangleright\rho&=\frac{\lambda_{21}}{\lambda_{11}}\,\rho^{-1}, & z\triangleright X&=\lambda_{11}\lambda_{21}X,
\end{aligned}$$ and squaring, $Y\triangleright r=(\lambda_{22}/\lambda_{21})^2r$ and $Y\triangleright X=(\lambda_{11}\lambda_{21})^2X$. The $y$-side is balanced in $\mathbb T$, a coordinate basis with $q=-1$ in $\mathbb P$ and commutative in $\mathbb Z$, which gives $z\triangleright\rho=\rho^{-1}$, $-\rho$, $\rho$.

*Type III* ($\sigma$ the transposition, $\tau$ the identity). Here $\mu_1=\lambda_{12}/\lambda_{11}$ and $\mu_2=\lambda_{22}/\lambda_{21}$, so $$\begin{aligned}
z\triangleright r&=\frac{\lambda_{12}}{\lambda_{11}}\,r^{-1}, & \kappa&=\frac{\lambda_{11}\lambda_{22}}{\lambda_{12}\lambda_{21}}, & e\triangleright\rho&=\frac{\lambda_{11}}{\lambda_{21}}\,\rho, & z\triangleright X&=\lambda_{11}^2\,e_2^2, & e\triangleright Y&=\frac{Y}{\lambda_{11}\lambda_{12}} .
\end{aligned}$$ The $y$-side is balanced in all three families. On the $x$-side, Lemma A.2 gives $e_2^2=X$ in $\mathbb U$, $e_2^2=-Xr^2$ in $\mathbb N$ (coordinate, $q=-1$) and $e_2^2=Xr^2$ in $\mathbb W$ (commutative). This gives the three values of $z\triangleright X$. In $\mathbb U$, $Y\triangleright X=\lambda_{11}^4X$. In $\mathbb N$ and $\mathbb W$, with $z\triangleright X=\pm\lambda_{11}^2Xr^2$, $$Y\triangleright X=\pm\lambda_{11}^2\,(z\triangleright X)(z\triangleright r)^2=\lambda_{11}^4\,Xr^2\Bigl(\frac{\lambda_{12}}{\lambda_{11}}\Bigr)^2r^{-2}=(\lambda_{11}\lambda_{12})^2X.$$ $\square$

**Remark.** The scalars in these lists only involve the $\lambda_{ab}$ and the plane parameters. In the proof of Theorem 7.2 they are evaluated on the data of Table 2; for instance, in $\mathbb O$, where $(\lambda_{ab})=h(1-s,1+s,1+s,1-s)$, one gets $\kappa=\bigl(\frac{1+s}{1-s}\bigr)^2=u^2$, $(\lambda_{11}\lambda_{21})^2=h^4(1-f)^2=v^2$, and the element $ez$ acts by $$\begin{aligned}
ez\triangleright r&=e\triangleright\Bigl(\frac{\lambda_{22}}{\lambda_{21}}r^{-1}\Bigr)=\frac{1-s}{1+s}\,r=u^{-1}r, & ez\triangleright\rho&=e\triangleright\rho^{-1}=\frac{\lambda_{11}}{\lambda_{21}}\,\rho=u^{-1}\rho,\\
ez\triangleright X&=\lambda_{11}\lambda_{21}X=vX, & ez\triangleright Y&=e\triangleright Y=\frac{Y}{\lambda_{21}\lambda_{22}}=v^{-1}Y,
\end{aligned}$$ as stated there.

# References

-   M. Artin, L. W. Small, J. J. Zhang, Generic flatness for strongly Noetherian algebras, J. Algebra 221 (1999), 579--610.

-   K. A. Brown, K. R. Goodearl, Lectures on Algebraic Quantum Groups, Advanced Courses in Mathematics CRM Barcelona, Birkhäuser, Basel, 2002.

-   K. Chan, J. Gaddis, R. Won, J. J. Zhang, Ozone groups and centers of skew polynomial rings, Int. Math. Res. Not. IMRN 2024, no. 7, 5689--5727.

-   K. Chan, J. Gaddis, R. Won, J. J. Zhang, Ozone groups of Artin--Schelter regular algebras satisfying a polynomial identity, Math. Z. 310 (2025), no. 4, Paper No. 69.

-   J. Gaddis, D. Yee, A family of algebras with trivial ozone group, arXiv:2512.09766.

-   J. Gómez, H. Venegas, The ozone groups of the algebras $B_q(f)$, arXiv:2608.28868.

-   J. Gómez, H. Venegas, The ozone group of $U_q^+(B_2)$, arXiv:2609.04919.

-   J. Gómez, H. Suárez, H. Venegas, Ozone groups and centers of two-parameter quantum Heisenberg algebras, arXiv:2610.31781.

-   J. Gómez, H. Venegas, The double extension family $\mathbb D$ of type $(14641)$: PI degree, center, ozone group and graded automorphisms, preprint (2026).

-   J. Gómez, H. Venegas, PI degree, center and ozone group of the double extension family $G$ of type $(14641)$, preprint (2026).

-   J. Gómez, H. Venegas, The double extension family $\mathbb K$ of type $(14641)$: PI degree, center, ozone group and a reflection, preprint (2026).

-   K. Herrera, S. Higuera, A. Rubiano, Finite Gröbner--Shirshov bases for double extension regular algebras of type (14641), arXiv:2509.05583.

-   O. Lezama, H. Venegas, Gelfand--Kirillov dimension for rings, São Paulo J. Math. Sci. 14 (2020), 207--222.

-   O. Lezama, H. Venegas, Center of skew PBW extensions, Internat. J. Algebra Comput. 30 (2020), no. 8, 1625--1650.

-   O. Lezama, H. Venegas, The center of the total ring of fractions, Serdica Math. J. 46 (2020), 109--120.

-   S. Liu, Q. Wu, R. Zhu, The ozone groups of PI Artin--Schelter regular algebras are abelian, arXiv:2512.21056.

-   J. C. McConnell, J. C. Robson, Noncommutative Noetherian Rings, Graduate Studies in Mathematics 30, Amer. Math. Soc., Providence, 2001.

-   J. C. McConnell, J. J. Pettit, Crossed products and multiplicative analogues of Weyl algebras, J. London Math. Soc. (2) 38 (1988), 47--55.

-   M. Newman, Integral Matrices, Academic Press, New York, 1972.

-   A. Rubiano, A. Reyes, Smooth geometry of double extension regular algebras of type (14641), arXiv:2409.10264.

-   A. Rubiano, Center of double extension regular algebras of type (14641), arXiv:2601.18928.

-   A. Rubiano, A note on explicit homological invariants of graded double Ore extensions, arXiv:2604.02456.

-   A. Rubiano, Normal ordering and Stirling-type combinatorics for double Ore extensions of type (14641), arXiv:2606.08396.

-   J. J. Zhang, J. Zhang, Double Ore extensions, J. Pure Appl. Algebra 212 (2008), 2668--2690.

-   J. J. Zhang, J. Zhang, Double extension regular algebras of type (14641), J. Algebra 322 (2009), 373--409.